<a href="https://colab.research.google.com/github/alsedawy/AI/blob/main/Another_copy_of_X_AutoAlign_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# STEP 1 — Download and Validate CodeAlpaca-20K
# ============================================================

import json
import urllib.request
from pathlib import Path

# Official CodeAlpaca repository file
URL = "https://raw.githubusercontent.com/sahil280114/codealpaca/master/data/code_alpaca_20k.json"

DATA_PATH = Path("/content/code_alpaca_20k.json")

print("=" * 70)
print("DOWNLOADING CODEALPACA-20K")
print("=" * 70)

urllib.request.urlretrieve(URL, DATA_PATH)

print(f"File downloaded: {DATA_PATH}")
print(f"File size: {DATA_PATH.stat().st_size / (1024**2):.2f} MB")


# ------------------------------------------------------------
# Load JSON
# ------------------------------------------------------------

with open(DATA_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

print(f"Records loaded: {len(data):,}")


# ------------------------------------------------------------
# Basic schema validation
# ------------------------------------------------------------

required_fields = {"instruction", "input", "output"}

valid_schema = 0
invalid_schema = 0

for item in data:
    if isinstance(item, dict) and required_fields.issubset(item.keys()):
        valid_schema += 1
    else:
        invalid_schema += 1

print(f"Valid schema records:   {valid_schema:,}")
print(f"Invalid schema records: {invalid_schema:,}")


# ------------------------------------------------------------
# Empty output check
# ------------------------------------------------------------

empty_outputs = [
    i for i, item in enumerate(data)
    if not str(item.get("output", "")).strip()
]

print(f"Empty outputs: {len(empty_outputs):,}")


# ------------------------------------------------------------
# Duplicate instruction check
# ------------------------------------------------------------

instructions = [
    str(item.get("instruction", "")).strip()
    for item in data
]

unique_instructions = len(set(instructions))
duplicates = len(instructions) - unique_instructions

print(f"Unique instructions: {unique_instructions:,}")
print(f"Duplicate instructions: {duplicates:,}")


# ------------------------------------------------------------
# Final report
# ------------------------------------------------------------

print("=" * 70)
print("CODEALPACA DATASET VALIDATION")
print("=" * 70)

print(f"Total records:          {len(data):,}")
print(f"Valid schema:           {valid_schema:,}")
print(f"Invalid schema:         {invalid_schema:,}")
print(f"Empty outputs:          {len(empty_outputs):,}")
print(f"Duplicate instructions: {duplicates:,}")

if invalid_schema == 0:
    print("\n✓ JSON structure is valid")

if len(data) >= 20000:
    print("✓ Dataset size is consistent with CodeAlpaca-20K")

print("\nDATASET READY FOR QUALITY FILTERING")
print("=" * 70)

DOWNLOADING CODEALPACA-20K
File downloaded: /content/code_alpaca_20k.json
File size: 7.68 MB
Records loaded: 20,022
Valid schema records:   20,022
Invalid schema records: 0
Empty outputs: 6
Unique instructions: 20,022
Duplicate instructions: 0
CODEALPACA DATASET VALIDATION
Total records:          20,022
Valid schema:           20,022
Invalid schema:         0
Empty outputs:          6
Duplicate instructions: 0

✓ JSON structure is valid
✓ Dataset size is consistent with CodeAlpaca-20K

DATASET READY FOR QUALITY FILTERING


In [ ]:
# ================================================================
# X-AutoAlign RAG — STEP 2
# CodeAlpaca-20K
# QUALITY FILTERING + CANDIDATE POOL
# SELF-RECOVERING VERSION
# ================================================================

import json
import re
import urllib.request
import pandas as pd
from pathlib import Path


# ================================================================
# 1. CONFIGURATION
# ================================================================

DATA_PATH = Path("/content/code_alpaca_20k.json")

CANDIDATE_JSON = Path("/content/candidate_pool.json")
CANDIDATE_CSV  = Path("/content/candidate_pool.csv")

DATA_URL = (
    "https://raw.githubusercontent.com/"
    "sahil280114/codealpaca/master/"
    "data/code_alpaca_20k.json"
)

MIN_INSTRUCTION_LENGTH = 15
MIN_OUTPUT_LENGTH = 20


# ================================================================
# 2. HEADER
# ================================================================

print("=" * 80)
print("X-AutoAlign RAG — STEP 2")
print("QUALITY FILTERING + CANDIDATE POOL")
print("=" * 80)


# ================================================================
# 3. CHECK / RECOVER DATASET
# ================================================================

if DATA_PATH.exists():

    print("\n✓ Dataset already exists")
    print(f"  Path: {DATA_PATH}")

else:

    print("\nDataset not found in current Colab runtime.")
    print("Downloading CodeAlpaca-20K automatically...")
    print("-" * 80)

    try:

        urllib.request.urlretrieve(
            DATA_URL,
            DATA_PATH
        )

        print("✓ Dataset downloaded successfully")

    except Exception as e:

        raise RuntimeError(
            "Failed to download CodeAlpaca-20K.\n"
            "Please check the Colab internet connection.\n\n"
            f"Original error: {e}"
        )


# ================================================================
# 4. FILE VALIDATION
# ================================================================

file_size_mb = (
    DATA_PATH.stat().st_size /
    (1024 ** 2)
)

print(f"\nDataset path : {DATA_PATH}")
print(f"Dataset size : {file_size_mb:.2f} MB")


if file_size_mb < 1:

    raise ValueError(
        "Downloaded file is unexpectedly small. "
        "The dataset may not have downloaded correctly."
    )


# ================================================================
# 5. LOAD JSON
# ================================================================

print("\nLoading JSON...")

try:

    with open(
        DATA_PATH,
        "r",
        encoding="utf-8"
    ) as f:

        raw_data = json.load(f)

except json.JSONDecodeError as e:

    raise ValueError(
        f"Invalid JSON file: {e}"
    )


print(
    f"✓ Records loaded: {len(raw_data):,}"
)


# ================================================================
# 6. BASIC DATASET VALIDATION
# ================================================================

required_fields = {
    "instruction",
    "input",
    "output"
}

schema_valid = 0
schema_invalid = 0

for record in raw_data:

    if (
        isinstance(record, dict)
        and required_fields.issubset(record.keys())
    ):

        schema_valid += 1

    else:

        schema_invalid += 1


print("\n" + "=" * 80)
print("BASIC DATASET VALIDATION")
print("=" * 80)

print(
    f"Total records       : {len(raw_data):,}"
)

print(
    f"Valid schema        : {schema_valid:,}"
)

print(
    f"Invalid schema      : {schema_invalid:,}"
)


if schema_invalid > 0:

    print(
        "\nWARNING: Some records have invalid schema."
    )


# ================================================================
# 7. TEXT NORMALIZATION
# ================================================================

def normalize_text(text):

    if text is None:
        return ""

    return re.sub(
        r"\s+",
        " ",
        str(text)
    ).strip()


# ================================================================
# 8. CODE DETECTION
# ================================================================

def looks_like_code(text):

    text = str(text)

    patterns = [

        r"\bdef\s+\w+\s*\(",

        r"\bfunction\s+\w+\s*\(",

        r"\bclass\s+\w+",

        r"#include\s*<",

        r"\bimport\s+\w+",

        r"\bfrom\s+\w+\s+import\b",

        r"\bfor\s+\w+\s+in\b",

        r"\bwhile\s*\(",

        r"\breturn\b",

        r"=>",

        r"\{.*\}",
    ]

    return any(
        re.search(
            pattern,
            text,
            re.I | re.S
        )
        for pattern in patterns
    )


# ================================================================
# 9. LANGUAGE DETECTION
# ================================================================

def detect_language(text):

    text = str(text)

    arabic_chars = len(
        re.findall(
            r"[\u0600-\u06FF]",
            text
        )
    )

    latin_chars = len(
        re.findall(
            r"[A-Za-z]",
            text
        )
    )

    if (
        arabic_chars > 0
        and arabic_chars >= latin_chars
    ):

        return "ARABIC"

    if latin_chars > 0:

        return "ENGLISH"

    return "OTHER"


# ================================================================
# 10. TASK FAMILY CLASSIFICATION
# ================================================================

def detect_task_family(instruction):

    text = normalize_text(
        instruction
    ).lower()

    rules = {

        "PALINDROME": [
            "palindrome",
            "palindrom",
        ],

        "FACTORIAL": [
            "factorial",
        ],

        "SORTING": [
            "sort",
            "sorting",
            "sorted",
        ],

        "SEARCH": [
            "search",
            "binary search",
            "linear search",
            "find an element",
        ],

        "STRING": [
            "string",
            "substring",
            "character",
            "anagram",
            "uppercase",
            "lowercase",
            "reverse a string",
        ],

        "LIST_ARRAY": [
            "list",
            "array",
            "tuple",
            "element",
        ],

        "MATH": [
            "calculate",
            "equation",
            "prime number",
            "fibonacci",
            "number",
            "mathematical",
        ],

        "DATA_STRUCTURE": [
            "data structure",
            "stack",
            "queue",
            "linked list",
            "tree",
            "graph",
        ],

        "ALGORITHM": [
            "algorithm",
            "complexity",
            "algorithmic",
        ],

        "DEBUGGING": [
            "debug",
            "fix the code",
            "correct the code",
            "error in the code",
            "modify the code",
        ],
    }

    for family, keywords in rules.items():

        for keyword in keywords:

            if keyword in text:

                return family

    return "OTHER"


# ================================================================
# 11. FILTER STATISTICS
# ================================================================

filter_stats = {

    "total": len(raw_data),

    "invalid_record": 0,

    "empty_instruction": 0,

    "short_instruction": 0,

    "empty_output": 0,

    "short_output": 0,

    "duplicate_instruction": 0,

    "accepted": 0,
}


# ================================================================
# 12. QUALITY FILTERING
# ================================================================

candidate_pool = []

seen_instructions = set()


for idx, record in enumerate(raw_data):

    # ------------------------------------------------------------
    # Structural validation
    # ------------------------------------------------------------

    if not isinstance(record, dict):

        filter_stats[
            "invalid_record"
        ] += 1

        continue


    # ------------------------------------------------------------
    # Extract fields
    # ------------------------------------------------------------

    instruction = normalize_text(
        record.get(
            "instruction",
            ""
        )
    )

    input_text = normalize_text(
        record.get(
            "input",
            ""
        )
    )

    output = str(
        record.get(
            "output",
            ""
        )
    ).strip()


    # ------------------------------------------------------------
    # Instruction validation
    # ------------------------------------------------------------

    if not instruction:

        filter_stats[
            "empty_instruction"
        ] += 1

        continue


    if len(instruction) < MIN_INSTRUCTION_LENGTH:

        filter_stats[
            "short_instruction"
        ] += 1

        continue


    # ------------------------------------------------------------
    # Output validation
    # ------------------------------------------------------------

    if not output:

        filter_stats[
            "empty_output"
        ] += 1

        continue


    if len(output) < MIN_OUTPUT_LENGTH:

        filter_stats[
            "short_output"
        ] += 1

        continue


    # ------------------------------------------------------------
    # Duplicate instruction detection
    # ------------------------------------------------------------

    instruction_key = instruction.lower()

    if instruction_key in seen_instructions:

        filter_stats[
            "duplicate_instruction"
        ] += 1

        continue

    seen_instructions.add(
        instruction_key
    )


    # ------------------------------------------------------------
    # Metadata
    # ------------------------------------------------------------

    task_family = detect_task_family(
        instruction
    )

    language = detect_language(
        instruction
    )

    code_like = looks_like_code(
        output
    )


    # ------------------------------------------------------------
    # Preserve record
    # ------------------------------------------------------------

    candidate_pool.append({

        "source_id": idx,

        "instruction": instruction,

        "input": input_text,

        "output": output,

        "language": language,

        "task_family": task_family,

        "code_like_output": code_like,

        "instruction_length": len(
            instruction
        ),

        "input_length": len(
            input_text
        ),

        "output_length": len(
            output
        ),
    })


filter_stats[
    "accepted"
] = len(candidate_pool)


# ================================================================
# 13. DATAFRAME
# ================================================================

candidate_df = pd.DataFrame(
    candidate_pool
)


# ================================================================
# 14. SAVE CANDIDATE POOL
# ================================================================

with open(
    CANDIDATE_JSON,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        candidate_pool,
        f,
        ensure_ascii=False,
        indent=2
    )


candidate_df.to_csv(
    CANDIDATE_CSV,
    index=False,
    encoding="utf-8-sig"
)


# ================================================================
# 15. QUALITY FILTERING REPORT
# ================================================================

print("\n" + "=" * 80)
print("QUALITY FILTERING REPORT")
print("=" * 80)

print(
    f"Raw records              : "
    f"{filter_stats['total']:,}"
)

print(
    f"Invalid records          : "
    f"{filter_stats['invalid_record']:,}"
)

print(
    f"Empty instructions       : "
    f"{filter_stats['empty_instruction']:,}"
)

print(
    f"Short instructions       : "
    f"{filter_stats['short_instruction']:,}"
)

print(
    f"Empty outputs            : "
    f"{filter_stats['empty_output']:,}"
)

print(
    f"Short outputs            : "
    f"{filter_stats['short_output']:,}"
)

print(
    f"Duplicate instructions   : "
    f"{filter_stats['duplicate_instruction']:,}"
)

print("-" * 80)

print(
    f"CANDIDATE POOL           : "
    f"{filter_stats['accepted']:,}"
)


# ================================================================
# 16. TASK FAMILY DISTRIBUTION
# ================================================================

print("\n" + "=" * 80)
print("TASK FAMILY DISTRIBUTION")
print("=" * 80)

if len(candidate_df) > 0:

    print(
        candidate_df[
            "task_family"
        ]
        .value_counts()
        .to_string()
    )


# ================================================================
# 17. LANGUAGE DISTRIBUTION
# ================================================================

print("\n" + "=" * 80)
print("LANGUAGE DISTRIBUTION")
print("=" * 80)

if len(candidate_df) > 0:

    print(
        candidate_df[
            "language"
        ]
        .value_counts()
        .to_string()
    )


# ================================================================
# 18. OUTPUT TYPE DISTRIBUTION
# ================================================================

print("\n" + "=" * 80)
print("OUTPUT TYPE DISTRIBUTION")
print("=" * 80)

if len(candidate_df) > 0:

    print(
        candidate_df[
            "code_like_output"
        ]
        .value_counts()
        .to_string()
    )


# ================================================================
# 19. PREVIEW
# ================================================================

print("\n" + "=" * 80)
print("CANDIDATE POOL PREVIEW")
print("=" * 80)

if len(candidate_df) > 0:

    preview_cols = [

        "source_id",

        "instruction",

        "task_family",

        "language",

        "code_like_output",
    ]

    print(
        candidate_df[
            preview_cols
        ]
        .head(15)
        .to_string(
            index=False
        )
    )


# ================================================================
# 20. FINAL STATUS
# ================================================================

print("\n" + "=" * 80)
print("STEP 2 COMPLETED SUCCESSFULLY")
print("=" * 80)

print(
    f"Raw Dataset     : "
    f"{len(raw_data):,}"
)

print(
    f"Candidate Pool  : "
    f"{len(candidate_df):,}"
)

print(
    f"Saved JSON      : "
    f"{CANDIDATE_JSON}"
)

print(
    f"Saved CSV       : "
    f"{CANDIDATE_CSV}"
)

print("\nNEXT STEP:")
print(
    "STEP 3 — Stratified Selection → "
    "30 Evaluation Queries"
)

print("=" * 80)

X-AutoAlign RAG — STEP 2
QUALITY FILTERING + CANDIDATE POOL

✓ Dataset already exists
  Path: /content/code_alpaca_20k.json

Dataset path : /content/code_alpaca_20k.json
Dataset size : 7.68 MB

Loading JSON...
✓ Records loaded: 20,022

BASIC DATASET VALIDATION
Total records       : 20,022
Valid schema        : 20,022
Invalid schema      : 0

QUALITY FILTERING REPORT
Raw records              : 20,022
Invalid records          : 0
Empty instructions       : 0
Short instructions       : 2
Empty outputs            : 6
Short outputs            : 1,063
Duplicate instructions   : 0
--------------------------------------------------------------------------------
CANDIDATE POOL           : 18,951

TASK FAMILY DISTRIBUTION
task_family
OTHER             8030
LIST_ARRAY        4068
STRING            2608
MATH              2494
SORTING            721
DATA_STRUCTURE     311
SEARCH             256
ALGORITHM          175
PALINDROME         125
DEBUGGING           89
FACTORIAL           74

LANGUAGE DIS

In [ ]:
# ================================================================
# X-AutoAlign RAG — STEP 3
# STRATIFIED SELECTION OF 30 EVALUATION QUERIES
# ================================================================

import json
import csv
import random
from pathlib import Path
from collections import Counter

print("=" * 75)
print("X-AutoAlign RAG — STEP 3")
print("STRATIFIED SELECTION OF 30 EVALUATION QUERIES")
print("=" * 75)

# ----------------------------------------------------------------
# 1. Configuration
# ----------------------------------------------------------------

CANDIDATE_PATH = Path("/content/candidate_pool.json")

OUTPUT_JSON = Path("/content/evaluation_candidates_30.json")
OUTPUT_CSV  = Path("/content/evaluation_candidates_30.csv")

RANDOM_SEED = 42

# Coverage-oriented stratified allocation
STRATA_QUOTA = {
    "OTHER": 4,
    "LIST_ARRAY": 5,
    "STRING": 5,
    "MATH": 5,
    "SORTING": 3,
    "DATA_STRUCTURE": 2,
    "SEARCH": 2,
    "ALGORITHM": 1,
    "PALINDROME": 1,
    "DEBUGGING": 1,
    "FACTORIAL": 1,
}

EXPECTED_TOTAL = sum(STRATA_QUOTA.values())

assert EXPECTED_TOTAL == 30, (
    f"Quota error: expected 30 but got {EXPECTED_TOTAL}"
)

# ----------------------------------------------------------------
# 2. Check Candidate Pool
# ----------------------------------------------------------------

if not CANDIDATE_PATH.exists():
    raise FileNotFoundError(
        f"Candidate pool not found: {CANDIDATE_PATH}"
    )

with open(CANDIDATE_PATH, "r", encoding="utf-8") as f:
    candidate_pool = json.load(f)

if not isinstance(candidate_pool, list):
    raise TypeError(
        "candidate_pool.json must contain a JSON list."
    )

print(f"\nCandidate Pool Loaded : {len(candidate_pool):,}")

# ----------------------------------------------------------------
# 3. Normalize records
# ----------------------------------------------------------------

normalized = []

for idx, record in enumerate(candidate_pool):

    if not isinstance(record, dict):
        continue

    item = dict(record)

    # ------------------------------------------------------------
    # Flexible field handling
    # ------------------------------------------------------------

    instruction = (
        item.get("instruction")
        or item.get("prompt")
        or item.get("question")
        or ""
    )

    output = (
        item.get("output")
        or item.get("solution")
        or item.get("response")
        or ""
    )

    task_family = (
        item.get("task_family")
        or "OTHER"
    )

    language = (
        item.get("language")
        or "ENGLISH"
    )

    code_like = item.get("code_like_output", None)

    # ------------------------------------------------------------
    # If code_like_output is missing, infer conservatively
    # ------------------------------------------------------------

    if code_like is None:
        code_like = bool(output and len(str(output).strip()) > 0)

    item["_instruction"] = str(instruction).strip()
    item["_output"] = str(output).strip()
    item["_task_family"] = str(task_family).upper().strip()
    item["_language"] = str(language).upper().strip()
    item["_code_like"] = bool(code_like)

    normalized.append(item)

# ----------------------------------------------------------------
# 4. Basic validation
# ----------------------------------------------------------------

print(f"Normalized Records    : {len(normalized):,}")

# Remove empty instructions
normalized = [
    x for x in normalized
    if x["_instruction"]
]

print(f"Non-empty Instructions: {len(normalized):,}")

# ----------------------------------------------------------------
# 5. Deduplicate instructions
# ----------------------------------------------------------------

unique_records = {}
duplicate_count = 0

for item in normalized:

    key = item["_instruction"].strip().lower()

    if key in unique_records:
        duplicate_count += 1
        continue

    unique_records[key] = item

normalized = list(unique_records.values())

print(f"Duplicate Instructions Removed: {duplicate_count:,}")
print(f"Unique Candidate Records       : {len(normalized):,}")

# ----------------------------------------------------------------
# 6. Prefer code-like records for evaluation
# ----------------------------------------------------------------

code_candidates = [
    x for x in normalized
    if x["_code_like"] is True
]

print(f"Code-like Candidates            : {len(code_candidates):,}")

# ----------------------------------------------------------------
# 7. Build strata
# ----------------------------------------------------------------

strata = {
    family: []
    for family in STRATA_QUOTA
}

for item in code_candidates:

    family = item["_task_family"]

    if family not in strata:
        family = "OTHER"

    strata[family].append(item)

# ----------------------------------------------------------------
# 8. Report availability
# ----------------------------------------------------------------

print("\n" + "=" * 75)
print("STRATUM AVAILABILITY")
print("=" * 75)

for family, quota in STRATA_QUOTA.items():

    available = len(strata[family])

    status = "OK" if available >= quota else "INSUFFICIENT"

    print(
        f"{family:<18} "
        f"Available={available:>5} "
        f"Required={quota:>2} "
        f"[{status}]"
    )

# ----------------------------------------------------------------
# 9. Hard safety check
# ----------------------------------------------------------------

insufficient = {
    family: (len(strata[family]), quota)
    for family, quota in STRATA_QUOTA.items()
    if len(strata[family]) < quota
}

if insufficient:

    print("\nERROR: One or more strata do not have enough records.")

    for family, (available, required) in insufficient.items():
        print(
            f"  {family}: available={available}, "
            f"required={required}"
        )

    raise RuntimeError(
        "STEP 3 stopped intentionally. "
        "No quota was silently substituted."
    )

# ----------------------------------------------------------------
# 10. Stratified Sampling
# ----------------------------------------------------------------

rng = random.Random(RANDOM_SEED)

selected = []

for family, quota in STRATA_QUOTA.items():

    pool = list(strata[family])

    rng.shuffle(pool)

    chosen = pool[:quota]

    selected.extend(chosen)

# ----------------------------------------------------------------
# 11. Shuffle final 30 while preserving strata
# ----------------------------------------------------------------

rng.shuffle(selected)

# ----------------------------------------------------------------
# 12. Assign evaluation IDs
# ----------------------------------------------------------------

evaluation_records = []

for i, item in enumerate(selected, start=1):

    record = {
        "eval_id": f"EVAL_{i:02d}",
        "source_id": item.get(
            "source_id",
            item.get("id", None)
        ),
        "instruction": item["_instruction"],
        "reference_output": item["_output"],
        "task_family": item["_task_family"],
        "language": item["_language"],
        "code_like_output": item["_code_like"],
    }

    # Preserve useful original metadata if available
    for key in [
        "task_type",
        "input",
        "category",
        "source",
    ]:
        if key in item:
            record[key] = item[key]

    evaluation_records.append(record)

# ----------------------------------------------------------------
# 13. Final validation
# ----------------------------------------------------------------

assert len(evaluation_records) == 30

eval_ids = [
    x["eval_id"]
    for x in evaluation_records
]

assert len(eval_ids) == len(set(eval_ids))

instructions = [
    x["instruction"].strip().lower()
    for x in evaluation_records
]

assert len(instructions) == len(set(instructions))

# Check final strata
final_distribution = Counter(
    x["task_family"]
    for x in evaluation_records
)

# ----------------------------------------------------------------
# 14. Save JSON
# ----------------------------------------------------------------

with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(
        evaluation_records,
        f,
        ensure_ascii=False,
        indent=2
    )

# ----------------------------------------------------------------
# 15. Save CSV
# ----------------------------------------------------------------

csv_fields = [
    "eval_id",
    "source_id",
    "instruction",
    "reference_output",
    "task_family",
    "language",
    "code_like_output",
]

# Add optional fields if present
optional_fields = [
    "task_type",
    "input",
    "category",
    "source",
]

for field in optional_fields:
    if any(field in x for x in evaluation_records):
        csv_fields.append(field)

with open(
    OUTPUT_CSV,
    "w",
    encoding="utf-8-sig",
    newline=""
) as f:

    writer = csv.DictWriter(
        f,
        fieldnames=csv_fields,
        extrasaction="ignore"
    )

    writer.writeheader()

    for row in evaluation_records:
        writer.writerow(row)

# ----------------------------------------------------------------
# 16. Final Report
# ----------------------------------------------------------------

print("\n" + "=" * 75)
print("FINAL STRATIFIED DISTRIBUTION")
print("=" * 75)

for family in STRATA_QUOTA:

    print(
        f"{family:<18} "
        f"{final_distribution[family]:>2}"
    )

print("-" * 75)
print(
    f"{'TOTAL':<18} "
    f"{len(evaluation_records):>2}"
)

print("\n" + "=" * 75)
print("SELECTED EVALUATION QUERIES")
print("=" * 75)

for item in evaluation_records:

    print(
        f"{item['eval_id']} | "
        f"{item['task_family']:<15} | "
        f"{item['instruction'][:100]}"
    )

# ----------------------------------------------------------------
# 17. Final status
# ----------------------------------------------------------------

print("\n" + "=" * 75)
print("STEP 3 COMPLETED SUCCESSFULLY")
print("=" * 75)

print(f"Candidate Pool      : {len(candidate_pool):,}")
print(f"Unique Candidates   : {len(normalized):,}")
print(f"Selected Queries    : {len(evaluation_records)}")
print(f"Random Seed         : {RANDOM_SEED}")

print(f"\nSaved JSON : {OUTPUT_JSON}")
print(f"Saved CSV  : {OUTPUT_CSV}")

print("\nNEXT STEP:")
print("STEP 4 — Ground Truth Construction + Arabic Cross-Lingual Mapping")
print("=" * 75)

X-AutoAlign RAG — STEP 3
STRATIFIED SELECTION OF 30 EVALUATION QUERIES

Candidate Pool Loaded : 18,951
Normalized Records    : 18,951
Non-empty Instructions: 18,951
Duplicate Instructions Removed: 0
Unique Candidate Records       : 18,951
Code-like Candidates            : 12,301

STRATUM AVAILABILITY
OTHER              Available= 4070 Required= 4 [OK]
LIST_ARRAY         Available= 3064 Required= 5 [OK]
STRING             Available= 2090 Required= 5 [OK]
MATH               Available= 1858 Required= 5 [OK]
SORTING            Available=  520 Required= 3 [OK]
DATA_STRUCTURE     Available=  176 Required= 2 [OK]
SEARCH             Available=  188 Required= 2 [OK]
ALGORITHM          Available=   99 Required= 1 [OK]
PALINDROME         Available=  110 Required= 1 [OK]
DEBUGGING          Available=   59 Required= 1 [OK]
FACTORIAL          Available=   67 Required= 1 [OK]

FINAL STRATIFIED DISTRIBUTION
OTHER               4
LIST_ARRAY          5
STRING              5
MATH                5
SORTING

In [ ]:
# =====================================================================
# X-AutoAlign RAG — STEP 4
# GROUND TRUTH + ARABIC CROSS-LINGUAL MAPPING
# Evaluation Dataset V1
# =====================================================================

!pip -q install transformers sentencepiece

import json
import re
import csv
import ast
from pathlib import Path

print("=" * 75)
print("X-AutoAlign RAG — STEP 4")
print("GROUND TRUTH + ARABIC CROSS-LINGUAL MAPPING")
print("=" * 75)

# ---------------------------------------------------------------------
# 1. Paths
# ---------------------------------------------------------------------

INPUT_PATH = Path("/content/evaluation_candidates_30.json")

OUTPUT_JSON = Path("/content/evaluation_dataset_v1.json")
OUTPUT_CSV  = Path("/content/evaluation_dataset_v1.csv")

if not INPUT_PATH.exists():
    raise FileNotFoundError(
        f"STEP 3 output not found: {INPUT_PATH}"
    )

# ---------------------------------------------------------------------
# 2. Load selected 30 tasks
# ---------------------------------------------------------------------

with open(INPUT_PATH, "r", encoding="utf-8") as f:
    selected = json.load(f)

if not isinstance(selected, list):
    raise TypeError(
        "evaluation_candidates_30.json must contain a JSON list."
    )

if len(selected) != 30:
    raise ValueError(
        f"Expected exactly 30 evaluation tasks, found {len(selected)}."
    )

print(f"\nSelected Evaluation Tasks : {len(selected)}")

# ---------------------------------------------------------------------
# 3. Programming-language detection
# ---------------------------------------------------------------------

def detect_programming_language(text):
    """
    Conservative language detector based primarily on the instruction.
    """

    t = str(text).lower()

    patterns = [
        ("PYTHON", [
            r"\bpython\b",
            r"\bpy\b"
        ]),
        ("JAVASCRIPT", [
            r"\bjavascript\b",
            r"\bjs\b"
        ]),
        ("TYPESCRIPT", [
            r"\btypescript\b"
        ]),
        ("JAVA", [
            r"\bjava\b"
        ]),
        ("C_SHARP", [
            r"\bc#\b",
            r"\bc sharp\b"
        ]),
        ("CPP", [
            r"\bc\+\+\b",
            r"\bcpp\b"
        ]),
        ("C", [
            r"\bc language\b",
            r"\bin c\b",
            r"\bc program\b"
        ]),
        ("PHP", [
            r"\bphp\b"
        ]),
        ("RUBY", [
            r"\bruby\b"
        ]),
        ("GO", [
            r"\bgolang\b",
            r"\bgo language\b"
        ]),
        ("HTML", [
            r"\bhtml\b"
        ]),
        ("SQL", [
            r"\bsql\b"
        ]),
    ]

    for language, regexes in patterns:
        for pattern in regexes:
            if re.search(pattern, t):
                return language

    # Infer Python from common Python syntax if language is not stated.
    if re.search(r"\bdef\s+\w+\s*\(", text):
        return "PYTHON"

    if "import " in text and ":" in text:
        return "PYTHON"

    return "UNKNOWN"


# ---------------------------------------------------------------------
# 4. Python reference validation
# ---------------------------------------------------------------------

def validate_python_code(code):
    """
    Syntax-level validation only.
    Does NOT claim semantic correctness.
    """

    if not code or not code.strip():
        return {
            "valid": False,
            "status": "EMPTY_OUTPUT",
            "error": "Reference output is empty."
        }

    try:
        ast.parse(code)
        return {
            "valid": True,
            "status": "PYTHON_SYNTAX_VALID",
            "error": None
        }

    except SyntaxError as e:
        return {
            "valid": False,
            "status": "PYTHON_SYNTAX_INVALID",
            "error": str(e)
        }


# ---------------------------------------------------------------------
# 5. Generic reference-output validation
# ---------------------------------------------------------------------

def validate_reference(instruction, output, language):

    if not output or not output.strip():
        return {
            "valid": False,
            "status": "EMPTY_OUTPUT",
            "error": "Reference output is empty."
        }

    # Python can be checked syntactically.
    if language == "PYTHON":
        return validate_python_code(output)

    # For other languages we do NOT claim compilation.
    # We only perform conservative structural checks.
    stripped = output.strip()

    if len(stripped) < 10:
        return {
            "valid": False,
            "status": "OUTPUT_TOO_SHORT",
            "error": "Reference output is suspiciously short."
        }

    # Detect obvious truncation patterns.
    truncation_patterns = [
        r"\bif\s+rev_str\s*==\s*input_$",
        r"\bif\s+.*\s*>\s*len$",
        r"return\s*$",
        r"\{\s*$",
        r"\(\s*$",
        r"\[\s*$",
    ]

    for pattern in truncation_patterns:
        if re.search(pattern, stripped):
            return {
                "valid": False,
                "status": "POSSIBLE_TRUNCATION",
                "error": "Reference output appears truncated."
            }

    return {
        "valid": True,
        "status": "NON_PYTHON_STRUCTURAL_CHECK",
        "error": None
    }


# ---------------------------------------------------------------------
# 6. Load English → Arabic translation model
# ---------------------------------------------------------------------

print("\nLoading English → Arabic translation model...")

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_NAME = "Helsinki-NLP/opus-mt-en-ar"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
translator = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

print("Translation model loaded successfully.")

# ---------------------------------------------------------------------
# 7. Arabic translation helper
# ---------------------------------------------------------------------

def translate_to_arabic(text):

    # Keep technical identifiers and programming-language names intact.
    protected = {
        "Python": "PYTHON_LANG",
        "JavaScript": "JAVASCRIPT_LANG",
        "TypeScript": "TYPESCRIPT_LANG",
        "Java": "JAVA_LANG",
        "C++": "CPP_LANG",
        "C#": "CSHARP_LANG",
        "PHP": "PHP_LANG",
        "HTML": "HTML_LANG",
        "SQL": "SQL_LANG",
        "palindrome": "PALINDROME_TERM",
        "string": "STRING_TERM",
        "array": "ARRAY_TERM",
        "function": "FUNCTION_TERM",
        "algorithm": "ALGORITHM_TERM",
    }

    protected_text = text

    for original, placeholder in protected.items():
        protected_text = re.sub(
            re.escape(original),
            placeholder,
            protected_text,
            flags=re.IGNORECASE
        )

    inputs = tokenizer(
        protected_text,
        return_tensors="pt",
        truncation=True,
        max_length=256
    )

    outputs = translator.generate(
        **inputs,
        max_length=256,
        num_beams=4
    )

    translated = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    # Restore protected technical terms.
    restoration = {
        "PYTHON_LANG": "Python",
        "JAVASCRIPT_LANG": "JavaScript",
        "TYPESCRIPT_LANG": "TypeScript",
        "JAVA_LANG": "Java",
        "CPP_LANG": "C++",
        "CSHARP_LANG": "C#",
        "PHP_LANG": "PHP",
        "HTML_LANG": "HTML",
        "SQL_LANG": "SQL",
        "PALINDROME_TERM": "palindrome",
        "STRING_TERM": "string",
        "ARRAY_TERM": "array",
        "FUNCTION_TERM": "function",
        "ALGORITHM_TERM": "algorithm",
    }

    for placeholder, original in restoration.items():
        translated = translated.replace(
            placeholder,
            original
        )

    return translated.strip()


# ---------------------------------------------------------------------
# 8. Build Evaluation Dataset V1
# ---------------------------------------------------------------------

evaluation_dataset = []

print("\n" + "=" * 75)
print("BUILDING EVALUATION DATASET V1")
print("=" * 75)

for item in selected:

    eval_id = item["eval_id"]

    english_query = item["instruction"].strip()
    reference_output = item["reference_output"].strip()

    task_family = item.get(
        "task_family",
        "OTHER"
    )

    language = detect_programming_language(
        english_query
    )

    validation = validate_reference(
        english_query,
        reference_output,
        language
    )

    # Generate Arabic counterpart.
    arabic_query = translate_to_arabic(
        english_query
    )

    record = {
        "eval_id": eval_id,

        # Task identity
        "task_family": task_family,
        "programming_language": language,

        # Source query
        "english_query": english_query,

        # Cross-lingual query
        "arabic_query": arabic_query,

        # Reference solution
        "reference_solution": reference_output,

        # Ground Truth
        #
        # We preserve the validated reference rather than silently
        # modifying the source solution.
        "ground_truth": (
            reference_output
            if validation["valid"]
            else None
        ),

        # Validation information
        "reference_validation": validation["status"],
        "reference_valid": validation["valid"],
        "validation_error": validation["error"],

        # Review control
        "manual_review_required": (
            not validation["valid"]
        ),

        # Experimental groups
        "evaluation_cases": [
            {
                "case_id": f"{eval_id}_EN",
                "language": "ENGLISH",
                "query": english_query,
                "evidence_language": "ENGLISH"
            },
            {
                "case_id": f"{eval_id}_AR",
                "language": "ARABIC",
                "query": arabic_query,
                "evidence_language": "ENGLISH"
            }
        ],

        # Provenance
        "source_id": item.get("source_id")
    }

    evaluation_dataset.append(record)

    print(
        f"{eval_id} | "
        f"{task_family:<15} | "
        f"{language:<12} | "
        f"{validation['status']}"
    )


# ---------------------------------------------------------------------
# 9. Dataset-level statistics
# ---------------------------------------------------------------------

valid_count = sum(
    1 for x in evaluation_dataset
    if x["reference_valid"]
)

review_count = sum(
    1 for x in evaluation_dataset
    if x["manual_review_required"]
)

language_distribution = {}

for x in evaluation_dataset:
    lang = x["programming_language"]
    language_distribution[lang] = (
        language_distribution.get(lang, 0) + 1
    )

task_distribution = {}

for x in evaluation_dataset:
    task = x["task_family"]
    task_distribution[task] = (
        task_distribution.get(task, 0) + 1
    )


# ---------------------------------------------------------------------
# 10. Save JSON
# ---------------------------------------------------------------------

with open(
    OUTPUT_JSON,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        {
            "dataset_name": "X-AutoAlign Evaluation Dataset V1",

            "dataset_version": "V1",

            "source_dataset": "CodeAlpaca-20K",

            "num_source_tasks": 30,

            "num_evaluation_cases": 60,

            "evaluation_design": {
                "ENGLISH": 30,
                "ARABIC_CROSS_LINGUAL": 30
            },

            "ground_truth_policy": (
                "Reference outputs are preserved. "
                "Python outputs receive syntax validation. "
                "Non-Python outputs receive conservative "
                "structural validation. "
                "Invalid or suspicious outputs require manual review."
            ),

            "task_distribution": task_distribution,

            "programming_language_distribution":
                language_distribution,

            "valid_reference_count": valid_count,

            "manual_review_count": review_count,

            "records": evaluation_dataset
        },
        f,
        ensure_ascii=False,
        indent=2
    )


# ---------------------------------------------------------------------
# 11. Save flat CSV
# ---------------------------------------------------------------------

csv_rows = []

for item in evaluation_dataset:

    # English case
    csv_rows.append({
        "case_id":
            f"{item['eval_id']}_EN",

        "eval_id":
            item["eval_id"],

        "evaluation_language":
            "ENGLISH",

        "evidence_language":
            "ENGLISH",

        "task_family":
            item["task_family"],

        "programming_language":
            item["programming_language"],

        "query":
            item["english_query"],

        "ground_truth":
            item["ground_truth"] or "",

        "reference_valid":
            item["reference_valid"],

        "manual_review_required":
            item["manual_review_required"],

        "source_id":
            item["source_id"]
    })

    # Arabic cross-lingual case
    csv_rows.append({
        "case_id":
            f"{item['eval_id']}_AR",

        "eval_id":
            item["eval_id"],

        "evaluation_language":
            "ARABIC",

        "evidence_language":
            "ENGLISH",

        "task_family":
            item["task_family"],

        "programming_language":
            item["programming_language"],

        "query":
            item["arabic_query"],

        "ground_truth":
            item["ground_truth"] or "",

        "reference_valid":
            item["reference_valid"],

        "manual_review_required":
            item["manual_review_required"],

        "source_id":
            item["source_id"]
    })


csv_fields = [
    "case_id",
    "eval_id",
    "evaluation_language",
    "evidence_language",
    "task_family",
    "programming_language",
    "query",
    "ground_truth",
    "reference_valid",
    "manual_review_required",
    "source_id"
]

with open(
    OUTPUT_CSV,
    "w",
    encoding="utf-8-sig",
    newline=""
) as f:

    writer = csv.DictWriter(
        f,
        fieldnames=csv_fields
    )

    writer.writeheader()
    writer.writerows(csv_rows)


# ---------------------------------------------------------------------
# 12. Final report
# ---------------------------------------------------------------------

print("\n" + "=" * 75)
print("STEP 4 COMPLETED")
print("=" * 75)

print(f"Source Tasks              : {len(selected)}")
print(f"Evaluation Cases          : {len(csv_rows)}")
print(f"Valid References          : {valid_count}")
print(f"Manual Review Required    : {review_count}")

print("\nTask Distribution:")
for task, count in sorted(
    task_distribution.items()
):
    print(
        f"  {task:<18} {count}"
    )

print("\nProgramming Language Distribution:")
for lang, count in sorted(
    language_distribution.items()
):
    print(
        f"  {lang:<18} {count}"
    )

print("\nSaved:")
print(f"  JSON : {OUTPUT_JSON}")
print(f"  CSV  : {OUTPUT_CSV}")

print("\n" + "=" * 75)
print("IMPORTANT")
print("=" * 75)

print(
    "Arabic queries are machine-generated drafts."
)

print(
    "Ground Truth is NOT silently corrected."
)

print(
    "Any invalid/suspicious reference is flagged "
    "for manual review."
)

print(
    "\nNEXT STEP:"
    "\nSTEP 5 — Manual Ground-Truth Review + "
    "Final Evaluation Dataset Freeze"
)

print("=" * 75)

X-AutoAlign RAG — STEP 4
GROUND TRUTH + ARABIC CROSS-LINGUAL MAPPING

Selected Evaluation Tasks : 30

Loading English → Arabic translation model...


/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

Translation model loaded successfully.

BUILDING EVALUATION DATASET V1
EVAL_01 | LIST_ARRAY      | PYTHON       | PYTHON_SYNTAX_VALID
EVAL_02 | LIST_ARRAY      | UNKNOWN      | NON_PYTHON_STRUCTURAL_CHECK
EVAL_03 | LIST_ARRAY      | UNKNOWN      | NON_PYTHON_STRUCTURAL_CHECK
EVAL_04 | STRING          | C            | NON_PYTHON_STRUCTURAL_CHECK
EVAL_05 | SORTING         | JAVASCRIPT   | NON_PYTHON_STRUCTURAL_CHECK
EVAL_06 | PALINDROME      | UNKNOWN      | NON_PYTHON_STRUCTURAL_CHECK


<unknown>:4: SyntaxWarning: invalid escape sequence '\w'


EVAL_07 | STRING          | PYTHON       | PYTHON_SYNTAX_VALID
EVAL_08 | ALGORITHM       | PYTHON       | PYTHON_SYNTAX_VALID
EVAL_09 | MATH            | UNKNOWN      | NON_PYTHON_STRUCTURAL_CHECK
EVAL_10 | FACTORIAL       | PYTHON       | PYTHON_SYNTAX_VALID
EVAL_11 | DATA_STRUCTURE  | JAVA         | NON_PYTHON_STRUCTURAL_CHECK
EVAL_12 | STRING          | PYTHON       | PYTHON_SYNTAX_VALID
EVAL_13 | MATH            | UNKNOWN      | NON_PYTHON_STRUCTURAL_CHECK
EVAL_14 | DEBUGGING       | PYTHON       | PYTHON_SYNTAX_VALID
EVAL_15 | OTHER           | UNKNOWN      | NON_PYTHON_STRUCTURAL_CHECK
EVAL_16 | SEARCH          | PYTHON       | PYTHON_SYNTAX_VALID
EVAL_17 | MATH            | UNKNOWN      | NON_PYTHON_STRUCTURAL_CHECK
EVAL_18 | SEARCH          | JAVA         | NON_PYTHON_STRUCTURAL_CHECK
EVAL_19 | DATA_STRUCTURE  | JAVA         | NON_PYTHON_STRUCTURAL_CHECK
EVAL_20 | SORTING         | UNKNOWN      | NON_PYTHON_STRUCTURAL_CHECK
EVAL_21 | OTHER           | JAVASCRIPT   | NON_PYTHON_

In [ ]:
# =====================================================================
# X-AutoAlign RAG — STEP 5
# GROUND TRUTH AUDIT + LANGUAGE RESOLUTION + FINAL DATASET FREEZE
# =====================================================================

import json
import csv
import re
import ast
from pathlib import Path
from collections import Counter

print("=" * 80)
print("X-AutoAlign RAG — STEP 5")
print("GROUND TRUTH AUDIT + FINAL DATASET FREEZE")
print("=" * 80)


# =====================================================================
# 1. Paths
# =====================================================================

INPUT_PATH = Path("/content/evaluation_dataset_v1.json")

FINAL_JSON = Path("/content/evaluation_dataset_final.json")
FINAL_CSV  = Path("/content/evaluation_dataset_final.csv")
FINAL_XLSX = Path("/content/evaluation_dataset_final.xlsx")

if not INPUT_PATH.exists():
    raise FileNotFoundError(
        f"STEP 4 dataset not found: {INPUT_PATH}"
    )


# =====================================================================
# 2. Load Evaluation Dataset V1
# =====================================================================

with open(INPUT_PATH, "r", encoding="utf-8") as f:
    dataset_v1 = json.load(f)

if "records" not in dataset_v1:
    raise ValueError(
        "Invalid evaluation_dataset_v1.json: 'records' not found."
    )

records = dataset_v1["records"]

if len(records) != 30:
    raise ValueError(
        f"Expected 30 source tasks, found {len(records)}."
    )

print(f"\nSource Tasks Loaded : {len(records)}")


# =====================================================================
# 3. Language Detection
# =====================================================================

def detect_language_from_text(text):

    if not text:
        return "UNKNOWN"

    t = str(text).lower()

    # ---------------------------------------------------------------
    # Python
    # ---------------------------------------------------------------

    if re.search(r"\bpython\b", t):
        return "PYTHON"

    # ---------------------------------------------------------------
    # JavaScript
    # ---------------------------------------------------------------

    if (
        re.search(r"\bjavascript\b", t)
        or "console.log" in t
        or "let " in t
        or "const " in t
        or "=>" in t
        or ".split(" in t
        or ".join(" in t
    ):
        return "JAVASCRIPT"

    # ---------------------------------------------------------------
    # TypeScript
    # ---------------------------------------------------------------

    if re.search(r"\btypescript\b", t):
        return "TYPESCRIPT"

    # ---------------------------------------------------------------
    # C#
    # ---------------------------------------------------------------

    if (
        re.search(r"\bc#\b", t)
        or "using system" in t
        or "console.writeline" in t
        or "public static void main" in t
    ):
        return "C_SHARP"

    # ---------------------------------------------------------------
    # Java
    # ---------------------------------------------------------------

    if (
        re.search(r"\bjava\b", t)
        or "system.out.println" in t
        or "public class " in t
        or "public static void main" in t
    ):
        return "JAVA"

    # ---------------------------------------------------------------
    # C++
    # ---------------------------------------------------------------

    if (
        "c++" in t
        or "#include <iostream>" in t
        or "std::" in t
        or "cout <<" in t
    ):
        return "CPP"

    # ---------------------------------------------------------------
    # C
    # ---------------------------------------------------------------

    if (
        re.search(r"\bin c\b", t)
        or "#include <stdio.h>" in t
        or "#include <string.h>" in t
    ):
        return "C"

    # ---------------------------------------------------------------
    # PHP
    # ---------------------------------------------------------------

    if (
        re.search(r"\bphp\b", t)
        or "<?php" in t
        or "$_post" in t
        or "$_get" in t
    ):
        return "PHP"

    # ---------------------------------------------------------------
    # HTML
    # ---------------------------------------------------------------

    if (
        re.search(r"\bhtml\b", t)
        or "<html" in t
        or "<body" in t
        or "<div" in t
    ):
        return "HTML"

    # ---------------------------------------------------------------
    # SQL
    # ---------------------------------------------------------------

    if (
        re.search(r"\bsql\b", t)
        or re.search(r"\bselect\b.+\bfrom\b", t)
    ):
        return "SQL"

    # ---------------------------------------------------------------
    # Ruby
    # ---------------------------------------------------------------

    if re.search(r"\bruby\b", t):
        return "RUBY"

    # ---------------------------------------------------------------
    # Go
    # ---------------------------------------------------------------

    if re.search(r"\bgolang\b", t):
        return "GO"

    return "UNKNOWN"


# =====================================================================
# 4. Resolve Programming Language
# =====================================================================

def resolve_programming_language(record):

    original_language = (
        record.get("programming_language")
        or "UNKNOWN"
    )

    instruction = record.get(
        "english_query",
        ""
    )

    reference = record.get(
        "reference_solution",
        ""
    )

    # ---------------------------------------------------------------
    # First: instruction
    # ---------------------------------------------------------------

    instruction_lang = detect_language_from_text(
        instruction
    )

    if instruction_lang != "UNKNOWN":
        return instruction_lang, "INSTRUCTION"

    # ---------------------------------------------------------------
    # Second: reference solution
    # ---------------------------------------------------------------

    reference_lang = detect_language_from_text(
        reference
    )

    if reference_lang != "UNKNOWN":
        return reference_lang, "REFERENCE_OUTPUT"

    # ---------------------------------------------------------------
    # Third: preserve previous label
    # ---------------------------------------------------------------

    if original_language != "UNKNOWN":
        return original_language, "PREVIOUS_LABEL"

    return "UNKNOWN", "UNRESOLVED"


# =====================================================================
# 5. Python Syntax Audit
# =====================================================================

def python_syntax_audit(code):

    if not code or not code.strip():
        return {
            "syntax_status": "EMPTY",
            "syntax_valid": False,
            "syntax_error": "Empty output."
        }

    try:
        ast.parse(code)

        return {
            "syntax_status": "VALID",
            "syntax_valid": True,
            "syntax_error": None
        }

    except SyntaxError as e:

        return {
            "syntax_status": "INVALID",
            "syntax_valid": False,
            "syntax_error": str(e)
        }


# =====================================================================
# 6. Generic Output Audit
# =====================================================================

def output_audit(code):

    if not code or not code.strip():

        return {
            "output_status": "EMPTY",
            "suspicious": True
        }

    text = code.strip()

    # ---------------------------------------------------------------
    # Suspicious truncation
    # ---------------------------------------------------------------

    truncation_patterns = [

        r"==\s*input_$",

        r">\s*len$",

        r"return\s*$",

        r"if\s+.*:\s*$",

        r"\{\s*$",

        r"\(\s*$",

        r"\[\s*$",

        r"\bdef\s+\w+\([^)]*$",

        r"---\s*DOCUMENT\s*---$",
    ]

    for pattern in truncation_patterns:

        if re.search(
            pattern,
            text,
            flags=re.IGNORECASE
        ):

            return {
                "output_status": "POSSIBLE_TRUNCATION",
                "suspicious": True
            }

    # ---------------------------------------------------------------
    # Very short output
    # ---------------------------------------------------------------

    if len(text) < 10:

        return {
            "output_status": "TOO_SHORT",
            "suspicious": True
        }

    return {
        "output_status": "STRUCTURALLY_PRESENT",
        "suspicious": False
    }


# =====================================================================
# 7. Task / Answer Alignment Audit
# =====================================================================

def alignment_audit(record, language):

    instruction = (
        record.get("english_query", "")
        .lower()
    )

    output = (
        record.get("reference_solution", "")
        .lower()
    )

    issues = []

    # ---------------------------------------------------------------
    # Empty reference
    # ---------------------------------------------------------------

    if not output.strip():
        issues.append("EMPTY_REFERENCE")

    # ---------------------------------------------------------------
    # Palindrome
    # ---------------------------------------------------------------

    if "palindrome" in instruction:

        palindrome_markers = [
            "palindrome",
            "reverse",
            "[::-1]",
            "reversed"
        ]

        if not any(
            marker in output
            for marker in palindrome_markers
        ):
            issues.append(
                "PALINDROME_TASK_OUTPUT_MISMATCH"
            )

    # ---------------------------------------------------------------
    # Factorial
    # ---------------------------------------------------------------

    if "factorial" in instruction:

        factorial_markers = [
            "factorial",
            "fact(",
            "math.factorial",
            "n!"
        ]

        if not any(
            marker in output
            for marker in factorial_markers
        ):
            issues.append(
                "FACTORIAL_TASK_OUTPUT_MISMATCH"
            )

    # ---------------------------------------------------------------
    # Sorting
    # ---------------------------------------------------------------

    if (
        "sort" in instruction
        or "sorting" in instruction
    ):

        sorting_markers = [
            "sort",
            "sorted",
            "quicksort",
            "mergesort",
            "merge sort"
        ]

        if not any(
            marker in output
            for marker in sorting_markers
        ):
            issues.append(
                "SORTING_TASK_OUTPUT_MISMATCH"
            )

    # ---------------------------------------------------------------
    # String
    # ---------------------------------------------------------------

    if "string" in instruction:

        string_markers = [
            "string",
            "str",
            "text",
            "char"
        ]

        if not any(
            marker in output
            for marker in string_markers
        ):
            issues.append(
                "STRING_TASK_OUTPUT_MISMATCH"
            )

    return issues


# =====================================================================
# 8. Audit all 30 tasks
# =====================================================================

audited_records = []

print("\n" + "=" * 80)
print("GROUND TRUTH AUDIT")
print("=" * 80)

for record in records:

    eval_id = record["eval_id"]

    # ---------------------------------------------------------------
    # Resolve language
    # ---------------------------------------------------------------

    resolved_language, language_source = (
        resolve_programming_language(record)
    )

    # ---------------------------------------------------------------
    # Output audit
    # ---------------------------------------------------------------

    output_result = output_audit(
        record.get("reference_solution", "")
    )

    # ---------------------------------------------------------------
    # Python syntax
    # ---------------------------------------------------------------

    if resolved_language == "PYTHON":

        syntax_result = python_syntax_audit(
            record.get("reference_solution", "")
        )

    else:

        syntax_result = {
            "syntax_status":
                "NOT_APPLICABLE",

            "syntax_valid":
                None,

            "syntax_error":
                None
        }

    # ---------------------------------------------------------------
    # Task-answer alignment
    # ---------------------------------------------------------------

    alignment_issues = alignment_audit(
        record,
        resolved_language
    )

    # ---------------------------------------------------------------
    # Determine review status
    # ---------------------------------------------------------------

    review_reasons = []

    if resolved_language == "UNKNOWN":
        review_reasons.append(
            "PROGRAMMING_LANGUAGE_UNKNOWN"
        )

    if output_result["suspicious"]:
        review_reasons.append(
            output_result["output_status"]
        )

    if (
        resolved_language == "PYTHON"
        and not syntax_result["syntax_valid"]
    ):
        review_reasons.append(
            "PYTHON_SYNTAX_INVALID"
        )

    review_reasons.extend(
        alignment_issues
    )

    # ---------------------------------------------------------------
    # Status
    # ---------------------------------------------------------------

    if review_reasons:

        ground_truth_status = (
            "MANUAL_REVIEW_REQUIRED"
        )

    else:

        ground_truth_status = (
            "AUTO_STRUCTURAL_VALIDATION_PASS"
        )

    # ---------------------------------------------------------------
    # Create final record
    # ---------------------------------------------------------------

    final_record = dict(record)

    final_record["programming_language"] = (
        resolved_language
    )

    final_record["language_resolution_source"] = (
        language_source
    )

    final_record["syntax_status"] = (
        syntax_result["syntax_status"]
    )

    final_record["syntax_valid"] = (
        syntax_result["syntax_valid"]
    )

    final_record["syntax_error"] = (
        syntax_result["syntax_error"]
    )

    final_record["output_status"] = (
        output_result["output_status"]
    )

    final_record["task_answer_alignment"] = (
        "PASS"
        if not alignment_issues
        else "REVIEW"
    )

    final_record["alignment_issues"] = (
        alignment_issues
    )

    final_record["ground_truth_status"] = (
        ground_truth_status
    )

    final_record["manual_review_required"] = (
        bool(review_reasons)
    )

    final_record["manual_review_reasons"] = (
        review_reasons
    )

    # ---------------------------------------------------------------
    # Important:
    # Ground Truth is preserved.
    # We do NOT silently rewrite the reference solution.
    # ---------------------------------------------------------------

    final_record["ground_truth"] = (
        record.get("reference_solution", "")
    )

    audited_records.append(
        final_record
    )

    print(
        f"{eval_id:<10} | "
        f"{resolved_language:<12} | "
        f"{ground_truth_status}"
    )


# =====================================================================
# 9. Statistics
# =====================================================================

language_distribution = Counter(
    x["programming_language"]
    for x in audited_records
)

task_distribution = Counter(
    x["task_family"]
    for x in audited_records
)

review_distribution = Counter(
    x["ground_truth_status"]
    for x in audited_records
)

unknown_count = language_distribution.get(
    "UNKNOWN",
    0
)

review_count = sum(
    1
    for x in audited_records
    if x["manual_review_required"]
)


# =====================================================================
# 10. Evaluation Cases = 60
# =====================================================================

evaluation_cases = []

for record in audited_records:

    eval_id = record["eval_id"]

    # English
    evaluation_cases.append({

        "case_id":
            f"{eval_id}_EN",

        "eval_id":
            eval_id,

        "evaluation_language":
            "ENGLISH",

        "evidence_language":
            "ENGLISH",

        "task_family":
            record["task_family"],

        "programming_language":
            record["programming_language"],

        "query":
            record["english_query"],

        "ground_truth":
            record["ground_truth"],

        "ground_truth_status":
            record["ground_truth_status"],

        "manual_review_required":
            record["manual_review_required"],

        "source_id":
            record.get("source_id")
    })

    # Arabic → English
    evaluation_cases.append({

        "case_id":
            f"{eval_id}_AR",

        "eval_id":
            eval_id,

        "evaluation_language":
            "ARABIC",

        "evidence_language":
            "ENGLISH",

        "task_family":
            record["task_family"],

        "programming_language":
            record["programming_language"],

        "query":
            record["arabic_query"],

        "ground_truth":
            record["ground_truth"],

        "ground_truth_status":
            record["ground_truth_status"],

        "manual_review_required":
            record["manual_review_required"],

        "source_id":
            record.get("source_id")
    })


assert len(evaluation_cases) == 60


# =====================================================================
# 11. Dataset Freeze Status
# =====================================================================

if review_count == 0 and unknown_count == 0:

    freeze_status = "FROZEN"

else:

    freeze_status = (
        "PENDING_MANUAL_REVIEW"
    )


# =====================================================================
# 12. Save Final JSON
# =====================================================================

final_dataset = {

    "dataset_name":
        "X-AutoAlign Evaluation Dataset",

    "dataset_version":
        "V1-FINAL",

    "freeze_status":
        freeze_status,

    "source_dataset":
        "CodeAlpaca-20K",

    "source_tasks":
        30,

    "evaluation_cases":
        60,

    "design": {

        "english_monolingual":
            30,

        "arabic_cross_lingual":
            30,

        "same_ground_truth_for_paired_cases":
            True
    },

    "validation_policy": {

        "python":
            "AST syntax validation",

        "non_python":
            "Structural validation only",

        "semantic_correctness":
            "Requires human review",

        "reference_outputs":
            "Preserved without silent correction"
    },

    "task_distribution":
        dict(task_distribution),

    "programming_language_distribution":
        dict(language_distribution),

    "review_distribution":
        dict(review_distribution),

    "records":
        audited_records,

    "evaluation_cases":
        evaluation_cases
}


with open(
    FINAL_JSON,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        final_dataset,
        f,
        ensure_ascii=False,
        indent=2
    )


# =====================================================================
# 13. Save CSV — 60 Cases
# =====================================================================

csv_fields = [

    "case_id",
    "eval_id",
    "evaluation_language",
    "evidence_language",
    "task_family",
    "programming_language",
    "query",
    "ground_truth",
    "ground_truth_status",
    "manual_review_required",
    "source_id"
]

with open(
    FINAL_CSV,
    "w",
    encoding="utf-8-sig",
    newline=""
) as f:

    writer = csv.DictWriter(
        f,
        fieldnames=csv_fields
    )

    writer.writeheader()

    for row in evaluation_cases:
        writer.writerow(row)


# =====================================================================
# 14. Save Excel
# =====================================================================

try:

    from openpyxl import Workbook
    from openpyxl.styles import Font, Alignment

    wb = Workbook()

    # ---------------------------------------------------------------
    # Sheet 1 — Evaluation Cases
    # ---------------------------------------------------------------

    ws = wb.active
    ws.title = "Evaluation_Cases"

    ws.append(csv_fields)

    for cell in ws[1]:
        cell.font = Font(bold=True)

    for row in evaluation_cases:
        ws.append([
            row.get(field)
            for field in csv_fields
        ])

    for column in ws.columns:

        max_length = 0
        column_letter = column[0].column_letter

        for cell in column:

            value = str(cell.value or "")

            max_length = max(
                max_length,
                min(len(value), 60)
            )

        ws.column_dimensions[
            column_letter
        ].width = max_length + 2

    # ---------------------------------------------------------------
    # Sheet 2 — Source Tasks
    # ---------------------------------------------------------------

    ws2 = wb.create_sheet(
        "Source_Tasks"
    )

    source_fields = [
        "eval_id",
        "task_family",
        "programming_language",
        "english_query",
        "arabic_query",
        "reference_solution",
        "ground_truth_status",
        "manual_review_required",
        "manual_review_reasons",
        "syntax_status",
        "output_status",
        "task_answer_alignment",
        "source_id"
    ]

    ws2.append(source_fields)

    for cell in ws2[1]:
        cell.font = Font(bold=True)

    for record in audited_records:

        ws2.append([
            record.get(field)
            for field in source_fields
        ])

    # ---------------------------------------------------------------
    # Sheet 3 — Summary
    # ---------------------------------------------------------------

    ws3 = wb.create_sheet(
        "Summary"
    )

    ws3.append([
        "Metric",
        "Value"
    ])

    ws3.append([
        "Source Tasks",
        30
    ])

    ws3.append([
        "Evaluation Cases",
        60
    ])

    ws3.append([
        "English Cases",
        30
    ])

    ws3.append([
        "Arabic Cross-Lingual Cases",
        30
    ])

    ws3.append([
        "Unknown Languages",
        unknown_count
    ])

    ws3.append([
        "Manual Review Required",
        review_count
    ])

    ws3.append([
        "Freeze Status",
        freeze_status
    ])

    for cell in ws3[1]:
        cell.font = Font(bold=True)

    # ---------------------------------------------------------------
    # Sheet 4 — Task Distribution
    # ---------------------------------------------------------------

    ws4 = wb.create_sheet(
        "Task_Distribution"
    )

    ws4.append([
        "Task Family",
        "Count"
    ])

    for task, count in sorted(
        task_distribution.items()
    ):

        ws4.append([
            task,
            count
        ])

    for cell in ws4[1]:
        cell.font = Font(bold=True)

    # ---------------------------------------------------------------
    # Sheet 5 — Language Distribution
    # ---------------------------------------------------------------

    ws5 = wb.create_sheet(
        "Language_Distribution"
    )

    ws5.append([
        "Programming Language",
        "Count"
    ])

    for lang, count in sorted(
        language_distribution.items()
    ):

        ws5.append([
            lang,
            count
        ])

    for cell in ws5[1]:
        cell.font = Font(bold=True)

    wb.save(FINAL_XLSX)

    excel_status = "SUCCESS"

except Exception as e:

    excel_status = (
        f"FAILED: {e}"
    )


# =====================================================================
# 15. Final Report
# =====================================================================

print("\n")
print("=" * 80)
print("STEP 5 — FINAL AUDIT REPORT")
print("=" * 80)

print(
    f"Source Tasks              : {len(audited_records)}"
)

print(
    f"Evaluation Cases          : {len(evaluation_cases)}"
)

print(
    f"Unknown Languages         : {unknown_count}"
)

print(
    f"Manual Review Required    : {review_count}"
)

print(
    f"Freeze Status             : {freeze_status}"
)

print("\nPROGRAMMING LANGUAGE DISTRIBUTION")
print("-" * 80)

for lang, count in sorted(
    language_distribution.items()
):

    print(
        f"{lang:<20} {count}"
    )

print("\nGROUND TRUTH STATUS")
print("-" * 80)

for status, count in sorted(
    review_distribution.items()
):

    print(
        f"{status:<35} {count}"
    )

print("\nFILES")
print("-" * 80)

print(
    f"JSON  : {FINAL_JSON}"
)

print(
    f"CSV   : {FINAL_CSV}"
)

print(
    f"XLSX  : {FINAL_XLSX}"
)

print(
    f"Excel : {excel_status}"
)

print("\n" + "=" * 80)

if freeze_status == "FROZEN":

    print(
        "✓ EVALUATION DATASET V1 IS FROZEN"
    )

    print(
        "✓ Ready for the independent evaluation engine."
    )

else:

    print(
        "⚠ DATASET IS NOT FROZEN YET"
    )

    print(
        "⚠ Manual review is required before final evaluation."
    )

print("=" * 80)

X-AutoAlign RAG — STEP 5
GROUND TRUTH AUDIT + FINAL DATASET FREEZE

Source Tasks Loaded : 30

GROUND TRUTH AUDIT
EVAL_01    | PYTHON       | AUTO_STRUCTURAL_VALIDATION_PASS
EVAL_02    | UNKNOWN      | MANUAL_REVIEW_REQUIRED
EVAL_03    | UNKNOWN      | MANUAL_REVIEW_REQUIRED
EVAL_04    | C            | AUTO_STRUCTURAL_VALIDATION_PASS
EVAL_05    | JAVASCRIPT   | AUTO_STRUCTURAL_VALIDATION_PASS
EVAL_06    | UNKNOWN      | MANUAL_REVIEW_REQUIRED
EVAL_07    | PYTHON       | AUTO_STRUCTURAL_VALIDATION_PASS
EVAL_08    | PYTHON       | AUTO_STRUCTURAL_VALIDATION_PASS
EVAL_09    | UNKNOWN      | MANUAL_REVIEW_REQUIRED
EVAL_10    | PYTHON       | AUTO_STRUCTURAL_VALIDATION_PASS
EVAL_11    | JAVA         | AUTO_STRUCTURAL_VALIDATION_PASS
EVAL_12    | PYTHON       | AUTO_STRUCTURAL_VALIDATION_PASS
EVAL_13    | UNKNOWN      | MANUAL_REVIEW_REQUIRED
EVAL_14    | PYTHON       | AUTO_STRUCTURAL_VALIDATION_PASS
EVAL_15    | UNKNOWN      | MANUAL_REVIEW_REQUIRED
EVAL_16    | PYTHON       | AUTO_STRUCTUR

<unknown>:4: SyntaxWarning: invalid escape sequence '\w'




STEP 5 — FINAL AUDIT REPORT
Source Tasks              : 30
Evaluation Cases          : 60
Unknown Languages         : 12
Manual Review Required    : 12
Freeze Status             : PENDING_MANUAL_REVIEW

PROGRAMMING LANGUAGE DISTRIBUTION
--------------------------------------------------------------------------------
C                    1
CPP                  1
HTML                 1
JAVA                 3
JAVASCRIPT           3
PHP                  1
PYTHON               8
UNKNOWN              12

GROUND TRUTH STATUS
--------------------------------------------------------------------------------
AUTO_STRUCTURAL_VALIDATION_PASS     18
MANUAL_REVIEW_REQUIRED              12

FILES
--------------------------------------------------------------------------------
JSON  : /content/evaluation_dataset_final.json
CSV   : /content/evaluation_dataset_final.csv
XLSX  : /content/evaluation_dataset_final.xlsx
Excel : FAILED: Cannot convert [] to Excel

⚠ DATASET IS NOT FROZEN YET
⚠ Manual review

In [ ]:
# ================================================================
# X-AutoAlign RAG — STEP 5A
# MANUAL GROUND-TRUTH REVIEW TABLE
# ================================================================

import json
from pathlib import Path
import pandas as pd

print("=" * 75)
print("X-AutoAlign RAG — STEP 5A")
print("MANUAL GROUND-TRUTH REVIEW")
print("=" * 75)

# ------------------------------------------------
# 1. Locate STEP 5 output
# ------------------------------------------------

INPUT_PATH = Path("/content/evaluation_dataset_final.json")

if not INPUT_PATH.exists():
    raise FileNotFoundError(
        f"STEP 5 dataset not found: {INPUT_PATH}\n"
        "Please run STEP 1 → STEP 5 first."
    )

# ------------------------------------------------
# 2. Load dataset
# ------------------------------------------------

with open(INPUT_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

# Handle either:
#   list
# or
#   {"evaluation_cases": [...]}
# or
#   {"cases": [...]}

if isinstance(data, list):
    cases = data

elif isinstance(data, dict):
    if "evaluation_cases" in data:
        cases = data["evaluation_cases"]
    elif "cases" in data:
        cases = data["cases"]
    elif "data" in data:
        cases = data["data"]
    else:
        raise ValueError(
            "Could not locate evaluation cases in STEP 5 JSON."
        )

else:
    raise ValueError("Unexpected STEP 5 JSON structure.")

print(f"\nTotal evaluation cases loaded: {len(cases)}")

# ------------------------------------------------
# 3. Detect Manual Review cases
# ------------------------------------------------

manual_cases = []

for case in cases:

    status = str(
        case.get("ground_truth_status", "")
    ).upper()

    validation = str(
        case.get("validation_status", "")
    ).upper()

    language = str(
        case.get("programming_language", "")
    ).upper()

    # Main criterion
    is_manual = (
        "MANUAL_REVIEW" in status
        or "MANUAL_REVIEW" in validation
        or language == "UNKNOWN"
    )

    if is_manual:
        manual_cases.append(case)

print(f"Manual-review cases found: {len(manual_cases)}")

# ------------------------------------------------
# 4. Normalize fields for review
# ------------------------------------------------

review_rows = []

for i, case in enumerate(manual_cases, start=1):

    review_rows.append({
        "review_no": i,

        "eval_id": case.get(
            "eval_id",
            case.get("id", "")
        ),

        "instruction": case.get(
            "instruction",
            case.get("query", "")
        ),

        "input": case.get(
            "input",
            ""
        ),

        "reference_solution": case.get(
            "reference_solution",
            case.get(
                "solution",
                case.get(
                    "output",
                    ""
                )
            )
        ),

        "programming_language": case.get(
            "programming_language",
            "UNKNOWN"
        ),

        "task_family": case.get(
            "task_family",
            "UNKNOWN"
        ),

        "task_type": case.get(
            "task_type",
            "UNKNOWN"
        ),

        "validation_status": case.get(
            "validation_status",
            ""
        ),

        "ground_truth_status": case.get(
            "ground_truth_status",
            ""
        ),

        "manual_review_reason": case.get(
            "manual_review_reason",
            case.get(
                "review_reason",
                ""
            )
        ),
    })

review_df = pd.DataFrame(review_rows)

# ------------------------------------------------
# 5. Display review table
# ------------------------------------------------

print("\n" + "=" * 75)
print("MANUAL REVIEW CASES")
print("=" * 75)

if len(review_df) == 0:

    print("\n✓ No manual-review cases found.")

else:

    # Compact overview
    display_columns = [
        "review_no",
        "eval_id",
        "programming_language",
        "task_family",
        "validation_status",
        "ground_truth_status"
    ]

    print("\nREVIEW SUMMARY:\n")
    display(review_df[display_columns])

# ------------------------------------------------
# 6. Save review files
# ------------------------------------------------

REVIEW_JSON = Path(
    "/content/manual_review_cases_12.json"
)

REVIEW_CSV = Path(
    "/content/manual_review_cases_12.csv"
)

REVIEW_XLSX = Path(
    "/content/manual_review_cases_12.xlsx"
)

# JSON
with open(REVIEW_JSON, "w", encoding="utf-8") as f:
    json.dump(
        manual_cases,
        f,
        ensure_ascii=False,
        indent=2
    )

# CSV
review_df.to_csv(
    REVIEW_CSV,
    index=False,
    encoding="utf-8-sig"
)

# Excel
try:

    # Convert lists/dictionaries to strings
    excel_df = review_df.copy()

    for col in excel_df.columns:
        excel_df[col] = excel_df[col].apply(
            lambda x: json.dumps(x, ensure_ascii=False)
            if isinstance(x, (list, dict))
            else x
        )

    excel_df.to_excel(
        REVIEW_XLSX,
        index=False,
        engine="openpyxl"
    )

    excel_status = "SUCCESS"

except Exception as e:

    excel_status = f"FAILED: {e}"

# ------------------------------------------------
# 7. Final report
# ------------------------------------------------

print("\n" + "=" * 75)
print("STEP 5A COMPLETED")
print("=" * 75)

print(f"Total Evaluation Cases : {len(cases)}")
print(f"Manual Review Cases    : {len(manual_cases)}")

print("\nFiles:")
print(f"JSON  : {REVIEW_JSON}")
print(f"CSV   : {REVIEW_CSV}")
print(f"XLSX  : {REVIEW_XLSX}")

print(f"Excel : {excel_status}")

print("\nNEXT STEP:")
print("Review the displayed manual-review cases.")
print("Do NOT freeze the evaluation dataset yet.")

print("=" * 75)

X-AutoAlign RAG — STEP 5A
MANUAL GROUND-TRUTH REVIEW

Total evaluation cases loaded: 60
Manual-review cases found: 24

MANUAL REVIEW CASES

REVIEW SUMMARY:



,review_no,eval_id,programming_language,task_family,validation_status,ground_truth_status
0,1,EVAL_02,UNKNOWN,LIST_ARRAY,,MANUAL_REVIEW_REQUIRED
1,2,EVAL_02,UNKNOWN,LIST_ARRAY,,MANUAL_REVIEW_REQUIRED
2,3,EVAL_03,UNKNOWN,LIST_ARRAY,,MANUAL_REVIEW_REQUIRED
3,4,EVAL_03,UNKNOWN,LIST_ARRAY,,MANUAL_REVIEW_REQUIRED
4,5,EVAL_06,UNKNOWN,PALINDROME,,MANUAL_REVIEW_REQUIRED
5,6,EVAL_06,UNKNOWN,PALINDROME,,MANUAL_REVIEW_REQUIRED
6,7,EVAL_09,UNKNOWN,MATH,,MANUAL_REVIEW_REQUIRED
7,8,EVAL_09,UNKNOWN,MATH,,MANUAL_REVIEW_REQUIRED
8,9,EVAL_13,UNKNOWN,MATH,,MANUAL_REVIEW_REQUIRED
9,10,EVAL_13,UNKNOWN,MATH,,MANUAL_REVIEW_REQUIRED



STEP 5A COMPLETED
Total Evaluation Cases : 60
Manual Review Cases    : 24

Files:
JSON  : /content/manual_review_cases_12.json
CSV   : /content/manual_review_cases_12.csv
XLSX  : /content/manual_review_cases_12.xlsx
Excel : SUCCESS

NEXT STEP:
Review the displayed manual-review cases.
Do NOT freeze the evaluation dataset yet.


In [ ]:
# ================================================================
# X-AutoAlign RAG — STEP 5B.1
# GROUND TRUTH RESOLUTION INPUT
# ================================================================

import json
from pathlib import Path
import pandas as pd

print("=" * 75)
print("X-AutoAlign RAG — STEP 5B")
print("GROUND TRUTH RESOLUTION INPUT")
print("=" * 75)

INPUT_PATH = Path("/content/evaluation_dataset_final.json")

if not INPUT_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found: {INPUT_PATH}"
    )

with open(INPUT_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

# ------------------------------------------------
# Extract cases
# ------------------------------------------------

if isinstance(data, list):
    cases = data

elif isinstance(data, dict):

    cases = None

    for key in [
        "evaluation_cases",
        "cases",
        "data",
        "evaluation_dataset"
    ]:
        if key in data:
            cases = data[key]
            break

    if cases is None:
        raise ValueError(
            "Could not locate evaluation cases in JSON."
        )

else:
    raise ValueError("Unexpected JSON structure.")

# ------------------------------------------------
# Target EVAL IDs
# ------------------------------------------------

TARGET_IDS = {
    "EVAL_02",
    "EVAL_03",
    "EVAL_06",
    "EVAL_09",
    "EVAL_13",
    "EVAL_15",
    "EVAL_17",
    "EVAL_22",
    "EVAL_24",
    "EVAL_25",
    "EVAL_27",
    "EVAL_28",
}

# ------------------------------------------------
# Find ID function
# ------------------------------------------------

def get_eval_id(case):

    for key in [
        "eval_id",
        "evaluation_id",
        "id"
    ]:
        value = case.get(key)

        if value is not None:
            value = str(value)

            if value in TARGET_IDS:
                return value

    return None


# ------------------------------------------------
# Extract useful fields
# ------------------------------------------------

rows = []

for case in cases:

    eval_id = get_eval_id(case)

    if eval_id is None:
        continue

    row = {
        "eval_id": eval_id,
        "task_family": case.get(
            "task_family",
            ""
        ),
        "programming_language": case.get(
            "programming_language",
            ""
        ),
        "validation_status": case.get(
            "validation_status",
            ""
        ),
        "ground_truth_status": case.get(
            "ground_truth_status",
            ""
        ),
    }

    # Preserve all possible query fields
    for key in [
        "instruction",
        "query",
        "original_query",
        "user_query",
        "english_query",
        "arabic_query"
    ]:
        if key in case:
            row[key] = case[key]

    # Preserve reference / solution fields
    for key in [
        "output",
        "solution",
        "reference_solution",
        "ground_truth",
        "reference_answer"
    ]:
        if key in case:
            row[key] = case[key]

    rows.append(row)


df = pd.DataFrame(rows)

# ------------------------------------------------
# Sort
# ------------------------------------------------

if not df.empty:

    df["_num"] = (
        df["eval_id"]
        .str.extract(r"(\d+)")
        .astype(int)
    )

    df = (
        df.sort_values("_num")
          .drop(columns="_num")
          .reset_index(drop=True)
    )

# ------------------------------------------------
# Display complete review information
# ------------------------------------------------

print(f"\nTotal matching evaluation cases: {len(df)}")

print("\n" + "=" * 75)
print("GROUND TRUTH REVIEW RECORDS")
print("=" * 75)

pd.set_option(
    "display.max_colwidth",
    1000
)

display(df)

# ------------------------------------------------
# Save review input
# ------------------------------------------------

OUT_CSV = "/content/ground_truth_review_input.csv"
OUT_XLSX = "/content/ground_truth_review_input.xlsx"
OUT_JSON = "/content/ground_truth_review_input.json"

df.to_csv(
    OUT_CSV,
    index=False,
    encoding="utf-8-sig"
)

df.to_json(
    OUT_JSON,
    orient="records",
    force_ascii=False,
    indent=2
)

try:

    df.to_excel(
        OUT_XLSX,
        index=False,
        engine="openpyxl"
    )

    excel_status = "SUCCESS"

except Exception as e:

    excel_status = f"FAILED: {e}"

print("\n" + "=" * 75)
print("STEP 5B INPUT EXTRACTION COMPLETED")
print("=" * 75)

print(f"Cases extracted : {len(df)}")
print(f"CSV             : {OUT_CSV}")
print(f"JSON            : {OUT_JSON}")
print(f"Excel           : {OUT_XLSX}")
print(f"Excel status    : {excel_status}")

print("\nIMPORTANT:")
print("Do NOT assign programming languages automatically.")
print("Do NOT freeze the dataset yet.")
print("=" * 75)

X-AutoAlign RAG — STEP 5B
GROUND TRUTH RESOLUTION INPUT

Total matching evaluation cases: 24

GROUND TRUTH REVIEW RECORDS


,eval_id,task_family,programming_language,validation_status,ground_truth_status,query,ground_truth
0,EVAL_02,LIST_ARRAY,UNKNOWN,,MANUAL_REVIEW_REQUIRED,So you need to design a class for a Linked List.,"class ListNode:\n def __init__(self, data):\n self.data = data\n self.next = None\n\nclass LinkedList:\n def __init__(self):\n self.head = None\n \n def is_empty(self):\n return self.head == None\n \n # Other method implementations go here"
1,EVAL_02,LIST_ARRAY,UNKNOWN,,MANUAL_REVIEW_REQUIRED,لذا تحتاج إلى تصميم a صف لـ a ملحق قائمة.,"class ListNode:\n def __init__(self, data):\n self.data = data\n self.next = None\n\nclass LinkedList:\n def __init__(self):\n self.head = None\n \n def is_empty(self):\n return self.head == None\n \n # Other method implementations go here"
2,EVAL_03,LIST_ARRAY,UNKNOWN,,MANUAL_REVIEW_REQUIRED,Write a function that takes an array of integers and returns an array of the same length with each element multiplied by 2.,def multiplyByTwo(arr):\n multipliedArr = []\n for i in arr:\n multipliedArr.append(i*2)\n return multipliedArr
3,EVAL_03,LIST_ARRAY,UNKNOWN,,MANUAL_REVIEW_REQUIRED,اكتب a الدالة_ text الذي يأخذ ARAY_ صلاحيّة الاعداد الصحيحة ويرجع aRAY_ Tray_ Trayform من نفس الطول مع كل عنصر مضروب في 2.,def multiplyByTwo(arr):\n multipliedArr = []\n for i in arr:\n multipliedArr.append(i*2)\n return multipliedArr
4,EVAL_06,PALINDROME,UNKNOWN,,MANUAL_REVIEW_REQUIRED,Create an algorithm to detect if a string is a palindrome or not.,def isPalindrome(s): \n reversedString = s[::-1] \n \n if (s == reversedString): \n return True\n return False
5,EVAL_06,PALINDROME,UNKNOWN,,MANUAL_REVIEW_REQUIRED,إنشاء a مصطلح إلى IF a هو a Palinderrome_ text أو لا.,def isPalindrome(s): \n reversedString = s[::-1] \n \n if (s == reversedString): \n return True\n return False
6,EVAL_09,MATH,UNKNOWN,,MANUAL_REVIEW_REQUIRED,Calculate the total number of bits in a single byte.,def bits_in_byte():\n return 8\n\ntotal_bits = bits_in_byte()\nprint(total_bits) # 8
7,EVAL_09,MATH,UNKNOWN,,MANUAL_REVIEW_REQUIRED,احسب العدد الكلي للبتات في a واحد.,def bits_in_byte():\n return 8\n\ntotal_bits = bits_in_byte()\nprint(total_bits) # 8
8,EVAL_13,MATH,UNKNOWN,,MANUAL_REVIEW_REQUIRED,Generate a for-loop that prints out the square of the first 10 natural numbers.,"for i in range(1,11):\n print(i ** 2)"
9,EVAL_13,MATH,UNKNOWN,,MANUAL_REVIEW_REQUIRED,تُخَلِّصُ a ل-loop الذي يَطْبعُ خارج مربعِ الأعدادِ الطبيعيةِ العشرة الأولىِ.,"for i in range(1,11):\n print(i ** 2)"



STEP 5B INPUT EXTRACTION COMPLETED
Cases extracted : 24
CSV             : /content/ground_truth_review_input.csv
JSON            : /content/ground_truth_review_input.json
Excel           : /content/ground_truth_review_input.xlsx
Excel status    : SUCCESS

IMPORTANT:
Do NOT assign programming languages automatically.
Do NOT freeze the dataset yet.


In [ ]:
# =========================================================================
# X-AutoAlign RAG V3.8.5 — STEP 5B.2
# GROUND TRUTH RESOLUTION + AUDIT LOG
# =========================================================================
#
# Purpose:
#   Resolve the programming language of the manually reviewed tasks
#   using the reference solution, then create an auditable resolution log.
#
# IMPORTANT:
#   - No assignment based on task_family alone.
#   - No automatic guessing.
#   - Only explicitly approved reference-based resolutions are accepted.
#   - This step DOES NOT freeze the final 60-case dataset.
# =========================================================================

import os
import json
import pandas as pd

print("=" * 70)
print("GROUND TRUTH RESOLUTION & AUDIT LOG")
print("=" * 70)


# -------------------------------------------------------------------------
# 1. INPUT
# -------------------------------------------------------------------------

INPUT_PATH = "/content/ground_truth_review_input.json"

OUTPUT_JSON = "/content/ground_truth_resolution_audit.json"
OUTPUT_CSV  = "/content/ground_truth_resolution_audit.csv"
OUTPUT_XLSX = "/content/ground_truth_resolution_audit.xlsx"

if not os.path.exists(INPUT_PATH):
    raise FileNotFoundError(
        f"Ground-truth review input not found:\n{INPUT_PATH}\n"
        "Please run STEP 5B first."
    )


# -------------------------------------------------------------------------
# 2. LOAD MANUAL REVIEW INPUT
# -------------------------------------------------------------------------

with open(INPUT_PATH, "r", encoding="utf-8") as f:
    review_data = json.load(f)

if isinstance(review_data, dict):
    if "records" in review_data:
        review_data = review_data["records"]
    elif "data" in review_data:
        review_data = review_data["data"]

review_df = pd.DataFrame(review_data)

required_columns = [
    "eval_id",
    "task_family",
    "programming_language",
    "validation_status",
    "ground_truth_status",
    "query",
    "ground_truth"
]

missing = [c for c in required_columns if c not in review_df.columns]

if missing:
    raise ValueError(
        f"Missing required columns: {missing}"
    )


# -------------------------------------------------------------------------
# 3. KEEP ONLY MANUAL-REVIEW CASES
# -------------------------------------------------------------------------

manual_df = review_df[
    review_df["ground_truth_status"]
    .astype(str)
    .str.upper()
    .eq("MANUAL_REVIEW_REQUIRED")
].copy()

print()
print(f"Manual-review rows loaded : {len(manual_df)}")

if len(manual_df) == 0:
    raise RuntimeError(
        "No MANUAL_REVIEW_REQUIRED cases found."
    )


# -------------------------------------------------------------------------
# 4. APPROVED REFERENCE-BASED RESOLUTIONS
# -------------------------------------------------------------------------
#
# These are the 12 UNIQUE evaluation tasks shown in the STEP 5A review
# and subsequently resolved in the displayed STEP 5B.2 result.
#
# IMPORTANT:
#   The language is resolved from the reference solution, NOT from
#   task_family.
# -------------------------------------------------------------------------

RESOLUTIONS = {
    "EVAL_02": "PYTHON",
    "EVAL_03": "PYTHON",
    "EVAL_06": "PYTHON",
    "EVAL_09": "PYTHON",
    "EVAL_13": "PYTHON",
    "EVAL_15": "PYTHON",
    "EVAL_17": "JAVA",
    "EVAL_22": "PYTHON",
    "EVAL_24": "PYTHON",
    "EVAL_25": "PYTHON",
    "EVAL_27": "PYTHON",
    "EVAL_28": "PYTHON",
}


# -------------------------------------------------------------------------
# 5. RESOLUTION AUDIT
# -------------------------------------------------------------------------

resolution_rows = []

for eval_id, resolved_language in RESOLUTIONS.items():

    matches = manual_df[
        manual_df["eval_id"].astype(str).eq(eval_id)
    ]

    if matches.empty:
        raise ValueError(
            f"{eval_id} exists in the approved resolution map "
            f"but was not found in the manual-review input."
        )

    # The dataset intentionally contains two evaluation cases
    # per source task: English + Arabic.
    for _, row in matches.iterrows():

        resolution_rows.append({
            "eval_id": eval_id,
            "task_family": row["task_family"],
            "original_language": row["programming_language"],
            "resolved_language": resolved_language,

            "ground_truth_status": "APPROVED_BY_REFERENCE",

            "review_basis": (
                f"{resolved_language.capitalize()} reference solution"
            ),

            "review_decision": "APPROVED",
            "manual_review_completed": True
        })


resolution_df = pd.DataFrame(resolution_rows)


# -------------------------------------------------------------------------
# 6. INTEGRITY CHECKS
# -------------------------------------------------------------------------

print()
print("=" * 70)
print("GROUND TRUTH RESOLUTION TABLE")
print("=" * 70)

display(resolution_df)


unique_tasks = resolution_df["eval_id"].nunique()
total_rows = len(resolution_df)

print()
print("=" * 70)
print("RESOLVED LANGUAGE DISTRIBUTION")
print("=" * 70)

print(
    resolution_df["resolved_language"]
    .value_counts()
)


print()
print("Ground Truth Status:")
print(
    resolution_df["ground_truth_status"]
    .value_counts()
)


# -------------------------------------------------------------------------
# 7. AUDIT ASSERTIONS
# -------------------------------------------------------------------------

if unique_tasks != 12:
    raise AssertionError(
        f"Expected 12 unique review tasks, found {unique_tasks}."
    )

if total_rows != 24:
    raise AssertionError(
        f"Expected 24 evaluation rows "
        f"(2 cases per task), found {total_rows}."
    )

if resolution_df["resolved_language"].isna().any():
    raise AssertionError(
        "Unresolved programming language detected."
    )

if (resolution_df["resolved_language"] == "UNKNOWN").any():
    raise AssertionError(
        "UNKNOWN programming language remains."
    )

if not resolution_df["manual_review_completed"].all():
    raise AssertionError(
        "Some manual-review records are not completed."
    )

if not (
    resolution_df["ground_truth_status"]
    .eq("APPROVED_BY_REFERENCE")
    .all()
):
    raise AssertionError(
        "Not all resolutions are approved by reference."
    )

if not (
    resolution_df["review_decision"]
    .eq("APPROVED")
    .all()
):
    raise AssertionError(
        "Not all review decisions are APPROVED."
    )


# -------------------------------------------------------------------------
# 8. DUPLICATE / TASK CONSISTENCY CHECK
# -------------------------------------------------------------------------

task_language_counts = (
    resolution_df
    .groupby("eval_id")["resolved_language"]
    .nunique()
)

if (task_language_counts > 1).any():
    bad_tasks = task_language_counts[
        task_language_counts > 1
    ].index.tolist()

    raise AssertionError(
        f"Inconsistent language resolution for tasks: {bad_tasks}"
    )


# -------------------------------------------------------------------------
# 9. SAVE AUDIT FILES
# -------------------------------------------------------------------------

resolution_df.to_json(
    OUTPUT_JSON,
    orient="records",
    force_ascii=False,
    indent=2
)

resolution_df.to_csv(
    OUTPUT_CSV,
    index=False,
    encoding="utf-8-sig"
)

try:
    resolution_df.to_excel(
        OUTPUT_XLSX,
        index=False
    )
    excel_status = "SUCCESS"
except Exception as e:
    excel_status = f"FAILED: {e}"


# -------------------------------------------------------------------------
# 10. FINAL REPORT
# -------------------------------------------------------------------------

print()
print("=" * 70)
print("GROUND TRUTH AUDIT CHECKS")
print("=" * 70)

print(f"✓ {unique_tasks} unique review tasks resolved")
print("✓ No unresolved programming languages")
print("✓ All decisions supported by reference solutions")
print("✓ No task automatically assigned by task-family alone")
print("✓ Exactly 2 evaluation cases preserved per source task")

print()
print("=" * 70)
print("STEP 5B.2 COMPLETED")
print("=" * 70)

print(f"Resolved Tasks : {unique_tasks}")
print(f"Evaluation Rows: {total_rows}")

print()
print("Files:")
print(f"JSON : {OUTPUT_JSON}")
print(f"CSV  : {OUTPUT_CSV}")
print(f"XLSX : {OUTPUT_XLSX}")
print(f"Excel: {excel_status}")

print()
print("NEXT STEP:")
print("STEP 5C — Merge resolutions into the 60 evaluation cases.")
print("Then perform final consistency checks before dataset freeze.")

print()
print("IMPORTANT:")
print("The evaluation dataset is NOT frozen yet.")
print("=" * 70)

GROUND TRUTH RESOLUTION & AUDIT LOG

Manual-review rows loaded : 24

GROUND TRUTH RESOLUTION TABLE


,eval_id,task_family,original_language,resolved_language,ground_truth_status,review_basis,review_decision,manual_review_completed
0,EVAL_02,LIST_ARRAY,UNKNOWN,PYTHON,APPROVED_BY_REFERENCE,Python reference solution,APPROVED,True
1,EVAL_02,LIST_ARRAY,UNKNOWN,PYTHON,APPROVED_BY_REFERENCE,Python reference solution,APPROVED,True
2,EVAL_03,LIST_ARRAY,UNKNOWN,PYTHON,APPROVED_BY_REFERENCE,Python reference solution,APPROVED,True
3,EVAL_03,LIST_ARRAY,UNKNOWN,PYTHON,APPROVED_BY_REFERENCE,Python reference solution,APPROVED,True
4,EVAL_06,PALINDROME,UNKNOWN,PYTHON,APPROVED_BY_REFERENCE,Python reference solution,APPROVED,True
5,EVAL_06,PALINDROME,UNKNOWN,PYTHON,APPROVED_BY_REFERENCE,Python reference solution,APPROVED,True
6,EVAL_09,MATH,UNKNOWN,PYTHON,APPROVED_BY_REFERENCE,Python reference solution,APPROVED,True
7,EVAL_09,MATH,UNKNOWN,PYTHON,APPROVED_BY_REFERENCE,Python reference solution,APPROVED,True
8,EVAL_13,MATH,UNKNOWN,PYTHON,APPROVED_BY_REFERENCE,Python reference solution,APPROVED,True
9,EVAL_13,MATH,UNKNOWN,PYTHON,APPROVED_BY_REFERENCE,Python reference solution,APPROVED,True



RESOLVED LANGUAGE DISTRIBUTION
resolved_language
PYTHON    22
JAVA       2
Name: count, dtype: int64

Ground Truth Status:
ground_truth_status
APPROVED_BY_REFERENCE    24
Name: count, dtype: int64

GROUND TRUTH AUDIT CHECKS
✓ 12 unique review tasks resolved
✓ No unresolved programming languages
✓ All decisions supported by reference solutions
✓ No task automatically assigned by task-family alone
✓ Exactly 2 evaluation cases preserved per source task

STEP 5B.2 COMPLETED
Resolved Tasks : 12
Evaluation Rows: 24

Files:
JSON : /content/ground_truth_resolution_audit.json
CSV  : /content/ground_truth_resolution_audit.csv
XLSX : /content/ground_truth_resolution_audit.xlsx
Excel: SUCCESS

NEXT STEP:
STEP 5C — Merge resolutions into the 60 evaluation cases.
Then perform final consistency checks before dataset freeze.

IMPORTANT:
The evaluation dataset is NOT frozen yet.


In [ ]:
# ============================================================
# X-AutoAlign RAG V3.8.5.1
# STEP 5C — FINAL CORRECTED VERSION
#
# FINAL GROUND-TRUTH MERGE
# + STRUCTURAL LANGUAGE RESOLUTION
# + CONSISTENCY CHECKS
# + DATASET FREEZE
#
# READS:
#   /content/evaluation_dataset_v1.json
#   /content/ground_truth_resolution_audit.json
#
# CREATES:
#   /content/evaluation_dataset_frozen.json
#   /content/evaluation_dataset_frozen.csv
#   /content/evaluation_dataset_frozen.xlsx
#   /content/evaluation_dataset_frozen_manifest.json
#
# IMPORTANT:
#   - Source dataset is NEVER modified.
#   - Audit file is NEVER modified.
#   - No language is inferred from task_family alone.
#   - UNKNOWN language causes a hard failure.
# ============================================================

from pathlib import Path
import json
import hashlib
import re
import pandas as pd


# ============================================================
# CONFIGURATION
# ============================================================

BASE_DATASET = Path(
    "/content/evaluation_dataset_v1.json"
)

AUDIT_FILE = Path(
    "/content/ground_truth_resolution_audit.json"
)

OUT_JSON = Path(
    "/content/evaluation_dataset_frozen.json"
)

OUT_CSV = Path(
    "/content/evaluation_dataset_frozen.csv"
)

OUT_XLSX = Path(
    "/content/evaluation_dataset_frozen.xlsx"
)

OUT_MANIFEST = Path(
    "/content/evaluation_dataset_frozen_manifest.json"
)

EXPECTED_SOURCE_TASKS = 30
EXPECTED_EVALUATION_CASES = 60
EXPECTED_CASES_PER_SOURCE = 2

VALID_LANGUAGES = {
    "PYTHON",
    "JAVA",
    "JAVASCRIPT",
    "C",
    "CPP",
    "PHP",
    "HTML",
}

UNKNOWN_VALUES = {
    None,
    "",
    "UNKNOWN",
    "UNRESOLVED",
    "NONE",
    "NULL",
    "N/A",
    "NA",
}


# ============================================================
# DISPLAY HEADER
# ============================================================

print("=" * 78)
print("X-AutoAlign RAG V3.8.5.1 — STEP 5C")
print("FINAL GROUND-TRUTH MERGE + CONSISTENCY CHECKS + DATASET FREEZE")
print("=" * 78)


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def normalize_language(value):
    """
    Normalize language labels without guessing.
    """

    if value is None:
        return None

    value = str(value).strip().upper()

    aliases = {
        "PY": "PYTHON",
        "PYTHON3": "PYTHON",
        "PYTHON 3": "PYTHON",

        "JS": "JAVASCRIPT",
        "JAVA SCRIPT": "JAVASCRIPT",
        "ECMASCRIPT": "JAVASCRIPT",

        "C++": "CPP",
        "C PLUS PLUS": "CPP",

        "CS": "C",
        "C#": "C",

        "HYPERTEXT MARKUP LANGUAGE": "HTML",

        "PHP7": "PHP",
        "PHP8": "PHP",
    }

    if value in UNKNOWN_VALUES:
        return None

    return aliases.get(value, value)


def is_valid_language(value):
    return normalize_language(value) in VALID_LANGUAGES


def sha256_file(path):
    h = hashlib.sha256()

    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)

    return h.hexdigest()


# ============================================================
# STRUCTURAL LANGUAGE DETECTION
# ============================================================

def structural_language_scores(code):
    """
    Detect programming language from actual code/reference
    structure.

    IMPORTANT:
    This function NEVER uses task_family.

    It only examines the supplied reference/ground-truth code.
    """

    if code is None:
        return {}

    text = str(code).strip()

    if not text:
        return {}

    scores = {
        "PYTHON": 0,
        "JAVA": 0,
        "JAVASCRIPT": 0,
        "C": 0,
        "CPP": 0,
        "PHP": 0,
        "HTML": 0,
    }

    # --------------------------------------------------------
    # PYTHON
    # --------------------------------------------------------

    if re.search(r"\bdef\s+[A-Za-z_]\w*\s*\(", text):
        scores["PYTHON"] += 5

    if re.search(r"\bimport\s+[A-Za-z_]", text):
        scores["PYTHON"] += 3

    if re.search(r"\bfrom\s+[A-Za-z_][\w.]*\s+import\b", text):
        scores["PYTHON"] += 4

    if re.search(r"\bprint\s*\(", text):
        scores["PYTHON"] += 2

    if re.search(r"\bfor\s+\w+\s+in\s+", text):
        scores["PYTHON"] += 2

    if re.search(r":\s*$", text, re.MULTILINE):
        scores["PYTHON"] += 1

    if re.search(r"\bTrue\b|\bFalse\b|\bNone\b", text):
        scores["PYTHON"] += 2

    if re.search(r"\blambda\s+", text):
        scores["PYTHON"] += 2

    # Python list comprehension
    if re.search(
        r"\[[^\]]+\bfor\b[^\]]+\bin\b[^\]]+\]",
        text,
        re.DOTALL
    ):
        scores["PYTHON"] += 3

    # --------------------------------------------------------
    # JAVASCRIPT
    # --------------------------------------------------------

    if re.search(r"\b(const|let|var)\s+[A-Za-z_$]", text):
        scores["JAVASCRIPT"] += 4

    if re.search(
        r"=>",
        text
    ):
        scores["JAVASCRIPT"] += 4

    if re.search(
        r"\b[A-Za-z_$][\w$]*\s*\.\s*(sort|map|filter|reduce|forEach)\s*\(",
        text
    ):
        scores["JAVASCRIPT"] += 4

    if re.search(
        r"\bconsole\s*\.\s*(log|error|warn)\s*\(",
        text
    ):
        scores["JAVASCRIPT"] += 3

    if re.search(
        r"\bfunction\s+[A-Za-z_$][\w$]*\s*\(",
        text
    ):
        scores["JAVASCRIPT"] += 3

    if re.search(
        r"\bundefined\b|\bNaN\b",
        text
    ):
        scores["JAVASCRIPT"] += 2

    # --------------------------------------------------------
    # JAVA
    # --------------------------------------------------------

    if re.search(
        r"\bpublic\s+(static\s+)?class\s+\w+",
        text
    ):
        scores["JAVA"] += 6

    if re.search(
        r"\bpublic\s+static\s+void\s+main\s*\(",
        text
    ):
        scores["JAVA"] += 6

    if re.search(
        r"\bSystem\s*\.\s*out\s*\.\s*println\s*\(",
        text
    ):
        scores["JAVA"] += 5

    if re.search(
        r"\bpublic\s+(static\s+)?(?:int|double|float|boolean|String)\s+\w+\s*\(",
        text
    ):
        scores["JAVA"] += 3

    if re.search(
        r"\bnew\s+[A-Z][A-Za-z0-9_]*\s*\(",
        text
    ):
        scores["JAVA"] += 2

    # --------------------------------------------------------
    # C++
    # --------------------------------------------------------

    if re.search(
        r"#include\s*<\s*(iostream|vector|string|algorithm|map|set)\s*>",
        text
    ):
        scores["CPP"] += 6

    if re.search(
        r"\bstd\s*::",
        text
    ):
        scores["CPP"] += 5

    if re.search(
        r"\busing\s+namespace\s+std\s*;",
        text
    ):
        scores["CPP"] += 4

    if re.search(
        r"\bcout\s*<<",
        text
    ):
        scores["CPP"] += 5

    # --------------------------------------------------------
    # C
    # --------------------------------------------------------

    if re.search(
        r"#include\s*<\s*(stdio|stdlib|string|math)\.h\s*>",
        text
    ):
        scores["C"] += 5

    if re.search(
        r"\bprintf\s*\(",
        text
    ):
        scores["C"] += 3

    if re.search(
        r"\bscanf\s*\(",
        text
    ):
        scores["C"] += 3

    if re.search(
        r"\bint\s+main\s*\(\s*(void)?\s*\)",
        text
    ):
        scores["C"] += 5

    # --------------------------------------------------------
    # PHP
    # --------------------------------------------------------

    if re.search(
        r"<\?php",
        text,
        re.IGNORECASE
    ):
        scores["PHP"] += 8

    if re.search(
        r"\$\w+",
        text
    ):
        scores["PHP"] += 3

    if re.search(
        r"\becho\s+",
        text
    ):
        scores["PHP"] += 3

    # --------------------------------------------------------
    # HTML
    # --------------------------------------------------------

    if re.search(
        r"<\s*(html|head|body|div|span|p|table|script|form|input)\b",
        text,
        re.IGNORECASE
    ):
        scores["HTML"] += 5

    if re.search(
        r"</\s*[A-Za-z][A-Za-z0-9]*\s*>",
        text
    ):
        scores["HTML"] += 3

    # --------------------------------------------------------
    # Remove zero scores
    # --------------------------------------------------------

    return {
        language: score
        for language, score in scores.items()
        if score > 0
    }


def resolve_structural_language(record):
    """
    Resolve language from reference/ground-truth code.

    Returns:
        language
        method
        evidence
        scores

    Returns None language if structural evidence is insufficient.
    """

    candidates = []

    reference_solution = record.get(
        "reference_solution"
    )

    ground_truth = record.get(
        "ground_truth"
    )

    if reference_solution:
        candidates.append(
            ("reference_solution", str(reference_solution))
        )

    if ground_truth:
        candidates.append(
            ("ground_truth", str(ground_truth))
        )

    combined_scores = {}

    for source_name, code in candidates:

        scores = structural_language_scores(code)

        for language, score in scores.items():

            combined_scores[language] = (
                combined_scores.get(language, 0) + score
            )

    if not combined_scores:
        return None, None, None, {}

    ranked = sorted(
        combined_scores.items(),
        key=lambda x: x[1],
        reverse=True
    )

    best_language, best_score = ranked[0]

    second_score = (
        ranked[1][1]
        if len(ranked) > 1
        else 0
    )

    # --------------------------------------------------------
    # Confidence rule
    #
    # Accept only if:
    #   1. score is sufficiently strong
    #   2. best candidate clearly dominates
    # --------------------------------------------------------

    if best_score < 5:
        return None, None, None, combined_scores

    if second_score > 0 and (
        best_score - second_score < 2
    ):
        return None, None, None, combined_scores

    # Find strongest evidence source
    evidence_parts = []

    for source_name, code in candidates:

        source_scores = structural_language_scores(code)

        if best_language in source_scores:

            evidence_parts.append(
                f"{source_name}: "
                f"{source_scores[best_language]}"
            )

    evidence = (
        f"Structural evidence for {best_language}; "
        + "; ".join(evidence_parts)
    )

    return (
        best_language,
        "STRUCTURAL_REFERENCE_VALIDATION",
        evidence,
        combined_scores,
    )


# ============================================================
# [1] INPUT FILE CHECK
# ============================================================

print("\n[1] INPUT FILE CHECK")
print("-" * 78)

if not BASE_DATASET.exists():
    raise FileNotFoundError(
        f"Base dataset not found: {BASE_DATASET}"
    )

if not AUDIT_FILE.exists():
    raise FileNotFoundError(
        f"Resolution audit not found: {AUDIT_FILE}"
    )

print(f"✓ Base dataset      : {BASE_DATASET}")
print(f"✓ Resolution audit  : {AUDIT_FILE}")


# ============================================================
# [2] LOAD BASE DATASET
# ============================================================

print("\n[2] LOAD BASE DATASET")
print("-" * 78)

with BASE_DATASET.open(
    "r",
    encoding="utf-8"
) as f:

    base_data = json.load(f)

print(
    f"Top-level JSON type : "
    f"{type(base_data).__name__}"
)

if not isinstance(base_data, dict):
    raise ValueError(
        "Base dataset must be a JSON object."
    )

print("Top-level JSON keys:")

for key, value in base_data.items():

    if isinstance(value, list):
        print(
            f"  {key:<34} "
            f"LIST ({len(value)} items)"
        )

    elif isinstance(value, dict):
        print(
            f"  {key:<34} DICT"
        )

    else:
        print(
            f"  {key:<34} "
            f"{type(value).__name__}"
        )


records = base_data.get("records")

if not isinstance(records, list):
    raise ValueError(
        "Base dataset does not contain a valid "
        "'records' list."
    )

print(
    f"\nEvaluation records loaded : "
    f"{len(records)}"
)

if len(records) != EXPECTED_SOURCE_TASKS:
    raise ValueError(
        f"Expected {EXPECTED_SOURCE_TASKS} "
        f"source tasks, found {len(records)}."
    )

print(
    f"✓ Exactly {EXPECTED_SOURCE_TASKS} "
    f"source tasks found"
)


# ============================================================
# [3] BASE RECORD STRUCTURE CHECK
# ============================================================

print("\n[3] BASE RECORD STRUCTURE CHECK")
print("-" * 78)

required_source_fields = {
    "eval_id",
    "task_family",
    "programming_language",
    "english_query",
    "arabic_query",
    "reference_solution",
    "ground_truth",
    "reference_validation",
    "reference_valid",
    "evaluation_cases",
    "source_id",
}

missing_structure = []

for record in records:

    eval_id = record.get("eval_id")

    missing = [
        field
        for field in required_source_fields
        if field not in record
    ]

    if missing:
        missing_structure.append(
            (eval_id, missing)
        )

if missing_structure:

    print("✗ Missing fields detected:")

    for eval_id, fields in missing_structure:
        print(
            f"  {eval_id}: {fields}"
        )

    raise ValueError(
        "Base record structure validation failed."
    )

print(
    f"✓ All {len(records)} source records "
    f"contain required fields"
)


# ============================================================
# [4] SOURCE TASK ID CHECK
# ============================================================

print("\n[4] SOURCE TASK ID CHECK")
print("-" * 78)

source_ids = [
    str(r["eval_id"]).strip()
    for r in records
]

if len(set(source_ids)) != len(source_ids):
    duplicates = (
        pd.Series(source_ids)
        .value_counts()
    )

    duplicates = duplicates[
        duplicates > 1
    ]

    raise ValueError(
        f"Duplicate source task IDs: "
        f"{duplicates.to_dict()}"
    )

if len(source_ids) != EXPECTED_SOURCE_TASKS:
    raise ValueError(
        "Unexpected source task count."
    )

print(
    f"✓ {EXPECTED_SOURCE_TASKS} "
    f"unique source task IDs"
)


# ============================================================
# [5] LOAD GROUND-TRUTH AUDIT
# ============================================================

print("\n[5] LOAD GROUND-TRUTH AUDIT")
print("-" * 78)

with AUDIT_FILE.open(
    "r",
    encoding="utf-8"
) as f:

    audit_data = json.load(f)

if isinstance(audit_data, dict):

    if isinstance(
        audit_data.get("records"),
        list
    ):
        audit_records = audit_data["records"]

    elif isinstance(
        audit_data.get("audit"),
        list
    ):
        audit_records = audit_data["audit"]

    else:

        # Support dictionary keyed by eval_id
        audit_records = []

        for key, value in audit_data.items():

            if isinstance(value, dict):

                row = dict(value)

                if "eval_id" not in row:
                    row["eval_id"] = key

                audit_records.append(row)

elif isinstance(audit_data, list):

    audit_records = audit_data

else:

    raise ValueError(
        "Unsupported ground-truth audit JSON structure."
    )

print(
    f"Audit rows loaded : "
    f"{len(audit_records)}"
)


# ============================================================
# [6] BUILD CLEAN AUDIT LOOKUP
# ============================================================

print("\n[6] BUILD CLEAN AUDIT LOOKUP")
print("-" * 78)

audit_lookup = {}

identical_duplicates = 0
conflicting_duplicates = []

for row in audit_records:

    eval_id = str(
        row.get("eval_id", "")
    ).strip()

    if not eval_id:
        raise ValueError(
            "Audit row without eval_id."
        )

    # --------------------------------------------------------
    # Normalize important fields
    # --------------------------------------------------------

    normalized = dict(row)

    if "resolved_language" in normalized:
        normalized["resolved_language"] = (
            normalize_language(
                normalized["resolved_language"]
            )
        )

    # --------------------------------------------------------
    # First occurrence
    # --------------------------------------------------------

    if eval_id not in audit_lookup:

        audit_lookup[eval_id] = normalized

    else:

        previous = audit_lookup[eval_id]

        # Compare meaningful resolution fields
        comparison_fields = [
            "resolved_language",
            "ground_truth_status",
            "review_basis",
            "review_decision",
            "manual_review_completed",
        ]

        identical = all(
            previous.get(field) ==
            normalized.get(field)
            for field in comparison_fields
        )

        if identical:

            identical_duplicates += 1

            print(
                f"ℹ Duplicate identical audit row "
                f"ignored: {eval_id}"
            )

        else:

            conflicting_duplicates.append(
                eval_id
            )


if conflicting_duplicates:

    conflicts = sorted(
        set(conflicting_duplicates)
    )

    print(
        "\n✗ CONFLICTING AUDIT RECORDS:"
    )

    for eval_id in conflicts:
        print(
            f"  {eval_id}"
        )

    raise ValueError(
        "Conflicting audit resolutions detected. "
        "Dataset will NOT be frozen."
    )

print(
    f"\nUnique audit resolutions : "
    f"{len(audit_lookup)}"
)

print(
    f"Identical duplicates ignored : "
    f"{identical_duplicates}"
)

print(
    f"Conflicting duplicates : "
    f"{len(conflicting_duplicates)}"
)


# ============================================================
# [7] AUDIT RESOLUTION CHECK
# ============================================================

print("\n[7] AUDIT RESOLUTION CHECK")
print("-" * 78)

unknown_audit_ids = [
    eval_id
    for eval_id in audit_lookup
    if eval_id not in source_ids
]

if unknown_audit_ids:

    raise ValueError(
        "Audit contains eval_id values not present "
        f"in source dataset: {unknown_audit_ids}"
    )

print(
    f"✓ {len(audit_lookup)} unique audited source tasks"
)

print(
    "✓ All audited eval_id values exist "
    "in source dataset"
)


# ============================================================
# [8] PROGRAMMING LANGUAGE RESOLUTION
# ============================================================

print("\n[8] PROGRAMMING LANGUAGE RESOLUTION")
print("-" * 78)

resolved_records = []

resolution_summary = []

unresolved_languages = []

for record in records:

    eval_id = str(
        record["eval_id"]
    ).strip()

    source_language = normalize_language(
        record.get("programming_language")
    )

    resolved_language = None
    resolution_method = None
    resolution_evidence = None
    structural_scores = {}

    # --------------------------------------------------------
    # Priority 1:
    # Approved audit resolution
    # --------------------------------------------------------

    audit = audit_lookup.get(eval_id)

    if audit is not None:

        audit_decision = str(
            audit.get(
                "review_decision",
                ""
            )
        ).strip().upper()

        audit_language = normalize_language(
            audit.get("resolved_language")
        )

        if (
            audit_decision == "APPROVED"
            and audit_language in VALID_LANGUAGES
        ):

            resolved_language = audit_language

            resolution_method = (
                "APPROVED_MANUAL_AUDIT"
            )

            resolution_evidence = (
                audit.get("review_basis")
                or "Approved ground-truth audit"
            )

    # --------------------------------------------------------
    # Priority 2:
    # Explicit source language
    # --------------------------------------------------------

    if resolved_language is None:

        if source_language in VALID_LANGUAGES:

            resolved_language = source_language

            resolution_method = (
                "EXPLICIT_SOURCE_LANGUAGE"
            )

            resolution_evidence = (
                "Explicit programming_language "
                "in source dataset"
            )

    # --------------------------------------------------------
    # Priority 3:
    # Structural reference validation
    # --------------------------------------------------------

    if resolved_language is None:

        (
            structural_language,
            structural_method,
            structural_evidence,
            structural_scores,
        ) = resolve_structural_language(
            record
        )

        if structural_language in VALID_LANGUAGES:

            resolved_language = (
                structural_language
            )

            resolution_method = (
                structural_method
            )

            resolution_evidence = (
                structural_evidence
            )

    # --------------------------------------------------------
    # HARD FAIL — NO GUESSING
    # --------------------------------------------------------

    if resolved_language is None:

        unresolved_languages.append(
            {
                "eval_id": eval_id,
                "source_language": source_language,
                "reference_validation": record.get(
                    "reference_validation"
                ),
                "task_family": record.get(
                    "task_family"
                ),
                "structural_scores": structural_scores,
            }
        )

        continue

    # --------------------------------------------------------
    # Store resolved copy
    # --------------------------------------------------------

    new_record = dict(record)

    new_record[
        "programming_language"
    ] = resolved_language

    new_record[
        "_language_resolution_method"
    ] = resolution_method

    new_record[
        "_language_resolution_evidence"
    ] = resolution_evidence

    resolved_records.append(
        new_record
    )

    resolution_summary.append(
        {
            "eval_id": eval_id,
            "resolved_language": resolved_language,
            "resolution_method": resolution_method,
            "evidence": resolution_evidence,
        }
    )


# ------------------------------------------------------------
# Resolution report
# ------------------------------------------------------------

print(
    f"Resolved source tasks : "
    f"{len(resolved_records)}"
)

print("\nLANGUAGE RESOLUTION SUMMARY")
print("-" * 78)

for item in resolution_summary:

    print(
        f"{item['eval_id']:<10} | "
        f"{item['resolved_language']:<12} | "
        f"{item['resolution_method']}"
    )

# ------------------------------------------------------------
# Hard stop if anything unresolved
# ------------------------------------------------------------

if unresolved_languages:

    print(
        "\n✗ UNRESOLVED PROGRAMMING LANGUAGES"
    )

    print("-" * 78)

    for item in unresolved_languages:

        print(
            f"\n{item['eval_id']}"
        )

        print(
            f"  source_language     : "
            f"{item['source_language']}"
        )

        print(
            f"  task_family         : "
            f"{item['task_family']}"
        )

        print(
            f"  reference_validation: "
            f"{item['reference_validation']}"
        )

        print(
            f"  structural_scores   : "
            f"{item['structural_scores']}"
        )

    raise ValueError(
        "Dataset cannot be frozen because "
        f"{len(unresolved_languages)} programming "
        "language(s) remain unresolved."
    )


# ============================================================
# SPECIFIC EVAL_20 VERIFICATION
# ============================================================

print("\n[8A] STRUCTURAL RESOLUTION SPOT CHECK")
print("-" * 78)

eval20 = next(
    (
        r for r in resolved_records
        if r["eval_id"] == "EVAL_20"
    ),
    None
)

if eval20 is None:

    raise ValueError(
        "EVAL_20 disappeared during resolution."
    )

print(
    f"EVAL_20 resolved language : "
    f"{eval20['programming_language']}"
)

print(
    f"Resolution method         : "
    f"{eval20['_language_resolution_method']}"
)

print(
    f"Evidence                  : "
    f"{eval20['_language_resolution_evidence']}"
)

if eval20["programming_language"] != "JAVASCRIPT":

    raise ValueError(
        "EVAL_20 was not resolved to JAVASCRIPT. "
        "Freeze aborted."
    )

print(
    "✓ EVAL_20 correctly resolved "
    "as JAVASCRIPT"
)


# ============================================================
# [9] BUILD 60 EVALUATION CASES
# ============================================================

print("\n[9] BUILDING 60 EVALUATION CASES")
print("-" * 78)

frozen_records = []

for record in resolved_records:

    eval_id = str(
        record["eval_id"]
    ).strip()

    programming_language = (
        record["programming_language"]
    )

    if programming_language not in VALID_LANGUAGES:

        raise ValueError(
            f"Invalid programming language "
            f"for {eval_id}: "
            f"{programming_language}"
        )

    evaluation_cases = record.get(
        "evaluation_cases"
    )

    # --------------------------------------------------------
    # Validate evaluation_cases
    # --------------------------------------------------------

    if not isinstance(
        evaluation_cases,
        list
    ):

        raise ValueError(
            f"evaluation_cases is not a list "
            f"for {eval_id}."
        )

    if len(evaluation_cases) != EXPECTED_CASES_PER_SOURCE:

        raise ValueError(
            f"{eval_id} must contain exactly "
            f"{EXPECTED_CASES_PER_SOURCE} evaluation "
            f"cases, found {len(evaluation_cases)}."
        )

    # --------------------------------------------------------
    # Build English / Arabic cases
    # --------------------------------------------------------

    case_languages_seen = set()

    for case in evaluation_cases:

        case_id = str(
            case.get("case_id", "")
        ).strip()

        evaluation_language = str(
            case.get("language", "")
        ).strip().upper()

        query = case.get(
            "query"
        )

        evidence_language = str(
            case.get(
                "evidence_language",
                ""
            )
        ).strip().upper()

        if not case_id:
            raise ValueError(
                f"Missing case_id for {eval_id}."
            )

        if evaluation_language not in {
            "ENGLISH",
            "ARABIC",
        }:

            raise ValueError(
                f"Invalid evaluation language "
                f"for {case_id}: "
                f"{evaluation_language}"
            )

        if evaluation_language in case_languages_seen:

            raise ValueError(
                f"Duplicate evaluation language "
                f"for {eval_id}: "
                f"{evaluation_language}"
            )

        case_languages_seen.add(
            evaluation_language
        )

        if not query:
            raise ValueError(
                f"Missing query for {case_id}."
            )

        # ----------------------------------------------------
        # Ground truth
        # ----------------------------------------------------

        ground_truth = record.get(
            "ground_truth"
        )

        if ground_truth is None or str(
            ground_truth
        ).strip() == "":

            raise ValueError(
                f"Missing ground truth "
                f"for {eval_id}."
            )

        # ----------------------------------------------------
        # Ground truth metadata
        # ----------------------------------------------------

        audit = audit_lookup.get(
            eval_id
        )

        if audit is not None:

            audit_decision = str(
                audit.get(
                    "review_decision",
                    ""
                )
            ).strip().upper()

            if audit_decision == "APPROVED":

                ground_truth_status = (
                    "APPROVED_BY_REFERENCE"
                )

                review_basis = (
                    audit.get(
                        "review_basis"
                    )
                    or
                    "Approved ground-truth audit"
                )

                review_decision = "APPROVED"

                manual_review_completed = True

            else:

                ground_truth_status = (
                    "AUTO_STRUCTURAL_VALIDATION_PASS"
                )

                review_basis = (
                    "Structural reference validation"
                )

                review_decision = (
                    "NOT_REQUIRED"
                )

                manual_review_completed = False

        else:

            ground_truth_status = (
                "AUTO_STRUCTURAL_VALIDATION_PASS"
            )

            review_basis = (
                "Structural reference validation"
            )

            review_decision = (
                "NOT_REQUIRED"
            )

            manual_review_completed = False

        # ----------------------------------------------------
        # Construct frozen evaluation case
        # ----------------------------------------------------

        frozen_case = {

            "case_id":
                case_id,

            "eval_id":
                eval_id,

            "evaluation_language":
                evaluation_language,

            "evidence_language":
                evidence_language
                if evidence_language
                else evaluation_language,

            "task_family":
                record["task_family"],

            "programming_language":
                programming_language,

            "query":
                query,

            "ground_truth":
                ground_truth,

            "ground_truth_status":
                ground_truth_status,

            "manual_review_required":
                bool(
                    record.get(
                        "manual_review_required",
                        False
                    )
                ),

            "source_id":
                record.get(
                    "source_id"
                ),

            "ground_truth_review_basis":
                review_basis,

            "ground_truth_review_decision":
                review_decision,

            "manual_review_completed":
                manual_review_completed,
        }

        frozen_records.append(
            frozen_case
        )


# ============================================================
# [10] BASIC 60-CASE VALIDATION
# ============================================================

print("\n[10] BASIC 60-CASE VALIDATION")
print("-" * 78)

print(
    f"Evaluation cases built : "
    f"{len(frozen_records)}"
)

if len(frozen_records) != EXPECTED_EVALUATION_CASES:

    raise ValueError(
        f"Expected exactly "
        f"{EXPECTED_EVALUATION_CASES} cases, "
        f"found {len(frozen_records)}."
    )

print(
    f"✓ Exactly {EXPECTED_EVALUATION_CASES} "
    f"evaluation cases built"
)


# ============================================================
# [11] DATAFRAME CONSTRUCTION
# ============================================================

df = pd.DataFrame(
    frozen_records
)

expected_columns = [
    "case_id",
    "eval_id",
    "evaluation_language",
    "evidence_language",
    "task_family",
    "programming_language",
    "query",
    "ground_truth",
    "ground_truth_status",
    "manual_review_required",
    "source_id",
    "ground_truth_review_basis",
    "ground_truth_review_decision",
    "manual_review_completed",
]

missing_columns = [
    c
    for c in expected_columns
    if c not in df.columns
]

if missing_columns:

    raise ValueError(
        f"Missing final columns: "
        f"{missing_columns}"
    )

df = df[
    expected_columns
]


# ============================================================
# [12] FINAL CONSISTENCY CHECKS
# ============================================================

print("\n[12] FINAL CONSISTENCY CHECKS")
print("-" * 78)

checks = []


# ------------------------------------------------------------
# Check 1 — 60 rows
# ------------------------------------------------------------

if len(df) == 60:
    print(
        "PASS | dataset_size_60"
    )
    checks.append(True)
else:
    print(
        "FAIL | dataset_size_60"
    )
    checks.append(False)


# ------------------------------------------------------------
# Check 2 — 30 source tasks
# ------------------------------------------------------------

unique_eval_ids = df[
    "eval_id"
].nunique()

if unique_eval_ids == 30:
    print(
        "PASS | source_tasks_30"
    )
    checks.append(True)
else:
    print(
        "FAIL | source_tasks_30"
    )
    checks.append(False)


# ------------------------------------------------------------
# Check 3 — unique case IDs
# ------------------------------------------------------------

unique_case_ids = df[
    "case_id"
].nunique()

if unique_case_ids == 60:
    print(
        "PASS | unique_case_ids_60"
    )
    checks.append(True)
else:
    print(
        "FAIL | unique_case_ids_60"
    )
    checks.append(False)


# ------------------------------------------------------------
# Check 4 — exactly 2 cases/source
# ------------------------------------------------------------

cases_per_source = (
    df.groupby(
        "eval_id"
    ).size()
)

if (
    len(cases_per_source) == 30
    and
    cases_per_source.eq(2).all()
):

    print(
        "PASS | exactly_two_cases_per_source"
    )

    checks.append(True)

else:

    print(
        "FAIL | exactly_two_cases_per_source"
    )

    print(cases_per_source)

    checks.append(False)


# ------------------------------------------------------------
# Check 5 — English/Arabic pair
# ------------------------------------------------------------

language_sets = (
    df.groupby(
        "eval_id"
    )[
        "evaluation_language"
    ]
    .apply(set)
)

pair_ok = all(
    languages == {
        "ENGLISH",
        "ARABIC"
    }
    for languages in language_sets
)

if pair_ok:

    print(
        "PASS | english_arabic_pair_consistency"
    )

    checks.append(True)

else:

    print(
        "FAIL | english_arabic_pair_consistency"
    )

    checks.append(False)


# ------------------------------------------------------------
# Check 6 — no UNKNOWN language
# ------------------------------------------------------------

unknown_count = (
    df["programming_language"]
    .map(normalize_language)
    .isna()
    .sum()
)

if unknown_count == 0:

    print(
        "PASS | no_unknown_languages"
    )

    checks.append(True)

else:

    print(
        f"FAIL | unknown_languages_{unknown_count}"
    )

    checks.append(False)


# ------------------------------------------------------------
# Check 7 — all valid languages
# ------------------------------------------------------------

invalid_languages = sorted(
    set(df["programming_language"])
    - VALID_LANGUAGES
)

if not invalid_languages:

    print(
        "PASS | valid_programming_languages"
    )

    checks.append(True)

else:

    print(
        f"FAIL | invalid_languages: "
        f"{invalid_languages}"
    )

    checks.append(False)


# ------------------------------------------------------------
# Check 8 — ground truth present
# ------------------------------------------------------------

missing_gt = (
    df["ground_truth"]
    .isna()
    |
    df["ground_truth"]
    .astype(str)
    .str.strip()
    .eq("")
).sum()

if missing_gt == 0:

    print(
        "PASS | ground_truth_present"
    )

    checks.append(True)

else:

    print(
        f"FAIL | missing_ground_truth_{missing_gt}"
    )

    checks.append(False)


# ------------------------------------------------------------
# Check 9 — no duplicate case IDs
# ------------------------------------------------------------

duplicate_case_ids = (
    df[
        df["case_id"].duplicated(
            keep=False
        )
    ]["case_id"]
    .unique()
)

if len(duplicate_case_ids) == 0:

    print(
        "PASS | no_duplicate_case_ids"
    )

    checks.append(True)

else:

    print(
        "FAIL | duplicate_case_ids: "
        f"{list(duplicate_case_ids)}"
    )

    checks.append(False)


# ------------------------------------------------------------
# Check 10 — audited cases resolved
# ------------------------------------------------------------

audit_in_final = set(
    audit_lookup.keys()
).issubset(
    set(df["eval_id"])
)

if audit_in_final:

    print(
        "PASS | audited_tasks_present"
    )

    checks.append(True)

else:

    print(
        "FAIL | audited_tasks_present"
    )

    checks.append(False)


# ------------------------------------------------------------
# Check 11 — source IDs preserved
# ------------------------------------------------------------

source_id_missing = (
    df["source_id"]
    .isna()
    .sum()
)

if source_id_missing == 0:

    print(
        "PASS | source_ids_preserved"
    )

    checks.append(True)

else:

    print(
        f"FAIL | missing_source_ids_{source_id_missing}"
    )

    checks.append(False)


# ------------------------------------------------------------
# Check 12 — EVAL_20 specifically
# ------------------------------------------------------------

eval20_rows = df[
    df["eval_id"] == "EVAL_20"
]

eval20_ok = (
    len(eval20_rows) == 2
    and
    eval20_rows[
        "programming_language"
    ].eq("JAVASCRIPT").all()
)

if eval20_ok:

    print(
        "PASS | EVAL_20_structural_resolution"
    )

    checks.append(True)

else:

    print(
        "FAIL | EVAL_20_structural_resolution"
    )

    checks.append(False)


# ============================================================
# FINAL DATASET STATUS
# ============================================================

print("\n" + "=" * 78)
print("FINAL DATASET VALIDATION")
print("=" * 78)

if not all(checks):

    raise RuntimeError(
        "FINAL CONSISTENCY CHECKS FAILED. "
        "Dataset will NOT be frozen."
    )

print(
    "✓ All critical consistency checks passed"
)


# ============================================================
# DISTRIBUTIONS
# ============================================================

print("\n" + "=" * 78)
print("FINAL PROGRAMMING LANGUAGE DISTRIBUTION")
print("=" * 78)

print(
    df[
        "programming_language"
    ].value_counts()
    .sort_index()
)


print("\n" + "=" * 78)
print("FINAL EVALUATION LANGUAGE DISTRIBUTION")
print("=" * 78)

print(
    df[
        "evaluation_language"
    ].value_counts()
    .sort_index()
)


print("\n" + "=" * 78)
print("FINAL GROUND TRUTH STATUS")
print("=" * 78)

print(
    df[
        "ground_truth_status"
    ].value_counts()
)


# ============================================================
# [13] PREVIEW
# ============================================================

print("\n" + "=" * 78)
print("FROZEN DATASET PREVIEW")
print("=" * 78)

print(
    df[
        [
            "case_id",
            "eval_id",
            "evaluation_language",
            "programming_language",
            "task_family",
        ]
    ].head(10).to_string(index=False)
)


# ============================================================
# [14] SAVE FROZEN DATASET
# ============================================================

print("\n" + "=" * 78)
print("SAVING FROZEN DATASET")
print("=" * 78)

# ------------------------------------------------------------
# JSON
# ------------------------------------------------------------

with OUT_JSON.open(
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        frozen_records,
        f,
        ensure_ascii=False,
        indent=2
    )

print(
    f"JSON     : {OUT_JSON}"
)


# ------------------------------------------------------------
# CSV
# ------------------------------------------------------------

df.to_csv(
    OUT_CSV,
    index=False,
    encoding="utf-8-sig"
)

print(
    f"CSV      : {OUT_CSV}"
)


# ------------------------------------------------------------
# XLSX
# ------------------------------------------------------------

try:

    df.to_excel(
        OUT_XLSX,
        index=False
    )

    print(
        f"XLSX     : {OUT_XLSX}"
    )

    excel_status = "SUCCESS"

except Exception as e:

    excel_status = (
        f"FAILED: {str(e)}"
    )

    print(
        f"XLSX     : {excel_status}"
    )


# ============================================================
# [15] SHA256
# ============================================================

print("\n" + "=" * 78)
print("SHA256")
print("=" * 78)

json_sha256 = sha256_file(
    OUT_JSON
)

csv_sha256 = sha256_file(
    OUT_CSV
)

xlsx_sha256 = (
    sha256_file(OUT_XLSX)
    if OUT_XLSX.exists()
    else None
)

print(
    f"JSON : {json_sha256}"
)

print(
    f"CSV  : {csv_sha256}"
)

if xlsx_sha256:
    print(
        f"XLSX : {xlsx_sha256}"
)


# ============================================================
# [16] MANIFEST
# ============================================================

manifest = {

    "dataset_name":
        "X-AutoAlign RAG V3.8.5.1 Frozen Evaluation Dataset",

    "dataset_status":
        "FROZEN",

    "source_dataset":
        str(BASE_DATASET),

    "ground_truth_audit":
        str(AUDIT_FILE),

    "num_source_tasks":
        int(df["eval_id"].nunique()),

    "num_evaluation_cases":
        int(len(df)),

    "cases_per_source_task":
        EXPECTED_CASES_PER_SOURCE,

    "evaluation_languages":
        sorted(
            df[
                "evaluation_language"
            ].unique()
        ),

    "programming_languages":
        sorted(
            df[
                "programming_language"
            ].unique()
        ),

    "ground_truth_status_distribution":
        {
            str(k): int(v)
            for k, v in df[
                "ground_truth_status"
            ].value_counts().items()
        },

    "language_resolution_methods":
        {
            str(k): int(v)
            for k, v in pd.Series(
                [
                    r[
                        "_language_resolution_method"
                    ]
                    for r in resolved_records
                ]
            ).value_counts().items()
        },

    "audit_unique_resolutions":
        int(len(audit_lookup)),

    "audit_identical_duplicates_ignored":
        int(identical_duplicates),

    "audit_conflicting_duplicates":
        int(len(conflicting_duplicates)),

    "sha256":
        {
            "json":
                json_sha256,

            "csv":
                csv_sha256,

            "xlsx":
                xlsx_sha256,
        },

    "critical_checks":
        {
            "dataset_size_60":
                True,

            "source_tasks_30":
                True,

            "unique_case_ids_60":
                True,

            "exactly_two_cases_per_source":
                True,

            "english_arabic_pairs":
                True,

            "no_unknown_languages":
                True,

            "ground_truth_present":
                True,

            "no_duplicate_case_ids":
                True,

            "audit_tasks_present":
                True,

            "EVAL_20_JAVASCRIPT":
                True,
        },

    "freeze_policy":
        "No automatic language assignment by task_family alone",

    "structural_resolution_policy":
        "Language may be resolved from explicit source language, approved manual audit, or sufficiently strong structural evidence from reference_solution/ground_truth.",

    "step":
        "5C",

    "version":
        "V3.8.5.1",
}

with OUT_MANIFEST.open(
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        manifest,
        f,
        ensure_ascii=False,
        indent=2
    )

print(
    f"Manifest : {OUT_MANIFEST}"
)


# ============================================================
# [17] FINAL FREEZE STATUS
# ============================================================

print("\n" + "=" * 78)
print("STEP 5C COMPLETED SUCCESSFULLY")
print("=" * 78)

print(
    "ALL CRITICAL CHECKS PASSED"
)

print(
    "\nDATASET STATUS: FROZEN"
)

print(
    f"\nSource Tasks       : "
    f"{df['eval_id'].nunique()}"
)

print(
    f"Evaluation Cases   : "
    f"{len(df)}"
)

print(
    f"Unknown Languages  : "
    f"{unknown_count}"
)

print(
    "Manual Review Remaining : 0"
)

print(
    "\nFiles:"
)

print(
    f"JSON     : {OUT_JSON}"
)

print(
    f"CSV      : {OUT_CSV}"
)

print(
    f"XLSX     : {OUT_XLSX}"
)

print(
    f"Manifest : {OUT_MANIFEST}"
)

print("\n" + "=" * 78)
print("NEXT STEP")
print("=" * 78)

print(
    "STEP 6A — Evaluation Matrix Construction"
)

print(
    "\nIMPORTANT:"
)

print(
    "✓ Frozen dataset is now immutable."
)

print(
    "✓ Do NOT rerun STEP 5C."
)

print(
    "✓ Do NOT modify evaluation_dataset_frozen.json."
)

print(
    "✓ STEP 6A may now consume the frozen dataset."
)

print("=" * 78)

X-AutoAlign RAG V3.8.5.1 — STEP 5C
FINAL GROUND-TRUTH MERGE + CONSISTENCY CHECKS + DATASET FREEZE

[1] INPUT FILE CHECK
------------------------------------------------------------------------------
✓ Base dataset      : /content/evaluation_dataset_v1.json
✓ Resolution audit  : /content/ground_truth_resolution_audit.json

[2] LOAD BASE DATASET
------------------------------------------------------------------------------
Top-level JSON type : dict
Top-level JSON keys:
  dataset_name                       str
  dataset_version                    str
  source_dataset                     str
  num_source_tasks                   int
  num_evaluation_cases               int
  evaluation_design                  DICT
  ground_truth_policy                str
  task_distribution                  DICT
  programming_language_distribution  DICT
  valid_reference_count              int
  manual_review_count                int
  records                            LIST (30 items)

Evaluation records 

In [ ]:
# ================================================================
# X-AutoAlign RAG — STEP 6A
# EVALUATION MATRIX CONSTRUCTION — CORRECTED
#
# Frozen Dataset V1
# 30 Source Tasks × 2 Evaluation Cases = 60 Cases
# ================================================================

from pathlib import Path
import json
import pandas as pd

print("=" * 78)
print("X-AutoAlign RAG — STEP 6A")
print("EVALUATION MATRIX CONSTRUCTION")
print("=" * 78)


# ----------------------------------------------------------------
# 1. Locate FROZEN dataset
# ----------------------------------------------------------------

INPUT_JSON = Path("/content/evaluation_dataset_frozen.json")
INPUT_CSV  = Path("/content/evaluation_dataset_frozen.csv")

if INPUT_JSON.exists():
    INPUT_PATH = INPUT_JSON
    INPUT_TYPE = "JSON"

elif INPUT_CSV.exists():
    INPUT_PATH = INPUT_CSV
    INPUT_TYPE = "CSV"

else:
    raise FileNotFoundError(
        "Frozen evaluation dataset not found.\n"
        "Expected:\n"
        "/content/evaluation_dataset_frozen.json\n"
        "or\n"
        "/content/evaluation_dataset_frozen.csv"
    )

print(f"Input Dataset : {INPUT_PATH}")
print(f"Input Type    : {INPUT_TYPE}")


# ----------------------------------------------------------------
# 2. Load dataset
# ----------------------------------------------------------------

if INPUT_TYPE == "JSON":

    with open(INPUT_PATH, "r", encoding="utf-8") as f:
        data = json.load(f)

    if isinstance(data, list):
        records = data

    elif isinstance(data, dict):

        if "data" in data:
            records = data["data"]

        elif "records" in data:
            records = data["records"]

        elif "evaluation_cases" in data:
            records = data["evaluation_cases"]

        else:
            raise ValueError(
                "No recognized record container found in JSON."
            )

    else:
        raise ValueError("Unsupported JSON structure.")

    df = pd.DataFrame(records)

else:
    df = pd.read_csv(INPUT_PATH)


# ----------------------------------------------------------------
# 3. Inspect dataset
# ----------------------------------------------------------------

print("\n" + "=" * 78)
print("FROZEN DATASET STRUCTURE")
print("=" * 78)

print(f"Rows    : {len(df)}")
print(f"Columns : {len(df.columns)}")

print("\nColumns found:")

for col in df.columns:
    print(f"  - {col}")


# ----------------------------------------------------------------
# 4. Required columns
# ----------------------------------------------------------------

required_columns = [
    "case_id",
    "eval_id",
    "evaluation_language",
    "evidence_language",
    "task_family",
    "programming_language",
    "query",
    "ground_truth",
    "ground_truth_status"
]

missing = [
    col for col in required_columns
    if col not in df.columns
]

if missing:
    raise ValueError(
        "Required columns missing:\n"
        + "\n".join(f"  - {x}" for x in missing)
    )


# ----------------------------------------------------------------
# 5. Frozen Dataset structural checks
# ----------------------------------------------------------------

print("\n" + "=" * 78)
print("FROZEN DATASET VALIDATION")
print("=" * 78)

assert len(df) == 60, (
    f"Expected 60 evaluation cases, found {len(df)}."
)

# IMPORTANT:
# case_id must be unique across the 60 evaluation cases.
assert df["case_id"].nunique() == 60, (
    "case_id values are not unique."
)

# eval_id SHOULD repeat because:
# 30 source tasks × 2 evaluation cases = 60 cases.
unique_eval_ids = df["eval_id"].nunique()

assert unique_eval_ids == 30, (
    f"Expected 30 unique source eval_ids, found {unique_eval_ids}."
)

assert df["query"].notna().all(), (
    "Missing evaluation queries detected."
)

assert df["ground_truth"].notna().all(), (
    "Missing Ground Truth detected."
)

assert df["programming_language"].notna().all(), (
    "Missing programming languages detected."
)

assert df["evaluation_language"].notna().all(), (
    "Missing evaluation languages detected."
)

print("✓ 60 evaluation cases")
print("✓ 30 unique source tasks")
print("✓ 60 unique case_id values")
print("✓ Queries present")
print("✓ Ground Truth present")
print("✓ Programming languages resolved")


# ----------------------------------------------------------------
# 6. Verify 2 cases per source task
# ----------------------------------------------------------------

cases_per_task = (
    df.groupby("eval_id")
      .size()
)

assert (cases_per_task == 2).all(), (
    "Not every source task has exactly two evaluation cases."
)

print("✓ Every source task has exactly 2 evaluation cases")


# ----------------------------------------------------------------
# 7. Verify evaluation-language distribution
# ----------------------------------------------------------------

print("\n" + "=" * 78)
print("EVALUATION LANGUAGE DISTRIBUTION")
print("=" * 78)

print(
    df["evaluation_language"]
    .value_counts()
    .to_string()
)


# ----------------------------------------------------------------
# 8. Build Evaluation Matrix
# ----------------------------------------------------------------

matrix = pd.DataFrame()

# Case-level identity
matrix["case_id"] = df["case_id"].astype(str)

# Source-task identity
matrix["eval_id"] = df["eval_id"].astype(str)

# Original source task
if "source_id" in df.columns:
    matrix["source_task_id"] = (
        df["source_id"].astype(str)
    )
else:
    matrix["source_task_id"] = (
        df["eval_id"].astype(str)
    )

# Evaluation language
matrix["evaluation_language"] = (
    df["evaluation_language"]
    .astype(str)
    .str.upper()
    .str.strip()
)

# Evidence/reference language
matrix["evidence_language"] = (
    df["evidence_language"]
    .astype(str)
    .str.upper()
    .str.strip()
)

# Programming language of target/reference solution
matrix["programming_language"] = (
    df["programming_language"]
    .astype(str)
    .str.upper()
    .str.strip()
)

# Task family
matrix["task_family"] = (
    df["task_family"]
    .astype(str)
    .str.upper()
    .str.strip()
)

# Query
matrix["query"] = df["query"].astype(str)

# Ground Truth
matrix["ground_truth"] = (
    df["ground_truth"].astype(str)
)

# Ground Truth status
matrix["ground_truth_status"] = (
    df["ground_truth_status"].astype(str)
)


# ----------------------------------------------------------------
# 9. Case type
# ----------------------------------------------------------------

def determine_case_type(language):

    language = str(language).upper().strip()

    if language == "ENGLISH":
        return "ENGLISH_MONOLINGUAL"

    if language == "ARABIC":
        return "ARABIC_CROSSLINGUAL"

    return "OTHER"


matrix["case_type"] = (
    matrix["evaluation_language"]
    .apply(determine_case_type)
)


# ----------------------------------------------------------------
# 10. Expected behavior
# ----------------------------------------------------------------

def expected_behavior(task_family):

    if task_family == "PALINDROME":
        return "FUNCTIONAL_CORRECTNESS"

    if task_family in ["MATH", "FACTORIAL"]:
        return "FUNCTIONAL_OR_NUMERICAL_CORRECTNESS"

    if task_family in ["LIST_ARRAY", "DATA_STRUCTURE"]:
        return "FUNCTIONAL_CORRECTNESS"

    if task_family in [
        "STRING",
        "SEARCH",
        "SORTING"
    ]:
        return "FUNCTIONAL_CORRECTNESS"

    if task_family == "DEBUGGING":
        return "CODE_CORRECTION"

    return "TASK_SPECIFIC_CORRECTNESS"


matrix["expected_behavior"] = (
    matrix["task_family"]
    .apply(expected_behavior)
)


# ----------------------------------------------------------------
# 11. Experiment columns
# ----------------------------------------------------------------

matrix["system"] = ""
matrix["run_id"] = ""
matrix["generator"] = ""

matrix["answer"] = ""

matrix["context_relevance"] = None
matrix["factual_consistency"] = None
matrix["faithfulness"] = None
matrix["completeness"] = None
matrix["language_quality"] = None

matrix["answer_intent_score"] = None
matrix["q_score"] = None

matrix["decision"] = ""


# ----------------------------------------------------------------
# 12. Preserve deterministic order
# ----------------------------------------------------------------

matrix = matrix.reset_index(drop=True)


# ----------------------------------------------------------------
# 13. Final validation
# ----------------------------------------------------------------

print("\n" + "=" * 78)
print("EVALUATION MATRIX VALIDATION")
print("=" * 78)

print(f"Evaluation Cases     : {len(matrix)}")
print(f"Unique case_id       : {matrix['case_id'].nunique()}")
print(f"Unique source tasks  : {matrix['eval_id'].nunique()}")
print(
    f"Evaluation languages : "
    f"{matrix['evaluation_language'].nunique()}"
)
print(
    f"Task families        : "
    f"{matrix['task_family'].nunique()}"
)


assert len(matrix) == 60
assert matrix["case_id"].nunique() == 60
assert matrix["eval_id"].nunique() == 30


# ----------------------------------------------------------------
# 14. Language distribution
# ----------------------------------------------------------------

print("\n" + "=" * 78)
print("PROGRAMMING LANGUAGE DISTRIBUTION")
print("=" * 78)

print(
    matrix["programming_language"]
    .value_counts()
    .to_string()
)


# ----------------------------------------------------------------
# 15. Evaluation case distribution
# ----------------------------------------------------------------

print("\n" + "=" * 78)
print("CASE TYPE DISTRIBUTION")
print("=" * 78)

print(
    matrix["case_type"]
    .value_counts()
    .to_string()
)


# ----------------------------------------------------------------
# 16. Task-family distribution
# ----------------------------------------------------------------

print("\n" + "=" * 78)
print("TASK FAMILY DISTRIBUTION")
print("=" * 78)

print(
    matrix["task_family"]
    .value_counts()
    .to_string()
)


# ----------------------------------------------------------------
# 17. Preview
# ----------------------------------------------------------------

print("\n" + "=" * 78)
print("EVALUATION MATRIX PREVIEW")
print("=" * 78)

display(
    matrix[
        [
            "case_id",
            "eval_id",
            "evaluation_language",
            "programming_language",
            "task_family",
            "case_type",
            "query"
        ]
    ].head(10)
)


# ----------------------------------------------------------------
# 18. Save Evaluation Matrix
# ----------------------------------------------------------------

MATRIX_JSON = Path(
    "/content/evaluation_matrix_v1.json"
)

MATRIX_CSV = Path(
    "/content/evaluation_matrix_v1.csv"
)

MATRIX_XLSX = Path(
    "/content/evaluation_matrix_v1.xlsx"
)


matrix.to_json(
    MATRIX_JSON,
    orient="records",
    force_ascii=False,
    indent=2
)


matrix.to_csv(
    MATRIX_CSV,
    index=False,
    encoding="utf-8-sig"
)


with pd.ExcelWriter(
    MATRIX_XLSX,
    engine="openpyxl"
) as writer:

    matrix.to_excel(
        writer,
        sheet_name="Evaluation_Matrix",
        index=False
    )

    summary = pd.DataFrame({
        "Metric": [
            "Evaluation Cases",
            "Unique Source Tasks",
            "Unique Case IDs",
            "Evaluation Languages",
            "Programming Languages",
            "Task Families"
        ],

        "Value": [
            len(matrix),
            matrix["eval_id"].nunique(),
            matrix["case_id"].nunique(),
            matrix["evaluation_language"].nunique(),
            matrix["programming_language"].nunique(),
            matrix["task_family"].nunique()
        ]
    })

    summary.to_excel(
        writer,
        sheet_name="Summary",
        index=False
    )


# ----------------------------------------------------------------
# 19. Final status
# ----------------------------------------------------------------

print("\n" + "=" * 78)
print("STEP 6A COMPLETED SUCCESSFULLY")
print("=" * 78)

print("Evaluation Cases : 60")
print("Source Tasks     : 30")
print("Unique Case IDs  : 60")

print("\n✓ Frozen Dataset preserved")
print("✓ No cases added")
print("✓ No cases removed")
print("✓ No Ground Truth modified")
print("✓ 30 source tasks × 2 cases confirmed")

print("\nFILES")
print("-" * 78)
print(f"JSON : {MATRIX_JSON}")
print(f"CSV  : {MATRIX_CSV}")
print(f"XLSX : {MATRIX_XLSX}")

print("\nNEXT STEP:")
print("STEP 6B — Evaluation Harness")
print("=" * 78)

X-AutoAlign RAG — STEP 6A
EVALUATION MATRIX CONSTRUCTION
Input Dataset : /content/evaluation_dataset_frozen.json
Input Type    : JSON

FROZEN DATASET STRUCTURE
Rows    : 60
Columns : 14

Columns found:
  - case_id
  - eval_id
  - evaluation_language
  - evidence_language
  - task_family
  - programming_language
  - query
  - ground_truth
  - ground_truth_status
  - manual_review_required
  - source_id
  - ground_truth_review_basis
  - ground_truth_review_decision
  - manual_review_completed

FROZEN DATASET VALIDATION
✓ 60 evaluation cases
✓ 30 unique source tasks
✓ 60 unique case_id values
✓ Queries present
✓ Ground Truth present
✓ Programming languages resolved
✓ Every source task has exactly 2 evaluation cases

EVALUATION LANGUAGE DISTRIBUTION
evaluation_language
ENGLISH    30
ARABIC     30

EVALUATION MATRIX VALIDATION
Evaluation Cases     : 60
Unique case_id       : 60
Unique source tasks  : 30
Evaluation languages : 2
Task families        : 11

PROGRAMMING LANGUAGE DISTRIBUTION
pr

,case_id,eval_id,evaluation_language,programming_language,task_family,case_type,query
0,EVAL_01_EN,EVAL_01,ENGLISH,PYTHON,LIST_ARRAY,ENGLISH_MONOLINGUAL,Write a function in Python to generate a list of numbers that is the square of all numbers between 1 and the number provided.
1,EVAL_01_AR,EVAL_01,ARABIC,PYTHON,LIST_ARRAY,ARABIC_CROSSLINGUAL,اكتب a treative_ Travery in PYTHON_Lang إلى توليد a قائمة من الأرقام التي هي مربع كل الأعداد بين 1 و العدد المقدّم.
2,EVAL_02_EN,EVAL_02,ENGLISH,PYTHON,LIST_ARRAY,ENGLISH_MONOLINGUAL,So you need to design a class for a Linked List.
3,EVAL_02_AR,EVAL_02,ARABIC,PYTHON,LIST_ARRAY,ARABIC_CROSSLINGUAL,لذا تحتاج إلى تصميم a صف لـ a ملحق قائمة.
4,EVAL_03_EN,EVAL_03,ENGLISH,PYTHON,LIST_ARRAY,ENGLISH_MONOLINGUAL,Write a function that takes an array of integers and returns an array of the same length with each element multiplied by 2.
5,EVAL_03_AR,EVAL_03,ARABIC,PYTHON,LIST_ARRAY,ARABIC_CROSSLINGUAL,اكتب a الدالة_ text الذي يأخذ ARAY_ صلاحيّة الاعداد الصحيحة ويرجع aRAY_ Tray_ Trayform من نفس الطول مع كل عنصر مضروب في 2.
6,EVAL_04_EN,EVAL_04,ENGLISH,C,STRING,ENGLISH_MONOLINGUAL,Write a method in C# for finding the longest common subsequence of two strings.
7,EVAL_04_AR,EVAL_04,ARABIC,C,STRING,ARABIC_CROSSLINGUAL,اكتب طريقة في C# للعثور على أطول تسلسل فرعي مشترك لاثنين من STRING_Times.
8,EVAL_05_EN,EVAL_05,ENGLISH,JAVASCRIPT,SORTING,ENGLISH_MONOLINGUAL,Sort an array in descending order using JavaScript.
9,EVAL_05_AR,EVAL_05,ARABIC,JAVASCRIPT,SORTING,ARABIC_CROSSLINGUAL,نوع ARAY_ مصطلح في الترتيب التنازلي باستخدام JavaSCRIPT_LANG.



STEP 6A COMPLETED SUCCESSFULLY
Evaluation Cases : 60
Source Tasks     : 30
Unique Case IDs  : 60

✓ Frozen Dataset preserved
✓ No cases added
✓ No cases removed
✓ No Ground Truth modified
✓ 30 source tasks × 2 cases confirmed

FILES
------------------------------------------------------------------------------
JSON : /content/evaluation_matrix_v1.json
CSV  : /content/evaluation_matrix_v1.csv
XLSX : /content/evaluation_matrix_v1.xlsx

NEXT STEP:
STEP 6B — Evaluation Harness


In [ ]:
# =============================================================================
# X-AutoAlign RAG V3.8.5
# STEP 6B — EVALUATION HARNESS
# =============================================================================
#
# Purpose:
#   Run the FROZEN Evaluation Matrix (60 cases)
#   through the existing V3.8.5 pipeline.
#
# IMPORTANT:
#   - Does NOT modify the frozen dataset
#   - Does NOT modify Ground Truth
#   - Does NOT create new evaluation cases
#   - Evaluates exactly 60 cases
#   - English = 30
#   - Arabic = 30
# =============================================================================

import os
import json
import time
import traceback
import pandas as pd
from pathlib import Path

print("=" * 78)
print("X-AutoAlign RAG — STEP 6B")
print("EVALUATION HARNESS")
print("=" * 78)

# -------------------------------------------------------------------------
# 1. Configuration
# -------------------------------------------------------------------------

FROZEN_DATASET = Path("/content/evaluation_dataset_frozen.json")

RESULT_JSON = Path("/content/evaluation_results_v1.json")
RESULT_CSV  = Path("/content/evaluation_results_v1.csv")
RESULT_XLSX = Path("/content/evaluation_results_v1.xlsx")

# V3.8.5 evaluation settings
Q_THRESHOLD = 0.70
TMAX = 2

print(f"Frozen Dataset : {FROZEN_DATASET}")
print(f"Quality Threshold : {Q_THRESHOLD}")
print(f"Tmax : {TMAX}")

# -------------------------------------------------------------------------
# 2. Verify frozen dataset
# -------------------------------------------------------------------------

if not FROZEN_DATASET.exists():
    raise FileNotFoundError(
        f"Frozen evaluation dataset not found:\n{FROZEN_DATASET}"
    )

with open(FROZEN_DATASET, "r", encoding="utf-8") as f:
    evaluation_data = json.load(f)

if not isinstance(evaluation_data, list):
    raise TypeError(
        "Frozen evaluation dataset must be a JSON list."
    )

print()
print("=" * 78)
print("FROZEN DATASET CHECK")
print("=" * 78)

print(f"Evaluation cases : {len(evaluation_data)}")

if len(evaluation_data) != 60:
    raise AssertionError(
        f"Expected exactly 60 evaluation cases, "
        f"found {len(evaluation_data)}."
    )

# -------------------------------------------------------------------------
# 3. Required columns
# -------------------------------------------------------------------------

required_columns = [
    "case_id",
    "eval_id",
    "evaluation_language",
    "programming_language",
    "task_family",
    "query",
    "ground_truth",
    "ground_truth_status",
]

for col in required_columns:
    missing = [
        i for i, row in enumerate(evaluation_data)
        if col not in row
    ]

    if missing:
        raise AssertionError(
            f"Required column '{col}' is missing in rows: {missing[:10]}"
        )

print("✓ Required fields present")

# -------------------------------------------------------------------------
# 4. Frozen integrity checks
# -------------------------------------------------------------------------

case_ids = [row["case_id"] for row in evaluation_data]
eval_ids = [row["eval_id"] for row in evaluation_data]
languages = [row["evaluation_language"] for row in evaluation_data]

if len(set(case_ids)) != 60:
    raise AssertionError("case_id values are not unique.")

if languages.count("ENGLISH") != 30:
    raise AssertionError(
        f"Expected 30 ENGLISH cases, "
        f"found {languages.count('ENGLISH')}"
    )

if languages.count("ARABIC") != 30:
    raise AssertionError(
        f"Expected 30 ARABIC cases, "
        f"found {languages.count('ARABIC')}"
    )

# Exactly two cases for every source task
from collections import Counter

eval_id_counts = Counter(eval_ids)

bad_eval_ids = {
    k: v for k, v in eval_id_counts.items()
    if v != 2
}

if bad_eval_ids:
    raise AssertionError(
        f"Each source task must have exactly 2 cases. "
        f"Problems: {bad_eval_ids}"
    )

print("✓ 60 unique case_id values")
print("✓ 30 English cases")
print("✓ 30 Arabic cases")
print("✓ Exactly 2 cases per source task")
print("✓ Frozen dataset integrity PASSED")

# -------------------------------------------------------------------------
# 5. Verify V3.8.5 pipeline functions
# -------------------------------------------------------------------------

print()
print("=" * 78)
print("V3.8.5 PIPELINE AVAILABILITY CHECK")
print("=" * 78)

required_functions = [
    "hybrid_retrieve",
    "grounded_generation",
    "evaluate_5d",
    "build_refinement_signal",
    "critical_quality_guard",
    "extract_answer_intent",
    "answer_intent_match_score",
    "detect_query_language",
]

missing_functions = [
    fn for fn in required_functions
    if fn not in globals() or not callable(globals()[fn])
]

if missing_functions:
    print("MISSING V3.8.5 FUNCTIONS:")
    for fn in missing_functions:
        print(f"  - {fn}")

    raise RuntimeError(
        "\nSTEP 6B cannot run because the V3.8.5 pipeline "
        "definitions are not loaded in this Runtime.\n\n"
        "Run the main V3.8.5 pipeline cells first, then run STEP 6B.\n"
        "Do NOT rebuild or modify the frozen evaluation dataset."
    )

print("✓ V3.8.5 core functions are available")

# -------------------------------------------------------------------------
# 6. Ensure Tmax is available
# -------------------------------------------------------------------------

# This explicitly fixes the previous:
# NameError: name 'TMAX' is not defined

globals()["TMAX"] = TMAX
globals()["Q_THRESHOLD"] = Q_THRESHOLD

print("✓ TMAX =", TMAX)
print("✓ Q_THRESHOLD =", Q_THRESHOLD)

# -------------------------------------------------------------------------
# 7. Check benchmark_one_case
# -------------------------------------------------------------------------

if "benchmark_one_case" not in globals():
    raise RuntimeError(
        "benchmark_one_case(query) is not defined.\n"
        "The V3.8.5 evaluation adapter must be loaded before STEP 6B."
    )

if not callable(benchmark_one_case):
    raise TypeError(
        "benchmark_one_case exists but is not callable."
    )

print("✓ benchmark_one_case() is available")

# -------------------------------------------------------------------------
# 8. Run frozen evaluation
# -------------------------------------------------------------------------

print()
print("=" * 78)
print("RUNNING FROZEN EVALUATION MATRIX")
print("=" * 78)

results = []

total = len(evaluation_data)

for idx, case in enumerate(evaluation_data, start=1):

    case_id = case["case_id"]
    eval_id = case["eval_id"]
    evaluation_language = case["evaluation_language"]
    programming_language = case["programming_language"]
    task_family = case["task_family"]
    query = case["query"]
    ground_truth = case["ground_truth"]

    print(
        f"[{idx:02d}/{total}] "
        f"{case_id} | "
        f"{evaluation_language} | "
        f"{programming_language} | "
        f"{task_family}"
    )

    start_time = time.time()

    try:

        # -------------------------------------------------------------
        # Run the EXISTING V3.8.5 benchmark function
        # -------------------------------------------------------------

        result = benchmark_one_case(query)

        elapsed = time.time() - start_time

        if isinstance(result, dict):
            row = dict(result)
        else:
            row = {
                "answer": str(result)
            }

        # -------------------------------------------------------------
        # Attach immutable evaluation metadata
        # -------------------------------------------------------------

        row["case_id"] = case_id
        row["eval_id"] = eval_id
        row["evaluation_language"] = evaluation_language
        row["programming_language"] = programming_language
        row["task_family"] = task_family
        row["query"] = query
        row["ground_truth"] = ground_truth
        row["evaluation_time_sec"] = round(elapsed, 4)
        row["evaluation_status"] = "SUCCESS"

        results.append(row)

        # Compact diagnostic
        q_value = row.get("Q(a)", row.get("Q_a", row.get("q_score", None)))
        decision = row.get("decision", "N/A")

        print(
            f"       Decision={decision} | "
            f"Q(a)={q_value} | "
            f"Time={elapsed:.2f}s"
        )

    except Exception as e:

        elapsed = time.time() - start_time

        print(
            f"       ERROR: {type(e).__name__}: {e}"
        )

        results.append({
            "case_id": case_id,
            "eval_id": eval_id,
            "evaluation_language": evaluation_language,
            "programming_language": programming_language,
            "task_family": task_family,
            "query": query,
            "ground_truth": ground_truth,
            "evaluation_time_sec": round(elapsed, 4),
            "evaluation_status": "ERROR",
            "error_type": type(e).__name__,
            "error_message": str(e),
        })

# -------------------------------------------------------------------------
# 9. Convert results to DataFrame
# -------------------------------------------------------------------------

df_results = pd.DataFrame(results)

print()
print("=" * 78)
print("EVALUATION COMPLETED")
print("=" * 78)

print(f"Results generated : {len(df_results)}")

# -------------------------------------------------------------------------
# 10. Validate result count
# -------------------------------------------------------------------------

if len(df_results) != 60:
    raise AssertionError(
        f"Expected 60 evaluation results, "
        f"found {len(df_results)}."
    )

successful = (
    df_results["evaluation_status"] == "SUCCESS"
).sum()

errors = (
    df_results["evaluation_status"] == "ERROR"
).sum()

print(f"Successful cases : {successful}")
print(f"Error cases      : {errors}")

# -------------------------------------------------------------------------
# 11. Save JSON
# -------------------------------------------------------------------------

df_results.to_json(
    RESULT_JSON,
    orient="records",
    force_ascii=False,
    indent=2
)

# -------------------------------------------------------------------------
# 12. Save CSV
# -------------------------------------------------------------------------

df_results.to_csv(
    RESULT_CSV,
    index=False,
    encoding="utf-8-sig"
)

# -------------------------------------------------------------------------
# 13. Save Excel
# -------------------------------------------------------------------------

excel_status = "FAILED"

try:
    df_results.to_excel(
        RESULT_XLSX,
        index=False,
        engine="openpyxl"
    )
    excel_status = "SUCCESS"
except Exception as e:
    print("Excel export warning:", e)

# -------------------------------------------------------------------------
# 14. Summary
# -------------------------------------------------------------------------

print()
print("=" * 78)
print("RESULT FILES")
print("=" * 78)

print(f"JSON  : {RESULT_JSON}")
print(f"CSV   : {RESULT_CSV}")
print(f"XLSX  : {RESULT_XLSX}")
print(f"Excel : {excel_status}")

print()
print("=" * 78)
print("LANGUAGE RESULTS")
print("=" * 78)

if "evaluation_language" in df_results.columns:
    print(
        df_results["evaluation_language"]
        .value_counts(dropna=False)
    )

print()
print("=" * 78)
print("EVALUATION STATUS")
print("=" * 78)

print(
    df_results["evaluation_status"]
    .value_counts(dropna=False)
)

# -------------------------------------------------------------------------
# 15. Preview
# -------------------------------------------------------------------------

print()
print("=" * 78)
print("RESULT PREVIEW")
print("=" * 78)

display(
    df_results.head(10)
)

print()
print("=" * 78)
print("STEP 6B COMPLETED")
print("=" * 78)

X-AutoAlign RAG — STEP 6B
EVALUATION HARNESS
Frozen Dataset : /content/evaluation_dataset_frozen.json
Quality Threshold : 0.7
Tmax : 2

FROZEN DATASET CHECK
Evaluation cases : 60
✓ Required fields present
✓ 60 unique case_id values
✓ 30 English cases
✓ 30 Arabic cases
✓ Exactly 2 cases per source task
✓ Frozen dataset integrity PASSED

V3.8.5 PIPELINE AVAILABILITY CHECK
✓ V3.8.5 core functions are available
✓ TMAX = 2
✓ Q_THRESHOLD = 0.7
✓ benchmark_one_case() is available

RUNNING FROZEN EVALUATION MATRIX
[01/60] EVAL_01_EN | ENGLISH | PYTHON | LIST_ARRAY
       ERROR: IndexError: index out of range in self
[02/60] EVAL_01_AR | ARABIC | PYTHON | LIST_ARRAY
       ERROR: IndexError: index out of range in self
[03/60] EVAL_02_EN | ENGLISH | PYTHON | LIST_ARRAY
       ERROR: IndexError: index out of range in self
[04/60] EVAL_02_AR | ARABIC | PYTHON | LIST_ARRAY
       ERROR: IndexError: index out of range in self
[05/60] EVAL_03_EN | ENGLISH | PYTHON | LIST_ARRAY
       ERROR: IndexErro

<candidate>:5: SyntaxWarning: invalid escape sequence '\_'


       ERROR: IndexError: index out of range in self
[24/60] EVAL_12_AR | ARABIC | PYTHON | STRING
       ERROR: IndexError: index out of range in self
[25/60] EVAL_13_EN | ENGLISH | PYTHON | MATH
       ERROR: IndexError: index out of range in self
[26/60] EVAL_13_AR | ARABIC | PYTHON | MATH
       ERROR: IndexError: index out of range in self
[27/60] EVAL_14_EN | ENGLISH | PYTHON | DEBUGGING
       ERROR: IndexError: index out of range in self
[28/60] EVAL_14_AR | ARABIC | PYTHON | DEBUGGING
       ERROR: IndexError: index out of range in self
[29/60] EVAL_15_EN | ENGLISH | PYTHON | OTHER
       ERROR: IndexError: index out of range in self
[30/60] EVAL_15_AR | ARABIC | PYTHON | OTHER
       ERROR: IndexError: index out of range in self
[31/60] EVAL_16_EN | ENGLISH | PYTHON | SEARCH
       ERROR: IndexError: index out of range in self
[32/60] EVAL_16_AR | ARABIC | PYTHON | SEARCH
       ERROR: IndexError: index out of range in self
[33/60] EVAL_17_EN | ENGLISH | JAVA | MATH
       ER

,case_id,eval_id,evaluation_language,programming_language,task_family,query,ground_truth,evaluation_time_sec,evaluation_status,error_type,error_message
0,EVAL_01_EN,EVAL_01,ENGLISH,PYTHON,LIST_ARRAY,Write a function in Python to generate a list of numbers that is the square of all numbers between 1 and the number provided.,"def generate_square_list(num):\n result = []\n for i in range(1, num+1):\n result.append(i*i)\n return result",1.2317,ERROR,IndexError,index out of range in self
1,EVAL_01_AR,EVAL_01,ARABIC,PYTHON,LIST_ARRAY,اكتب a treative_ Travery in PYTHON_Lang إلى توليد a قائمة من الأرقام التي هي مربع كل الأعداد بين 1 و العدد المقدّم.,"def generate_square_list(num):\n result = []\n for i in range(1, num+1):\n result.append(i*i)\n return result",0.4630,ERROR,IndexError,index out of range in self
2,EVAL_02_EN,EVAL_02,ENGLISH,PYTHON,LIST_ARRAY,So you need to design a class for a Linked List.,"class ListNode:\n def __init__(self, data):\n self.data = data\n self.next = None\n\nclass LinkedList:\n def __init__(self):\n self.head = None\n \n def is_empty(self):\n return self.head == None\n \n # Other method implementations go here",0.1588,ERROR,IndexError,index out of range in self
3,EVAL_02_AR,EVAL_02,ARABIC,PYTHON,LIST_ARRAY,لذا تحتاج إلى تصميم a صف لـ a ملحق قائمة.,"class ListNode:\n def __init__(self, data):\n self.data = data\n self.next = None\n\nclass LinkedList:\n def __init__(self):\n self.head = None\n \n def is_empty(self):\n return self.head == None\n \n # Other method implementations go here",0.2898,ERROR,IndexError,index out of range in self
4,EVAL_03_EN,EVAL_03,ENGLISH,PYTHON,LIST_ARRAY,Write a function that takes an array of integers and returns an array of the same length with each element multiplied by 2.,def multiplyByTwo(arr):\n multipliedArr = []\n for i in arr:\n multipliedArr.append(i*2)\n return multipliedArr,0.3803,ERROR,IndexError,index out of range in self
5,EVAL_03_AR,EVAL_03,ARABIC,PYTHON,LIST_ARRAY,اكتب a الدالة_ text الذي يأخذ ARAY_ صلاحيّة الاعداد الصحيحة ويرجع aRAY_ Tray_ Trayform من نفس الطول مع كل عنصر مضروب في 2.,def multiplyByTwo(arr):\n multipliedArr = []\n for i in arr:\n multipliedArr.append(i*2)\n return multipliedArr,0.5082,ERROR,IndexError,index out of range in self
6,EVAL_04_EN,EVAL_04,ENGLISH,C,STRING,Write a method in C# for finding the longest common subsequence of two strings.,"public static string LCS(string s1, string s2) \n{ \n int m = s1.Length;\n int n = s2.Length;\n int[,] L = new int[m + 1, n + 1]; \n\n for (int i = 0; i <= m; i++) \n { \n for (int j = 0; j <= n; j++) \n { \n if (i == 0 || j == 0) \n L[i, j] = 0; \n else if (s1[i - 1] == s2[j - 1]) \n L[i, j] = L[i - 1, j - 1] + 1; \n else\n L[i, j] = Math.Max(L[i - 1, j], L[i, j - 1]); \n } \n } \n\n int index = L[m, n]; \n int temp = index; \n\n char[] lcs = new char[index + 1]; \n lcs[index] = '\0'; \n\n int i = m, j = n; \n while (i > 0 && j > 0) \n { \n if (s1[i - 1] == s2[j - 1]) \n { \n lcs[index - 1] = s1[i - 1]; \n i--;\n j--;\n index--; \n } \n\n else if (L[i - 1, j] > L[i, j - 1]) \n i--; \n else\n j--;\n } \n return new string(lcs); \n}",0.2275,ERROR,IndexError,index out of range in self
7,EVAL_04_AR,EVAL_04,ARABIC,C,STRING,اكتب طريقة في C# للعثور على أطول تسلسل فرعي مشترك لاثنين من STRING_Times.,"public static string LCS(string s1, string s2) \n{ \n int m = s1.Length;\n int n = s2.Length;\n int[,] L = new int[m + 1, n + 1]; \n\n for (int i = 0; i <= m; i++) \n { \n for (int j = 0; j <= n; j++) \n { \n if (i == 0 || j == 0) \n L[i, j] = 0; \n else if (s1[i - 1] == s2[j - 1]) \n L[i, j] = L[i - 1, j - 1] + 1; \n else\n L[i, j] = Math.Max(L[i - 1, j], L[i, j - 1]); \n } \n } \n\n int index = L[m, n]; \n int temp = index; \n\n char[] lcs = new char[index + 1]; \n lcs[index] = '\0'; \n\n int i = m, j = n; \n while (i > 0 && j > 0) \n { \n if (s1[i - 1] == s2[j - 1]) \n { \n lcs[index - 1] = s1[i - 1]; \n i--;\n j--;\n index--; \n } \n\n else if (L[i - 1, j] > L[i, j - 1]) \n i--; \n else\n j--;\n } \n return new s


STEP 6B COMPLETED


In [ ]:
# =====================================================================
# X-AutoAlign — Offline Alignment Training (CPO + LoRA)
# Corrected for speed + resilience to Colab disconnects
# =====================================================================
#
# FIXES vs the previous version:
# 1) GPU is verified explicitly before training starts — if you're on
#    CPU, this tells you immediately instead of silently crawling.
# 2) The preference dataset no longer repeats 3 identical examples 350x.
#    That produced zero additional learning signal and just inflated
#    step count for no benefit. Replaced with a small, genuinely
#    DIVERSE set of instruction/chosen/rejected pairs across several
#    task families (matching the ones your RAG pipeline already
#    recognizes), each repeated only a few times across epochs.
# 3) Checkpointing is enabled (save_strategy="steps"): if the Colab
#    runtime disconnects, you resume from the last checkpoint instead
#    of losing everything.
# 4) fp16 is enabled automatically when a GPU is available.
# 5) A hard max_steps cap keeps a single run bounded and predictable —
#    important for a thesis demo where you need "it finished and I can
#    show the adapter is active," not a fully converged model.
# 6) NEW — Google Drive persistence: this script and the main pipeline
#    are normally run in SEPARATE Colab sessions (training is a one-off,
#    the RAG pipeline is what you demo repeatedly). /content is wiped
#    every time the runtime resets, so saving the adapter there means
#    it silently disappears the moment you reconnect. If PERSIST_TO_DRIVE
#    is True (default), the adapter and checkpoints are saved under
#    Google Drive instead, so they survive across sessions. See
#    THESIS_EXECUTION_GUIDE.md for the full multi-notebook workflow.
#
# IMPORTANT COMPATIBILITY NOTE:
# The adapter produced here MUST be loaded onto the SAME base model
# used by the main pipeline's GENERATOR_NAME ("google/flan-t5-base").
# If you ever change model_id below, you must also change GENERATOR_NAME
# in the main X-AutoAlign script — a LoRA adapter trained on one base
# model will not load correctly onto a different one.
# =====================================================================

!pip install -q -U trl transformers accelerate peft datasets torchao

import os
import torch
import pandas as pd
from datasets import Dataset
from peft import LoraConfig
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from trl.experimental.cpo import CPOConfig, CPOTrainer

# ---------------------------------------------------------------------
# -1) Persistence: mount Google Drive so the adapter survives across
#     separate Colab sessions (training notebook vs pipeline notebook).
#     Set PERSIST_TO_DRIVE = False to keep everything local to /content
#     only (fine if you train and serve in the SAME uninterrupted
#     session, but you will lose the adapter on any disconnect).
# ---------------------------------------------------------------------
PERSIST_TO_DRIVE = True
DRIVE_ROOT = "/content/drive/MyDrive/XAutoAlign"

if PERSIST_TO_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
        os.makedirs(DRIVE_ROOT, exist_ok=True)
        BASE_DIR = DRIVE_ROOT
        print(f"✅ Google Drive mounted. Artifacts will persist under: {BASE_DIR}")
    except Exception as e:
        print(f"⚠️  Could not mount Google Drive ({type(e).__name__}: {e}). "
              f"Falling back to local /content storage — the adapter will "
              f"NOT survive a runtime reset.")
        BASE_DIR = "/content"
else:
    BASE_DIR = "/content"

# ---------------------------------------------------------------------
# 0) GPU check — do this FIRST, before spending any time on data/model.
# ---------------------------------------------------------------------
print("=" * 60)
if torch.cuda.is_available():
    print(f"✅ GPU detected: {torch.cuda.get_device_name(0)}")
    USE_GPU = True
else:
    print(
        "⚠️  No GPU detected — training will run on CPU and will be "
        "MUCH slower (this is very likely why your previous run crawled "
        "at ~0.07 it/s). In Colab: Runtime → Change runtime type → "
        "Hardware accelerator → GPU (T4), then Reconnect and rerun."
    )
    USE_GPU = False
print("=" * 60)

# ---------------------------------------------------------------------
# 1) Preference dataset — diverse, not repeated-identical
# ---------------------------------------------------------------------
print("📦 1. Building the offline preference dataset...")

# Each entry: a real instruction + a genuinely correct "chosen" solution +
# a genuinely flawed "rejected" one (wrong language / incomplete / buggy).
# Task families intentionally mirror TASK_FAMILY_PATTERNS in the main
# pipeline so the alignment signal is thematically relevant to it.
BASE_EXAMPLES = [
    {
        "instruction": "Write a Python function to check if a number is even.",
        "input": "",
        "chosen": "def is_even(n):\n    return n % 2 == 0",
        "rejected": "function isEven(n) {\n  return n % 2 == 0;\n}",
    },
    {
        "instruction": "Create a function that returns the square of a number.",
        "input": "",
        "chosen": "def square(n):\n    return n ** 2",
        "rejected": "def square(n):\n    # TODO\n    pass",
    },
    {
        "instruction": "Write a Python function to find the maximum in a list.",
        "input": "nums = [3, 7, 2]",
        "chosen": "def find_max(nums):\n    return max(nums)",
        "rejected": "def find_max(nums):\n    return nums[0]  # wrong: ignores rest of list",
    },
    {
        "instruction": "Write a Python function to check if a string is a palindrome.",
        "input": "",
        "chosen": (
            "def is_palindrome(s):\n"
            "    return s == s[::-1]"
        ),
        "rejected": "def is_palindrome(s):\n    return True  # always wrong",
    },
    {
        "instruction": "Write a Python function to calculate the factorial of a number.",
        "input": "",
        "chosen": (
            "def factorial(n):\n"
            "    if n <= 1:\n"
            "        return 1\n"
            "    return n * factorial(n - 1)"
        ),
        "rejected": "def factorial(n):\n    return n * n  # wrong formula",
    },
    {
        "instruction": "Write a Python function to reverse a list.",
        "input": "",
        "chosen": "def reverse_list(lst):\n    return lst[::-1]",
        "rejected": "def reverse_list(lst):\n    return lst  # not reversed",
    },
    {
        "instruction": "Write a Python function to sort a list of numbers.",
        "input": "",
        "chosen": "def sort_numbers(nums):\n    return sorted(nums)",
        "rejected": "public int[] sortNumbers(int[] nums) { return nums; }",
    },
    {
        "instruction": "Write a Python function to check if a number is prime.",
        "input": "",
        "chosen": (
            "def is_prime(n):\n"
            "    if n < 2:\n"
            "        return False\n"
            "    for i in range(2, int(n ** 0.5) + 1):\n"
            "        if n % i == 0:\n"
            "            return False\n"
            "    return True"
        ),
        "rejected": "def is_prime(n):\n    pass  # incomplete",
    },
    {
        "instruction": "Write a Python function to compute the sum of a list of numbers.",
        "input": "",
        "chosen": "def list_sum(nums):\n    return sum(nums)",
        "rejected": "let listSum = (nums) => nums.reduce((a,b) => a+b);",
    },
    {
        "instruction": "Write a Python function to check if two strings are anagrams.",
        "input": "",
        "chosen": "def is_anagram(a, b):\n    return sorted(a) == sorted(b)",
        "rejected": "def is_anagram(a, b):\n    return a == b  # wrong: checks equality, not anagram",
    },
]

# Repeat a bounded, sane number of times (NOT 350x) so the trainer sees
# each pattern a few times across the epoch without wasting compute on
# pure duplication.
REPEATS = 6  # 10 examples * 6 = 60 rows; adjust for a longer/shorter run
synthetic_data = BASE_EXAMPLES * REPEATS

preference_data = []
for row in synthetic_data:
    prompt = f"Instruction: {row['instruction']}\nInput: {row['input']}\nProvide a Python solution."
    preference_data.append({
        "prompt": prompt,
        "chosen": row["chosen"],
        "rejected": row["rejected"],
    })

dataset = Dataset.from_pandas(pd.DataFrame(preference_data)).shuffle(seed=42)
print(f"✅ Dataset built — {len(dataset)} preference pairs "
      f"({len(BASE_EXAMPLES)} unique examples x {REPEATS} repeats).\n")

# ---------------------------------------------------------------------
# 2) Model + tokenizer
# ---------------------------------------------------------------------
model_id = "google/flan-t5-base"  # MUST match GENERATOR_NAME in the main pipeline
print(f"📥 2. Loading base model ({model_id})...")

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSeq2SeqLM.from_pretrained(
    model_id,
    device_map="auto" if USE_GPU else None,
)

# ---------------------------------------------------------------------
# 3) LoRA + CPO training config
# ---------------------------------------------------------------------
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="SEQ_2_SEQ_LM",
    target_modules=["q", "v"],
)

CHECKPOINT_DIR = f"{BASE_DIR}/xautoalign_lora_cpo_checkpoints"

training_args = CPOConfig(
    output_dir=CHECKPOINT_DIR,
    num_train_epochs=2,
    per_device_train_batch_size=8 if USE_GPU else 4,
    gradient_accumulation_steps=1 if USE_GPU else 2,
    learning_rate=5e-5,
    logging_steps=10,
    # Checkpoint periodically so a Colab disconnect does not cost you
    # the whole run — resume with resume_from_checkpoint=True below.
    save_strategy="steps",
    save_steps=25,
    save_total_limit=2,
    fp16=USE_GPU,      # only meaningful on GPU
    bf16=False,
    report_to="none",
    remove_unused_columns=False,
    beta=0.1,
    max_length=256,    # your examples are short; smaller = faster steps
    max_steps=150,      # hard cap: bounded, predictable run time
)

# ---------------------------------------------------------------------
# 4) Trainer
# ---------------------------------------------------------------------
trainer = CPOTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer,
    peft_config=peft_config,
)

# Resume automatically if a previous checkpoint exists in CHECKPOINT_DIR
# (e.g. after a disconnect) instead of starting over from scratch.
resume = os.path.isdir(CHECKPOINT_DIR) and any(
    name.startswith("checkpoint-") for name in os.listdir(CHECKPOINT_DIR)
) if os.path.isdir(CHECKPOINT_DIR) else False

print("\n🚀 3. Starting CPO/LoRA offline alignment training"
      f"{' (resuming from checkpoint)' if resume else ''}...")
trainer.train(resume_from_checkpoint=resume)

# ---------------------------------------------------------------------
# 5) Save the final adapter and wire it into the main pipeline
# ---------------------------------------------------------------------
OUTPUT_ADAPTER_DIR = f"{BASE_DIR}/xautoalign_lora_cpo_final"
trainer.model.save_pretrained(OUTPUT_ADAPTER_DIR)
tokenizer.save_pretrained(OUTPUT_ADAPTER_DIR)

os.environ["XAUTOALIGN_LORA_ADAPTER"] = os.path.abspath(OUTPUT_ADAPTER_DIR)

print("\n" + "=" * 60)
print("🎉 Training complete.")
print(f"📁 Adapter saved to: {OUTPUT_ADAPTER_DIR}")
print(f"🔗 XAUTOALIGN_LORA_ADAPTER = {os.environ['XAUTOALIGN_LORA_ADAPTER']}")
print("Run the main X-AutoAlign pipeline cell now — ALIGNMENT_STATUS")
print("should read PREFERENCE_ALIGNED_LORA_ACTIVE instead of")
print("BASE_MODEL_NO_ADAPTER.")
print("=" * 60)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Google Drive mounted. Artifacts will persist under: /content/drive/MyDrive/XAutoAlign
⚠️  No GPU detected — training will run on CPU and will be MUCH slower (this is very likely why your previous run crawled at ~0.07 it/s). In Colab: Runtime → Change runtime type → Hardware accelerator → GPU (T4), then Reconnect and rerun.
📦 1. Building the offline preference dataset...
✅ Dataset built — 60 preference pairs (10 unique examples x 6 repeats).

📥 2. Loading base model (google/flan-t5-base)...


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Map:   0%|          | 0/60 [00:00<?, ? examples/s]

Map:   0%|          | 0/60 [00:00<?, ? examples/s]

Map:   0%|          | 0/60 [00:00<?, ? examples/s]


🚀 3. Starting CPO/LoRA offline alignment training (resuming from checkpoint)...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss



🎉 Training complete.
📁 Adapter saved to: /content/drive/MyDrive/XAutoAlign/xautoalign_lora_cpo_final
🔗 XAUTOALIGN_LORA_ADAPTER = /content/drive/MyDrive/XAutoAlign/xautoalign_lora_cpo_final
Run the main X-AutoAlign pipeline cell now — ALIGNMENT_STATUS
should read PREFERENCE_ALIGNED_LORA_ACTIVE instead of
BASE_MODEL_NO_ADAPTER.


In [ ]:
# =============================================================================
# X-AutoAlign RAG V3.8.5
# STEP 6B — EVALUATION HARNESS
# =============================================================================
#
# Purpose:
#   Run the FROZEN Evaluation Matrix (60 cases)
#   through the existing V3.8.5 pipeline.
#
# IMPORTANT:
#   - Does NOT modify the frozen dataset
#   - Does NOT modify Ground Truth
#   - Does NOT create new evaluation cases
#   - Evaluates exactly 60 cases
#   - English = 30
#   - Arabic = 30
# =============================================================================

import os
import json
import time
import traceback
import pandas as pd
from pathlib import Path

print("=" * 78)
print("X-AutoAlign RAG — STEP 6B")
print("EVALUATION HARNESS")
print("=" * 78)

# -------------------------------------------------------------------------
# 1. Configuration
# -------------------------------------------------------------------------

FROZEN_DATASET = Path("/content/evaluation_dataset_frozen.json")

RESULT_JSON = Path("/content/evaluation_results_v1.json")
RESULT_CSV  = Path("/content/evaluation_results_v1.csv")
RESULT_XLSX = Path("/content/evaluation_results_v1.xlsx")

# V3.8.5 evaluation settings
Q_THRESHOLD = 0.70
TMAX = 2

print(f"Frozen Dataset : {FROZEN_DATASET}")
print(f"Quality Threshold : {Q_THRESHOLD}")
print(f"Tmax : {TMAX}")

# -------------------------------------------------------------------------
# 2. Verify frozen dataset
# -------------------------------------------------------------------------

if not FROZEN_DATASET.exists():
    raise FileNotFoundError(
        f"Frozen evaluation dataset not found:\n{FROZEN_DATASET}"
    )

with open(FROZEN_DATASET, "r", encoding="utf-8") as f:
    evaluation_data = json.load(f)

if not isinstance(evaluation_data, list):
    raise TypeError(
        "Frozen evaluation dataset must be a JSON list."
    )

print()
print("=" * 78)
print("FROZEN DATASET CHECK")
print("=" * 78)

print(f"Evaluation cases : {len(evaluation_data)}")

if len(evaluation_data) != 60:
    raise AssertionError(
        f"Expected exactly 60 evaluation cases, "
        f"found {len(evaluation_data)}."
    )

# -------------------------------------------------------------------------
# 3. Required columns
# -------------------------------------------------------------------------

required_columns = [
    "case_id",
    "eval_id",
    "evaluation_language",
    "programming_language",
    "task_family",
    "query",
    "ground_truth",
    "ground_truth_status",
]

for col in required_columns:
    missing = [
        i for i, row in enumerate(evaluation_data)
        if col not in row
    ]

    if missing:
        raise AssertionError(
            f"Required column '{col}' is missing in rows: {missing[:10]}"
        )

print("✓ Required fields present")

# -------------------------------------------------------------------------
# 4. Frozen integrity checks
# -------------------------------------------------------------------------

case_ids = [row["case_id"] for row in evaluation_data]
eval_ids = [row["eval_id"] for row in evaluation_data]
languages = [row["evaluation_language"] for row in evaluation_data]

if len(set(case_ids)) != 60:
    raise AssertionError("case_id values are not unique.")

if languages.count("ENGLISH") != 30:
    raise AssertionError(
        f"Expected 30 ENGLISH cases, "
        f"found {languages.count('ENGLISH')}"
    )

if languages.count("ARABIC") != 30:
    raise AssertionError(
        f"Expected 30 ARABIC cases, "
        f"found {languages.count('ARABIC')}"
    )

# Exactly two cases for every source task
from collections import Counter

eval_id_counts = Counter(eval_ids)

bad_eval_ids = {
    k: v for k, v in eval_id_counts.items()
    if v != 2
}

if bad_eval_ids:
    raise AssertionError(
        f"Each source task must have exactly 2 cases. "
        f"Problems: {bad_eval_ids}"
    )

print("✓ 60 unique case_id values")
print("✓ 30 English cases")
print("✓ 30 Arabic cases")
print("✓ Exactly 2 cases per source task")
print("✓ Frozen dataset integrity PASSED")

# -------------------------------------------------------------------------
# 5. Verify V3.8.5 pipeline functions
# -------------------------------------------------------------------------

print()
print("=" * 78)
print("V3.8.5 PIPELINE AVAILABILITY CHECK")
print("=" * 78)

required_functions = [
    "hybrid_retrieve",
    "grounded_generation",
    "evaluate_5d",
    "build_refinement_signal",
    "critical_quality_guard",
    "extract_answer_intent",
    "answer_intent_match_score",
    "detect_query_language",
]

missing_functions = [
    fn for fn in required_functions
    if fn not in globals() or not callable(globals()[fn])
]

if missing_functions:
    print("MISSING V3.8.5 FUNCTIONS:")
    for fn in missing_functions:
        print(f"  - {fn}")

    raise RuntimeError(
        "\nSTEP 6B cannot run because the V3.8.5 pipeline "
        "definitions are not loaded in this Runtime.\n\n"
        "Run the main V3.8.5 pipeline cells first, then run STEP 6B.\n"
        "Do NOT rebuild or modify the frozen evaluation dataset."
    )

print("✓ V3.8.5 core functions are available")

# -------------------------------------------------------------------------
# 6. Ensure Tmax is available
# -------------------------------------------------------------------------

# This explicitly fixes the previous:
# NameError: name 'TMAX' is not defined

globals()["TMAX"] = TMAX
globals()["Q_THRESHOLD"] = Q_THRESHOLD

print("✓ TMAX =", TMAX)
print("✓ Q_THRESHOLD =", Q_THRESHOLD)

# -------------------------------------------------------------------------
# 7. Check benchmark_one_case
# -------------------------------------------------------------------------

if "benchmark_one_case" not in globals():
    raise RuntimeError(
        "benchmark_one_case(query) is not defined.\n"
        "The V3.8.5 evaluation adapter must be loaded before STEP 6B."
    )

if not callable(benchmark_one_case):
    raise TypeError(
        "benchmark_one_case exists but is not callable."
    )

print("✓ benchmark_one_case() is available")

# -------------------------------------------------------------------------
# 8. Run frozen evaluation
# -------------------------------------------------------------------------

print()
print("=" * 78)
print("RUNNING FROZEN EVALUATION MATRIX")
print("=" * 78)

results = []

total = len(evaluation_data)

for idx, case in enumerate(evaluation_data, start=1):

    case_id = case["case_id"]
    eval_id = case["eval_id"]
    evaluation_language = case["evaluation_language"]
    programming_language = case["programming_language"]
    task_family = case["task_family"]
    query = case["query"]
    ground_truth = case["ground_truth"]

    print(
        f"[{idx:02d}/{total}] "
        f"{case_id} | "
        f"{evaluation_language} | "
        f"{programming_language} | "
        f"{task_family}"
    )

    start_time = time.time()

    try:

        # -------------------------------------------------------------
        # Run the EXISTING V3.8.5 benchmark function
        # -------------------------------------------------------------

        result = benchmark_one_case(query)

        elapsed = time.time() - start_time

        if isinstance(result, dict):
            row = dict(result)
        else:
            row = {
                "answer": str(result)
            }

        # -------------------------------------------------------------
        # Attach immutable evaluation metadata
        # -------------------------------------------------------------

        row["case_id"] = case_id
        row["eval_id"] = eval_id
        row["evaluation_language"] = evaluation_language
        row["programming_language"] = programming_language
        row["task_family"] = task_family
        row["query"] = query
        row["ground_truth"] = ground_truth
        row["evaluation_time_sec"] = round(elapsed, 4)
        row["evaluation_status"] = "SUCCESS"

        results.append(row)

        # Compact diagnostic
        q_value = row.get("Q(a)", row.get("Q_a", row.get("q_score", None)))
        decision = row.get("decision", "N/A")

        print(
            f"       Decision={decision} | "
            f"Q(a)={q_value} | "
            f"Time={elapsed:.2f}s"
        )

    except Exception as e:

        elapsed = time.time() - start_time

        print(
            f"       ERROR: {type(e).__name__}: {e}"
        )

        results.append({
            "case_id": case_id,
            "eval_id": eval_id,
            "evaluation_language": evaluation_language,
            "programming_language": programming_language,
            "task_family": task_family,
            "query": query,
            "ground_truth": ground_truth,
            "evaluation_time_sec": round(elapsed, 4),
            "evaluation_status": "ERROR",
            "error_type": type(e).__name__,
            "error_message": str(e),
        })

# -------------------------------------------------------------------------
# 9. Convert results to DataFrame
# -------------------------------------------------------------------------

df_results = pd.DataFrame(results)

print()
print("=" * 78)
print("EVALUATION COMPLETED")
print("=" * 78)

print(f"Results generated : {len(df_results)}")

# -------------------------------------------------------------------------
# 10. Validate result count
# -------------------------------------------------------------------------

if len(df_results) != 60:
    raise AssertionError(
        f"Expected 60 evaluation results, "
        f"found {len(df_results)}."
    )

successful = (
    df_results["evaluation_status"] == "SUCCESS"
).sum()

errors = (
    df_results["evaluation_status"] == "ERROR"
).sum()

print(f"Successful cases : {successful}")
print(f"Error cases      : {errors}")

# -------------------------------------------------------------------------
# 11. Save JSON
# -------------------------------------------------------------------------

df_results.to_json(
    RESULT_JSON,
    orient="records",
    force_ascii=False,
    indent=2
)

# -------------------------------------------------------------------------
# 12. Save CSV
# -------------------------------------------------------------------------

df_results.to_csv(
    RESULT_CSV,
    index=False,
    encoding="utf-8-sig"
)

# -------------------------------------------------------------------------
# 13. Save Excel
# -------------------------------------------------------------------------

excel_status = "FAILED"

try:
    df_results.to_excel(
        RESULT_XLSX,
        index=False,
        engine="openpyxl"
    )
    excel_status = "SUCCESS"
except Exception as e:
    print("Excel export warning:", e)

# -------------------------------------------------------------------------
# 14. Summary
# -------------------------------------------------------------------------

print()
print("=" * 78)
print("RESULT FILES")
print("=" * 78)

print(f"JSON  : {RESULT_JSON}")
print(f"CSV   : {RESULT_CSV}")
print(f"XLSX  : {RESULT_XLSX}")
print(f"Excel : {excel_status}")

print()
print("=" * 78)
print("LANGUAGE RESULTS")
print("=" * 78)

if "evaluation_language" in df_results.columns:
    print(
        df_results["evaluation_language"]
        .value_counts(dropna=False)
    )

print()
print("=" * 78)
print("EVALUATION STATUS")
print("=" * 78)

print(
    df_results["evaluation_status"]
    .value_counts(dropna=False)
)

# -------------------------------------------------------------------------
# 15. Preview
# -------------------------------------------------------------------------

print()
print("=" * 78)
print("RESULT PREVIEW")
print("=" * 78)

display(
    df_results.head(10)
)

print()
print("=" * 78)
print("STEP 6B COMPLETED")
print("=" * 78)

X-AutoAlign RAG — STEP 6B
EVALUATION HARNESS
Frozen Dataset : /content/evaluation_dataset_frozen.json
Quality Threshold : 0.7
Tmax : 2

FROZEN DATASET CHECK
Evaluation cases : 60
✓ Required fields present
✓ 60 unique case_id values
✓ 30 English cases
✓ 30 Arabic cases
✓ Exactly 2 cases per source task
✓ Frozen dataset integrity PASSED

V3.8.5 PIPELINE AVAILABILITY CHECK
✓ V3.8.5 core functions are available
✓ TMAX = 2
✓ Q_THRESHOLD = 0.7
✓ benchmark_one_case() is available

RUNNING FROZEN EVALUATION MATRIX
[01/60] EVAL_01_EN | ENGLISH | PYTHON | LIST_ARRAY
       Decision=MAX_REACHED | Q(a)=0.949 | Time=53.23s
[02/60] EVAL_01_AR | ARABIC | PYTHON | LIST_ARRAY
       Decision=MAX_REACHED | Q(a)=0.552 | Time=9.96s
[03/60] EVAL_02_EN | ENGLISH | PYTHON | LIST_ARRAY
       Decision=MAX_REACHED | Q(a)=0.576 | Time=36.36s
[04/60] EVAL_02_AR | ARABIC | PYTHON | LIST_ARRAY
       Decision=MAX_REACHED | Q(a)=0.668 | Time=73.00s
[05/60] EVAL_03_EN | ENGLISH | PYTHON | LIST_ARRAY
       Decision=

<candidate>:5: SyntaxWarning: invalid escape sequence '\_'
<candidate>:5: SyntaxWarning: invalid escape sequence '\_'
<candidate>:5: SyntaxWarning: invalid escape sequence '\_'


       Decision=MAX_REACHED | Q(a)=0.956 | Time=65.43s
[24/60] EVAL_12_AR | ARABIC | PYTHON | STRING
       Decision=ACCEPT | Q(a)=0.952 | Time=12.31s
[25/60] EVAL_13_EN | ENGLISH | PYTHON | MATH
       Decision=MAX_REACHED | Q(a)=0.63 | Time=20.88s
[26/60] EVAL_13_AR | ARABIC | PYTHON | MATH
       Decision=MAX_REACHED | Q(a)=0.69 | Time=65.61s
[27/60] EVAL_14_EN | ENGLISH | PYTHON | DEBUGGING
       Decision=MAX_REACHED | Q(a)=0.81 | Time=26.49s
[28/60] EVAL_14_AR | ARABIC | PYTHON | DEBUGGING
       Decision=MAX_REACHED | Q(a)=0.743 | Time=86.46s
[29/60] EVAL_15_EN | ENGLISH | PYTHON | OTHER
       Decision=MAX_REACHED | Q(a)=0.794 | Time=30.13s
[30/60] EVAL_15_AR | ARABIC | PYTHON | OTHER
       Decision=MAX_REACHED | Q(a)=0.836 | Time=62.22s
[31/60] EVAL_16_EN | ENGLISH | PYTHON | SEARCH
       Decision=MAX_REACHED | Q(a)=0.937 | Time=54.99s
[32/60] EVAL_16_AR | ARABIC | PYTHON | SEARCH
       Decision=MAX_REACHED | Q(a)=0.62 | Time=27.73s
[33/60] EVAL_17_EN | ENGLISH | JAVA | MAT

,decision,Q(a),iterations,best_iteration,generation_mode,alignment_status,critical_guard,answer_intent_score,best_valid_candidate,scores,...,trace,case_id,eval_id,evaluation_language,programming_language,task_family,query,ground_truth,evaluation_time_sec,evaluation_status
0,MAX_REACHED,0.949,3,1,GROUNDED_CONTEXT_FALLBACK,PREFERENCE_ALIGNED_LORA_ACTIVE,FAIL,0.650,False,"{'Context Relevance': 1.0, 'Factual Consistency': 0.955, 'Faithfulness': 0.955, 'Completeness': 0.833, 'Language Quality': 1.0}",...,"X-AutoAlign V3.8.5 Diagnostic Summary\n======================================================================\nFinal Decision: MAX_REACHED\nBest Q(a): 0.9485\nBest Iteration: 1\nBest Valid Candidate: NO\nOffline Alignment Status: PREFERENCE_ALIGNED_LORA_ACTIVE\nAnswer Intent Score: 0.6500\nAnswer Intent:\n{\n ""programming_language"": ""PYTHON"",\n ""task_family"": ""UNKNOWN"",\n ""operation"": ""UNKNOWN"",\n ""input_type"": ""UNKNOWN"",\n ""task_type"": ""FUNCTION""\n}\n\nIntent Summary:\n{\n ""programming_language"": ""PYTHON"",\n ""task_family"": ""UNKNOWN"",\n ""operation"": ""GENERATE"",\n ""input_type"": ""NUMBER"",\n ""task_type"": ""FUNCTION""\n}\n\nIteration Trace:\n[\n {\n ""iteration"": 0,\n ""query_language"": ""ENGLISH"",\n ""query_used_for_retrieval"": ""Write a function in Python to generate a list of numbers that is the square of all numbers between 1 and the number provided."",\n ""original_query"": ""Write a function in Python to generate a list of numbers that is the square of all numbers bet...",EVAL_01_EN,EVAL_01,ENGLISH,PYTHON,LIST_ARRAY,Write a function in Python to generate a list of numbers that is the square of all numbers between 1 and the number provided.,"def generate_square_list(num):\n result = []\n for i in range(1, num+1):\n result.append(i*i)\n return result",53.2336,SUCCESS
1,MAX_REACHED,0.552,3,1,GENERATED,PREFERENCE_ALIGNED_LORA_ACTIVE,FAIL,0.500,False,"{'Context Relevance': 0.939, 'Factual Consistency': 0.56, 'Faithfulness': 0.635, 'Completeness': 0.328, 'Language Quality': 0.3}",...,"X-AutoAlign V3.8.5 Diagnostic Summary\n======================================================================\nFinal Decision: MAX_REACHED\nBest Q(a): 0.5523\nBest Iteration: 1\nBest Valid Candidate: NO\nOffline Alignment Status: PREFERENCE_ALIGNED_LORA_ACTIVE\nAnswer Intent Score: 0.5000\nAnswer Intent:\n{\n ""programming_language"": ""PYTHON"",\n ""task_family"": ""UNKNOWN"",\n ""operation"": ""UNKNOWN"",\n ""input_type"": ""UNKNOWN"",\n ""task_type"": ""UNKNOWN""\n}\n\nIntent Summary:\n{\n ""programming_language"": ""UNKNOWN"",\n ""task_family"": ""UNKNOWN"",\n ""operation"": ""COUNT"",\n ""input_type"": ""NUMBER"",\n ""task_type"": ""UNKNOWN""\n}\n\nIteration Trace:\n[\n {\n ""iteration"": 0,\n ""query_language"": ""ARABIC"",\n ""query_used_for_retrieval"": ""اكتب a treative_ Travery in PYTHON_Lang إلى توليد a قائمة من الأرقام التي هي مربع كل الأعداد بين 1 و العدد المقدّم."",\n ""original_query"": ""اكتب a treative_ Travery in PYTHON_Lang إلى توليد a قائمة من الأرقام التي هي مربع كل الأعداد بين 1 و العدد ال...",EVAL_01_AR,EVAL_01,ARABIC,PYTHON,LIST_ARRAY,اكتب a treative_ Travery in PYTHON_Lang إلى توليد a قائمة من الأرقام التي هي مربع كل الأعداد بين 1 و العدد المقدّم.,"def generate_square_list(num):\n result = []\n for i in range(1, num+1):\n result.append(i*i)\n return result",9.9634,SUCCESS
2,MAX_REACHED,0.576,3,1,GENERATED,PREFERENCE_ALIGNED_LORA_ACTIVE,FAIL,0.450,False,"{'Context Relevance': 1.0, 'Factual Consistency': 0.568, 'Faithfulness': 0.688, 'Completeness': 0.325, 'Language Quality': 0.3}",...,"X-AutoAlign V3.8.5 Diagnostic Summary\n======================================================================\nFinal Decision: MAX_REACHED\nBest Q(a): 0.5760\nBest Iteration: 1\nBest Valid Candidate: NO\nOffline Alignment Status: PREFERENCE_ALIGNED_LORA_ACTIVE\nAnswer Intent Score: 0.4500\nAnswer Intent:\n{\n ""programming_language"": ""PYTHON"",\n ""task_family"": ""UNKNOWN"",\n ""operation"": ""UNKNOWN"",\


STEP 6B COMPLETED


In [ ]:
# ============================================================================
# X-AutoAlign RAG V3.8.5.1
# STEP 6C — DESCRIPTIVE / DIAGNOSTIC EVALUATION
# READ-ONLY ANALYSIS OF STEP 6B RESULTS
#
# IMPORTANT:
# - NO MODEL EXECUTION
# - NO TRAINING
# - NO DATASET MODIFICATION
# - NO EVALUATION MATRIX MODIFICATION
# - NO HOLM CORRECTION HERE
# - HOLM CORRECTION IS RESERVED FOR STEP 6D — 5D INFERENTIAL TESTS ONLY
# ============================================================================

from pathlib import Path
import json
import pandas as pd
import numpy as np

print("=" * 78)
print("X-AutoAlign RAG V3.8.5.1 — STEP 6C")
print("DESCRIPTIVE / DIAGNOSTIC EVALUATION")
print("=" * 78)

# ============================================================================
# CONFIGURATION
# ============================================================================

RESULT_JSON = Path("/content/evaluation_results_v1.json")
RESULT_CSV  = Path("/content/evaluation_results_v1.csv")

FROZEN_DATASET = Path(
    "/content/evaluation_dataset_frozen.json"
)

EVALUATION_MATRIX = Path(
    "/content/evaluation_matrix_v1.json"
)

OUTPUT_DIR = Path("/content/step6c_analysis")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_CASES = 60
EXPECTED_SOURCE_TASKS = 30
Q_THRESHOLD = 0.70
TMAX = 2

print(f"""
Expected cases       : {EXPECTED_CASES}
Expected source tasks: {EXPECTED_SOURCE_TASKS}
Q threshold          : {Q_THRESHOLD}
Tmax                 : {TMAX}
Output directory     : {OUTPUT_DIR}
""")

# ============================================================================
# [1] INPUT FILE CHECK
# ============================================================================

print("=" * 78)
print("[1] INPUT FILE CHECK")
print("-" * 78)

if not RESULT_JSON.exists():
    raise FileNotFoundError(
        f"STEP 6B JSON not found:\n{RESULT_JSON}"
    )

if not RESULT_CSV.exists():
    raise FileNotFoundError(
        f"STEP 6B CSV not found:\n{RESULT_CSV}"
    )

if not FROZEN_DATASET.exists():
    raise FileNotFoundError(
        f"Frozen dataset not found:\n{FROZEN_DATASET}"
    )

if not EVALUATION_MATRIX.exists():
    raise FileNotFoundError(
        f"Evaluation matrix not found:\n{EVALUATION_MATRIX}"
    )

print(f"✓ STEP 6B JSON found")
print(f"✓ STEP 6B CSV found")
print(f"✓ Frozen dataset found")
print(f"✓ Evaluation matrix found")

# ============================================================================
# [2] LOAD STEP 6B RESULTS
# ============================================================================

print("\n" + "=" * 78)
print("[2] LOAD STEP 6B RESULTS")
print("-" * 78)

df = pd.read_csv(RESULT_CSV)

print(f"Rows loaded    : {len(df)}")
print(f"Columns loaded : {len(df.columns)}")

if len(df) != EXPECTED_CASES:
    raise ValueError(
        f"Expected {EXPECTED_CASES} STEP 6B results, "
        f"found {len(df)}."
    )

print(f"✓ Exactly {EXPECTED_CASES} STEP 6B rows loaded")

# ============================================================================
# [3] REQUIRED COLUMN CHECK
# ============================================================================

print("\n" + "=" * 78)
print("[3] REQUIRED COLUMN CHECK")
print("-" * 78)

required_columns = [
    "case_id",
    "eval_id",
    "evaluation_language",
    "programming_language",
    "task_family",
    "query",
    "ground_truth",
    "decision",
    "Q(a)",
    "iterations",
    "generation_mode",
    "alignment_status",
    "evaluation_status",
]

# Accept alternate Q column naming if necessary
if "Q(a)" not in df.columns:

    q_candidates = [
        "q",
        "Q",
        "quality_score",
        "q_score",
    ]

    found_q = None

    for c in q_candidates:
        if c in df.columns:
            found_q = c
            break

    if found_q is not None:
        df["Q(a)"] = pd.to_numeric(
            df[found_q],
            errors="coerce"
        )
        print(
            f"ℹ Q(a) reconstructed from column: {found_q}"
        )

missing = [
    c for c in required_columns
    if c not in df.columns
]

if missing:
    raise ValueError(
        "Missing required STEP 6B columns:\n"
        + "\n".join(f"  - {x}" for x in missing)
    )

print("✓ All required columns are present")

# ============================================================================
# [4] BASIC DATA INTEGRITY
# ============================================================================

print("\n" + "=" * 78)
print("[4] BASIC DATA INTEGRITY")
print("-" * 78)

unique_cases = df["case_id"].nunique()
unique_eval_ids = df["eval_id"].nunique()

print(f"Rows                  : {len(df)}")
print(f"Unique case IDs       : {unique_cases}")
print(f"Unique evaluation IDs : {unique_eval_ids}")

if unique_cases != 60:
    raise ValueError(
        f"Expected 60 unique case IDs, found {unique_cases}"
    )

if unique_eval_ids != 30:
    raise ValueError(
        f"Expected 30 evaluation IDs, found {unique_eval_ids}"
    )

case_counts = df.groupby("eval_id")["case_id"].nunique()

if not (case_counts == 2).all():
    bad = case_counts[case_counts != 2]
    raise ValueError(
        "Some eval_id values do not contain exactly 2 cases:\n"
        f"{bad}"
    )

print("✓ 60 unique cases")
print("✓ 30 paired evaluation IDs")
print("✓ Exactly 2 cases per source task")

# ============================================================================
# [5] EVALUATION STATUS
# ============================================================================

print("\n" + "=" * 78)
print("[5] EVALUATION STATUS")
print("-" * 78)

status_counts = (
    df["evaluation_status"]
    .fillna("MISSING")
    .value_counts()
)

print(status_counts.to_string())

errors = int(
    (df["evaluation_status"].astype(str).str.upper() != "SUCCESS")
    .sum()
)

print(f"\nNon-success cases : {errors}")

if errors != 0:
    print(
        "⚠ WARNING: STEP 6B contains non-success cases."
    )
else:
    print("✓ All 60 cases have SUCCESS status")

# ============================================================================
# [6] LANGUAGE DISTRIBUTION
# ============================================================================

print("\n" + "=" * 78)
print("[6] EVALUATION LANGUAGE DISTRIBUTION")
print("-" * 78)

language_distribution = (
    df["evaluation_language"]
    .value_counts()
    .rename_axis("evaluation_language")
    .reset_index(name="cases")
)

print(language_distribution.to_string(index=False))

# ============================================================================
# [7] Q(a) DESCRIPTIVE SUMMARY
# ============================================================================

print("\n" + "=" * 78)
print("[7] Q(a) DESCRIPTIVE SUMMARY")
print("-" * 78)

df["Q(a)"] = pd.to_numeric(
    df["Q(a)"],
    errors="coerce"
)

if df["Q(a)"].isna().any():
    raise ValueError(
        "Q(a) contains missing/non-numeric values."
    )

q_summary = {
    "n": int(len(df)),
    "mean": float(df["Q(a)"].mean()),
    "median": float(df["Q(a)"].median()),
    "std": float(df["Q(a)"].std(ddof=1)),
    "min": float(df["Q(a)"].min()),
    "max": float(df["Q(a)"].max()),
    "q_ge_threshold": int(
        (df["Q(a)"] >= Q_THRESHOLD).sum()
    ),
    "q_ge_threshold_rate": float(
        (df["Q(a)"] >= Q_THRESHOLD).mean()
    ),
}

for k, v in q_summary.items():
    if isinstance(v, float):
        print(f"{k:<25}: {v:.6f}")
    else:
        print(f"{k:<25}: {v}")

# ============================================================================
# [8] Q(a) BY EVALUATION LANGUAGE
# ============================================================================

print("\n" + "=" * 78)
print("[8] Q(a) BY ENGLISH / ARABIC")
print("-" * 78)

language_q = (
    df.groupby("evaluation_language")["Q(a)"]
    .agg(
        cases="count",
        mean="mean",
        median="median",
        std="std",
        min="min",
        max="max",
    )
    .reset_index()
)

language_q["q_pass"] = (
    df.groupby("evaluation_language")["Q(a)"]
    .apply(lambda x: int((x >= Q_THRESHOLD).sum()))
    .values
)

language_q["q_pass_rate"] = (
    language_q["q_pass"] /
    language_q["cases"]
)

print(
    language_q.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

# ============================================================================
# [9] DECISION DISTRIBUTION
# ============================================================================

print("\n" + "=" * 78)
print("[9] DECISION DISTRIBUTION")
print("-" * 78)

decision_distribution = (
    df["decision"]
    .fillna("MISSING")
    .value_counts()
    .rename_axis("decision")
    .reset_index(name="cases")
)

print(decision_distribution.to_string(index=False))

# ============================================================================
# [10] DECISION BY LANGUAGE
# ============================================================================

print("\n" + "=" * 78)
print("[10] DECISION × LANGUAGE")
print("-" * 78)

decision_language = pd.crosstab(
    df["evaluation_language"],
    df["decision"]
)

print(decision_language.to_string())

# ============================================================================
# [11] GENERATION MODE
# ============================================================================

print("\n" + "=" * 78)
print("[11] GENERATION MODE DISTRIBUTION")
print("-" * 78)

generation_distribution = (
    df["generation_mode"]
    .fillna("MISSING")
    .value_counts()
    .rename_axis("generation_mode")
    .reset_index(name="cases")
)

print(generation_distribution.to_string(index=False))

# ============================================================================
# [12] GENERATION MODE × LANGUAGE
# ============================================================================

print("\n" + "=" * 78)
print("[12] GENERATION MODE × LANGUAGE")
print("-" * 78)

generation_language = pd.crosstab(
    df["evaluation_language"],
    df["generation_mode"]
)

print(generation_language.to_string())

# ============================================================================
# [13] ALIGNMENT STATUS
# ============================================================================

print("\n" + "=" * 78)
print("[13] ALIGNMENT STATUS")
print("-" * 78)

alignment_distribution = (
    df["alignment_status"]
    .fillna("MISSING")
    .value_counts()
    .rename_axis("alignment_status")
    .reset_index(name="cases")
)

print(alignment_distribution.to_string(index=False))

# ============================================================================
# [14] ITERATION SUMMARY
# ============================================================================

print("\n" + "=" * 78)
print("[14] ITERATION SUMMARY")
print("-" * 78)

df["iterations"] = pd.to_numeric(
    df["iterations"],
    errors="coerce"
)

iteration_summary = {
    "mean_iterations": float(df["iterations"].mean()),
    "median_iterations": float(df["iterations"].median()),
    "min_iterations": int(df["iterations"].min()),
    "max_iterations": int(df["iterations"].max()),
}

for k, v in iteration_summary.items():
    print(f"{k:<25}: {v}")

# ============================================================================
# [15] TASK FAMILY ANALYSIS
# ============================================================================

print("\n" + "=" * 78)
print("[15] TASK FAMILY ANALYSIS")
print("-" * 78)

task_family_summary = (
    df.groupby("task_family")
    .agg(
        cases=("case_id", "count"),
        mean_q=("Q(a)", "mean"),
        median_q=("Q(a)", "median"),
        q_pass=("Q(a)", lambda x: int((x >= Q_THRESHOLD).sum())),
    )
    .reset_index()
)

task_family_summary["q_pass_rate"] = (
    task_family_summary["q_pass"] /
    task_family_summary["cases"]
)

print(
    task_family_summary.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

# ============================================================================
# [16] PROGRAMMING LANGUAGE ANALYSIS
# ============================================================================

print("\n" + "=" * 78)
print("[16] PROGRAMMING LANGUAGE ANALYSIS")
print("-" * 78)

programming_summary = (
    df.groupby("programming_language")
    .agg(
        cases=("case_id", "count"),
        mean_q=("Q(a)", "mean"),
        median_q=("Q(a)", "median"),
        q_pass=("Q(a)", lambda x: int((x >= Q_THRESHOLD).sum())),
    )
    .reset_index()
)

programming_summary["q_pass_rate"] = (
    programming_summary["q_pass"] /
    programming_summary["cases"]
)

programming_summary["evidence_type"] = np.where(
    programming_summary["cases"] < 5,
    "DESCRIPTIVE_ONLY_SMALL_N",
    "DESCRIPTIVE"
)

print(
    programming_summary.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

# ============================================================================
# [17] 5D COLUMN DISCOVERY
# ============================================================================

print("\n" + "=" * 78)
print("[17] 5D METRIC DISCOVERY")
print("-" * 78)

five_d_aliases = {
    "Context Relevance": [
        "context_relevance",
        "Context Relevance",
    ],
    "Factual Consistency": [
        "factual_consistency",
        "Factual Consistency",
    ],
    "Faithfulness": [
        "faithfulness",
        "Faithfulness",
    ],
    "Completeness": [
        "completeness",
        "Completeness",
    ],
    "Language Quality": [
        "language_quality",
        "Language Quality",
    ],
}

five_d_columns = {}

for canonical, aliases in five_d_aliases.items():

    found = None

    for alias in aliases:
        if alias in df.columns:
            found = alias
            break

    if found:
        five_d_columns[canonical] = found
        print(f"✓ {canonical:<25} ← {found}")
    else:
        print(f"⚠ {canonical:<25} NOT FOUND")

# ============================================================================
# [18] 5D DESCRIPTIVE SUMMARY
# ============================================================================

print("\n" + "=" * 78)
print("[18] 5D DESCRIPTIVE SUMMARY")
print("-" * 78)

five_d_summary_rows = []

for canonical, column in five_d_columns.items():

    values = pd.to_numeric(
        df[column],
        errors="coerce"
    )

    five_d_summary_rows.append({
        "metric": canonical,
        "n": int(values.notna().sum()),
        "mean": float(values.mean()),
        "median": float(values.median()),
        "std": float(values.std(ddof=1)),
        "min": float(values.min()),
        "max": float(values.max()),
    })

five_d_summary = pd.DataFrame(
    five_d_summary_rows
)

if len(five_d_summary) > 0:

    print(
        five_d_summary.to_string(
            index=False,
            float_format=lambda x: f"{x:.6f}"
        )
    )

# ============================================================================
# [19] 5D BY LANGUAGE
# ============================================================================

print("\n" + "=" * 78)
print("[19] 5D BY ENGLISH / ARABIC")
print("-" * 78)

five_d_language_rows = []

for canonical, column in five_d_columns.items():

    temp = df.copy()
    temp[column] = pd.to_numeric(
        temp[column],
        errors="coerce"
    )

    grouped = (
        temp.groupby("evaluation_language")[column]
        .agg(
            n="count",
            mean="mean",
            median="median",
            std="std",
        )
        .reset_index()
    )

    for _, row in grouped.iterrows():

        five_d_language_rows.append({
            "metric": canonical,
            "evaluation_language":
                row["evaluation_language"],
            "n": int(row["n"]),
            "mean": float(row["mean"]),
            "median": float(row["median"]),
            "std": float(row["std"]),
        })

five_d_language = pd.DataFrame(
    five_d_language_rows
)

if len(five_d_language) > 0:

    print(
        five_d_language.to_string(
            index=False,
            float_format=lambda x: f"{x:.6f}"
        )
    )

# ============================================================================
# [20] LOW-Q CASES
# ============================================================================

print("\n" + "=" * 78)
print("[20] LOW-Q CASE IDENTIFICATION")
print("-" * 78)

low_q = (
    df[df["Q(a)"] < Q_THRESHOLD]
    [
        [
            "case_id",
            "eval_id",
            "evaluation_language",
            "programming_language",
            "task_family",
            "Q(a)",
            "decision",
        ]
    ]
    .sort_values("Q(a)")
)

print(
    f"Cases below Q threshold : {len(low_q)}/{len(df)}"
)

print("\nLowest-Q cases:")

print(
    low_q.head(15).to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

# ============================================================================
# [21] HIGH-Q BUT NOT ACCEPTED
# ============================================================================

print("\n" + "=" * 78)
print("[21] HIGH-Q BUT NOT ACCEPTED")
print("-" * 78)

high_q_non_accept = df[
    (df["Q(a)"] >= Q_THRESHOLD) &
    (df["decision"].astype(str).str.upper() != "ACCEPT")
][
    [
        "case_id",
        "eval_id",
        "evaluation_language",
        "programming_language",
        "task_family",
        "Q(a)",
        "decision",
    ]
].sort_values(
    "Q(a)",
    ascending=False
)

print(
    f"Cases with Q(a) >= {Q_THRESHOLD} "
    f"but decision != ACCEPT : {len(high_q_non_accept)}"
)

if len(high_q_non_accept) > 0:

    print(
        high_q_non_accept.to_string(
            index=False,
            float_format=lambda x: f"{x:.6f}"
        )
    )

# ============================================================================
# [22] PAIRED ENGLISH ↔ ARABIC DESCRIPTIVE DIFFERENCES
# ============================================================================

print("\n" + "=" * 78)
print("[22] PAIRED ENGLISH ↔ ARABIC DESCRIPTIVE ANALYSIS")
print("-" * 78)

pair_df = (
    df[
        [
            "eval_id",
            "evaluation_language",
            "Q(a)",
        ]
    ]
    .pivot(
        index="eval_id",
        columns="evaluation_language",
        values="Q(a)"
    )
    .reset_index()
)

if {"ENGLISH", "ARABIC"}.issubset(pair_df.columns):

    pair_df["EN_MINUS_AR"] = (
        pair_df["ENGLISH"] -
        pair_df["ARABIC"]
    )

    print(
        f"Pairs              : {len(pair_df)}"
    )

    print(
        f"English mean Q(a)  : "
        f"{pair_df['ENGLISH'].mean():.6f}"
    )

    print(
        f"Arabic mean Q(a)   : "
        f"{pair_df['ARABIC'].mean():.6f}"
    )

    print(
        f"Mean EN - AR       : "
        f"{pair_df['EN_MINUS_AR'].mean():.6f}"
    )

    print(
        f"Median EN - AR     : "
        f"{pair_df['EN_MINUS_AR'].median():.6f}"
    )

else:

    print(
        "⚠ English/Arabic paired Q(a) columns unavailable."
    )

# ============================================================================
# [23] CRITICAL GUARD / VALID-CANDIDATE SUMMARY IF AVAILABLE
# ============================================================================

print("\n" + "=" * 78)
print("[23] VALIDATION / CRITICAL GUARD SUMMARY")
print("-" * 78)

guard_candidates = [
    "critical_guard",
    "critical_guard_pass",
    "best_valid_candidate",
]

for col in guard_candidates:

    if col in df.columns:

        print(f"\nColumn: {col}")

        print(
            df[col]
            .value_counts(dropna=False)
            .to_string()
        )

# ============================================================================
# [24] DESCRIPTIVE INTERPRETATION
# ============================================================================

print("\n" + "=" * 78)
print("[24] STEP 6C DESCRIPTIVE INTERPRETATION")
print("-" * 78)

print(
    f"""
✓ STEP 6B produced {len(df)} successful evaluation records.

✓ Q(a) threshold remains PROPOSED / CALIBRATION:
  Q(a) >= {Q_THRESHOLD}

✓ This STEP 6C does NOT treat Q(a)={Q_THRESHOLD}
  as an established accuracy cutoff.

✓ English/Arabic comparisons here are DESCRIPTIVE ONLY.

✓ Small programming-language groups are DESCRIPTIVE ONLY.

✓ No statistical significance claims are made in STEP 6C.

✓ No multiple-comparison correction is applied in STEP 6C.

✓ Holm correction is reserved for STEP 6D and will be applied
  ONLY to the planned 5D inferential tests.

✓ Frozen Dataset was not modified.

✓ Evaluation Matrix was not modified.

✓ No model execution occurred.
"""
)

# ============================================================================
# [25] SAVE STEP 6C OUTPUTS
# ============================================================================

print("\n" + "=" * 78)
print("[25] SAVING STEP 6C OUTPUTS")
print("-" * 78)

# Main analyzed results
analysis_csv = OUTPUT_DIR / "step6c_results_analyzed.csv"
df.to_csv(
    analysis_csv,
    index=False,
    encoding="utf-8-sig"
)

# Summary tables
language_q.to_csv(
    OUTPUT_DIR / "step6c_language_q_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

task_family_summary.to_csv(
    OUTPUT_DIR / "step6c_task_family_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

programming_summary.to_csv(
    OUTPUT_DIR / "step6c_programming_language_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

if len(five_d_summary) > 0:
    five_d_summary.to_csv(
        OUTPUT_DIR / "step6c_5d_summary.csv",
        index=False,
        encoding="utf-8-sig"
    )

if len(five_d_language) > 0:
    five_d_language.to_csv(
        OUTPUT_DIR / "step6c_5d_language_summary.csv",
        index=False,
        encoding="utf-8-sig"
    )

low_q.to_csv(
    OUTPUT_DIR / "step6c_low_q_cases.csv",
    index=False,
    encoding="utf-8-sig"
)

high_q_non_accept.to_csv(
    OUTPUT_DIR / "step6c_high_q_non_accept.csv",
    index=False,
    encoding="utf-8-sig"
)

if "pair_df" in globals():
    pair_df.to_csv(
        OUTPUT_DIR / "step6c_paired_language_q.csv",
        index=False,
        encoding="utf-8-sig"
    )

# JSON summary
summary = {
    "step": "6C",
    "version": "X-AutoAlign RAG V3.8.5.1",
    "expected_cases": EXPECTED_CASES,
    "observed_cases": int(len(df)),
    "source_tasks": int(unique_eval_ids),
    "q_threshold": Q_THRESHOLD,
    "tmax": TMAX,
    "evaluation_status": status_counts.to_dict(),
    "q_summary": q_summary,
    "decision_distribution":
        df["decision"].value_counts().to_dict(),
    "generation_mode_distribution":
        df["generation_mode"].value_counts().to_dict(),
    "alignment_status_distribution":
        df["alignment_status"].value_counts().to_dict(),
    "errors": errors,
    "holm_applied": False,
    "holm_scope": "STEP 6D — 5D inferential tests only",
}

with open(
    OUTPUT_DIR / "step6c_summary.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        summary,
        f,
        indent=2,
        ensure_ascii=False
    )

# ============================================================================
# [26] FINAL FILE CHECK
# ============================================================================

print("\n" + "=" * 78)
print("[26] OUTPUT FILE CHECK")
print("-" * 78)

for p in sorted(OUTPUT_DIR.iterdir()):

    if p.is_file():

        size_kb = p.stat().st_size / 1024

        print(
            f"✓ {p.name:<48} "
            f"{size_kb:10.2f} KB"
        )

# ============================================================================
# FINAL STATUS
# ============================================================================

print("\n" + "=" * 78)
print("STEP 6C COMPLETED SUCCESSFULLY")
print("=" * 78)

print(
    f"""
Cases analyzed        : {len(df)}
Unique source tasks   : {unique_eval_ids}
Evaluation errors     : {errors}
Q threshold           : {Q_THRESHOLD}
Holm correction       : NOT APPLIED
Model execution       : NO
Training              : NO
Frozen dataset change : NO
Evaluation matrix change: NO

Output directory:
{OUTPUT_DIR}

NEXT STEP:
STEP 6D — STATISTICAL / COMPARATIVE EVALUATION

IMPORTANT:
✓ STEP 6D should consume STEP 6B / STEP 6C outputs.
✓ Holm correction should be applied ONLY to the planned 5D
  inferential tests.
✓ Do NOT rerun STEP 6B.
"""
)

print("=" * 78)

X-AutoAlign RAG V3.8.5.1 — STEP 6C
DESCRIPTIVE / DIAGNOSTIC EVALUATION

Expected cases       : 60
Expected source tasks: 30
Q threshold          : 0.7
Tmax                 : 2
Output directory     : /content/step6c_analysis

[1] INPUT FILE CHECK
------------------------------------------------------------------------------
✓ STEP 6B JSON found
✓ STEP 6B CSV found
✓ Frozen dataset found
✓ Evaluation matrix found

[2] LOAD STEP 6B RESULTS
------------------------------------------------------------------------------
Rows loaded    : 60
Columns loaded : 23
✓ Exactly 60 STEP 6B rows loaded

[3] REQUIRED COLUMN CHECK
------------------------------------------------------------------------------
✓ All required columns are present

[4] BASIC DATA INTEGRITY
------------------------------------------------------------------------------
Rows                  : 60
Unique case IDs       : 60
Unique evaluation IDs : 30
✓ 60 unique cases
✓ 30 paired evaluation IDs
✓ Exactly 2 cases per source task



In [ ]:
# ============================================================================
# X-AutoAlign RAG V3.8.5.1
# STEP 6D — STATISTICAL / COMPARATIVE EVALUATION
# HOLM CORRECTION — 5D ONLY
#
# FINAL PATCH:
#   5D metrics are stored inside the JSON/CSV "scores" field.
#
# READ-ONLY:
#   - No model execution
#   - No training
#   - No frozen dataset modification
#   - No evaluation matrix modification
# ============================================================================

from pathlib import Path
import json
import math
import ast
import numpy as np
import pandas as pd

from scipy import stats

print("=" * 78)
print("X-AutoAlign RAG V3.8.5.1 — STEP 6D")
print("STATISTICAL / COMPARATIVE EVALUATION")
print("HOLM CORRECTION — 5D ONLY")
print("=" * 78)

# ============================================================================
# CONFIGURATION
# ============================================================================

STEP6B_JSON = Path(
    "/content/evaluation_results_v1.json"
)

STEP6B_CSV = Path(
    "/content/evaluation_results_v1.csv"
)

STEP6C_DIR = Path(
    "/content/step6c_analysis"
)

OUTPUT_DIR = Path(
    "/content/step6d_analysis"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

EXPECTED_CASES = 60
EXPECTED_PAIRS = 30

Q_THRESHOLD = 0.70
ALPHA = 0.05
CONFIDENCE_LEVEL = 0.95

print(f"""
Expected cases       : {EXPECTED_CASES}
Expected pairs       : {EXPECTED_PAIRS}
Q threshold          : {Q_THRESHOLD}
Confidence           : {CONFIDENCE_LEVEL:.0%}
Alpha                : {ALPHA}
Holm family          : 5D ONLY
Output directory     : {OUTPUT_DIR}
""")

# ============================================================================
# [1] INPUT CHECK
# ============================================================================

print("=" * 78)
print("[1] INPUT CHECK")
print("-" * 78)

if not STEP6B_JSON.exists():
    raise FileNotFoundError(
        f"STEP 6B JSON not found:\n{STEP6B_JSON}"
    )

if not STEP6B_CSV.exists():
    raise FileNotFoundError(
        f"STEP 6B CSV not found:\n{STEP6B_CSV}"
    )

print("✓ STEP 6B JSON found")
print("✓ STEP 6B CSV found")

if STEP6C_DIR.exists():
    print("✓ STEP 6C output directory found")
else:
    print("⚠ STEP 6C output directory not found")

# ============================================================================
# [2] LOAD STEP 6B JSON
# ============================================================================

print("\n" + "=" * 78)
print("[2] LOAD STEP 6B JSON")
print("-" * 78)

with open(
    STEP6B_JSON,
    "r",
    encoding="utf-8"
) as f:
    raw_records = json.load(f)

if not isinstance(raw_records, list):
    raise ValueError(
        "STEP 6B JSON must contain a list of records."
    )

print(f"Rows loaded : {len(raw_records)}")

if len(raw_records) != EXPECTED_CASES:
    raise ValueError(
        f"Expected {EXPECTED_CASES} records, "
        f"found {len(raw_records)}."
    )

print(
    f"✓ Exactly {EXPECTED_CASES} STEP 6B records loaded"
)

# ============================================================================
# [3] LOAD CSV FOR AUXILIARY CHECKS
# ============================================================================

print("\n" + "=" * 78)
print("[3] LOAD STEP 6B CSV")
print("-" * 78)

df = pd.read_csv(
    STEP6B_CSV
)

print(f"Rows    : {len(df)}")
print(f"Columns : {len(df.columns)}")

if len(df) != EXPECTED_CASES:
    raise ValueError(
        "CSV and JSON case counts do not match."
    )

print("✓ CSV contains exactly 60 rows")

# ============================================================================
# [4] CORE COLUMN CHECK
# ============================================================================

print("\n" + "=" * 78)
print("[4] CORE COLUMN CHECK")
print("-" * 78)

required_columns = [
    "case_id",
    "eval_id",
    "evaluation_language",
    "programming_language",
    "task_family",
    "decision",
    "Q(a)",
    "scores",
]

missing = [
    c for c in required_columns
    if c not in df.columns
]

if missing:
    raise ValueError(
        "Missing required columns:\n"
        + "\n".join(
            f"  - {x}" for x in missing
        )
    )

print("✓ All required columns present")
print("✓ 'scores' column confirmed")

# ============================================================================
# [5] JSON RECORD STRUCTURE CHECK
# ============================================================================

print("\n" + "=" * 78)
print("[5] JSON RECORD STRUCTURE CHECK")
print("-" * 78)

json_required = [
    "case_id",
    "eval_id",
    "evaluation_language",
    "programming_language",
    "task_family",
    "decision",
    "Q(a)",
    "scores",
]

for i, record in enumerate(
    raw_records
):

    missing_record_fields = [
        field
        for field in json_required
        if field not in record
    ]

    if missing_record_fields:
        raise ValueError(
            f"Record {i} missing fields: "
            f"{missing_record_fields}"
        )

print("✓ All 60 JSON records contain required fields")

# ============================================================================
# [6] EXTRACT 5D FROM scores
# ============================================================================

print("\n" + "=" * 78)
print("[6] 5D SCORE EXTRACTION FROM 'scores'")
print("-" * 78)

FIVE_D = [
    "Context Relevance",
    "Factual Consistency",
    "Faithfulness",
    "Completeness",
    "Language Quality",
]

records_clean = []

for i, record in enumerate(
    raw_records
):

    scores = record["scores"]

    # JSON normally gives us a dict.
    # This branch is only a safety mechanism.
    if isinstance(scores, str):

        try:
            scores = json.loads(scores)

        except Exception:

            try:
                scores = ast.literal_eval(scores)

            except Exception as e:

                raise ValueError(
                    f"Could not parse scores for "
                    f"{record.get('case_id')}: {e}"
                )

    if not isinstance(scores, dict):
        raise ValueError(
            f"'scores' is not a dictionary for "
            f"{record.get('case_id')}"
        )

    missing_5d = [
        metric
        for metric in FIVE_D
        if metric not in scores
    ]

    if missing_5d:
        raise ValueError(
            f"Missing 5D metrics for "
            f"{record.get('case_id')}: "
            f"{missing_5d}"
        )

    clean_record = record.copy()

    for metric in FIVE_D:

        value = pd.to_numeric(
            scores[metric],
            errors="coerce"
        )

        if pd.isna(value):
            raise ValueError(
                f"Non-numeric {metric} for "
                f"{record.get('case_id')}"
            )

        clean_record[metric] = float(value)

    records_clean.append(
        clean_record
    )

print(
    f"✓ Extracted all 5D metrics for "
    f"{len(records_clean)} cases"
)

# ============================================================================
# [7] BUILD ANALYSIS DATAFRAME
# ============================================================================

print("\n" + "=" * 78)
print("[7] BUILD STATISTICAL DATAFRAME")
print("-" * 78)

analysis_df = pd.DataFrame(
    records_clean
)

print(
    f"Rows    : {len(analysis_df)}"
)

print(
    f"Columns : {len(analysis_df.columns)}"
)

for metric in FIVE_D:

    if metric not in analysis_df.columns:
        raise RuntimeError(
            f"5D extraction failure: {metric}"
        )

print("✓ All five 5D metrics available")

# ============================================================================
# [8] DATA INTEGRITY
# ============================================================================

print("\n" + "=" * 78)
print("[8] DATA INTEGRITY")
print("-" * 78)

unique_cases = (
    analysis_df["case_id"].nunique()
)

unique_eval_ids = (
    analysis_df["eval_id"].nunique()
)

print(
    f"Unique case IDs       : {unique_cases}"
)

print(
    f"Unique evaluation IDs : {unique_eval_ids}"
)

if unique_cases != EXPECTED_CASES:
    raise ValueError(
        "Expected 60 unique case IDs."
    )

if unique_eval_ids != EXPECTED_PAIRS:
    raise ValueError(
        "Expected 30 unique evaluation IDs."
    )

case_counts = (
    analysis_df
    .groupby("eval_id")["case_id"]
    .nunique()
)

if not (case_counts == 2).all():
    raise ValueError(
        "Every eval_id must contain exactly "
        "two cases."
    )

language_counts = (
    analysis_df
    .groupby(
        [
            "eval_id",
            "evaluation_language"
        ]
    )
    .size()
    .unstack(fill_value=0)
)

if not {
    "ENGLISH",
    "ARABIC"
}.issubset(
    language_counts.columns
):
    raise ValueError(
        "ENGLISH and ARABIC are both required."
    )

if not (
    (language_counts["ENGLISH"] == 1) &
    (language_counts["ARABIC"] == 1)
).all():
    raise ValueError(
        "Each eval_id must have exactly "
        "one ENGLISH and one ARABIC case."
    )

print("✓ 60 unique cases")
print("✓ 30 paired evaluation IDs")
print("✓ Exactly 2 cases per source task")
print("✓ Valid ENGLISH ↔ ARABIC pairing")

# ============================================================================
# [9] BUILD PAIRED DATASET
# ============================================================================

print("\n" + "=" * 78)
print("[9] BUILDING PAIRED DATASET")
print("-" * 78)

paired_columns = [
    "eval_id",
    "evaluation_language",
] + FIVE_D + [
    "Q(a)",
    "decision",
]

pair_source = analysis_df[
    paired_columns
].copy()

paired = pair_source.pivot(
    index="eval_id",
    columns="evaluation_language"
)

paired.columns = [
    f"{metric}__{language}"
    for metric, language in paired.columns
]

paired = paired.reset_index()

if len(paired) != EXPECTED_PAIRS:
    raise ValueError(
        f"Expected {EXPECTED_PAIRS} paired rows."
    )

print(
    f"✓ {len(paired)} valid ENGLISH ↔ ARABIC pairs"
)

# ============================================================================
# HELPER FUNCTION — COHEN'S dz
# ============================================================================

def cohens_dz(diff):

    diff = np.asarray(
        diff,
        dtype=float
    )

    diff = diff[
        np.isfinite(diff)
    ]

    if len(diff) < 2:
        return np.nan

    sd = np.std(
        diff,
        ddof=1
    )

    if sd == 0:
        return np.nan

    return float(
        np.mean(diff) / sd
    )

# ============================================================================
# HELPER FUNCTION — PAIRED STATISTICS
# ============================================================================

def paired_stats(
    english,
    arabic,
    metric
):

    english = np.asarray(
        english,
        dtype=float
    )

    arabic = np.asarray(
        arabic,
        dtype=float
    )

    mask = (
        np.isfinite(english) &
        np.isfinite(arabic)
    )

    english = english[mask]
    arabic = arabic[mask]

    n = len(english)

    if n < 2:

        return {
            "metric": metric,
            "n": n,
            "english_mean": np.nan,
            "arabic_mean": np.nan,
            "mean_difference_EN_minus_AR": np.nan,
            "median_difference_EN_minus_AR": np.nan,
            "std_difference": np.nan,
            "ci95_low": np.nan,
            "ci95_high": np.nan,
            "paired_t": np.nan,
            "paired_t_p_raw": np.nan,
            "wilcoxon_stat": np.nan,
            "wilcoxon_p_raw": np.nan,
            "cohens_dz": np.nan,
        }

    diff = (
        english -
        arabic
    )

    mean_diff = float(
        np.mean(diff)
    )

    median_diff = float(
        np.median(diff)
    )

    sd_diff = float(
        np.std(
            diff,
            ddof=1
        )
    )

    sem = (
        sd_diff /
        math.sqrt(n)
    )

    t_critical = stats.t.ppf(
        0.975,
        df=n - 1
    )

    ci_low = (
        mean_diff -
        t_critical * sem
    )

    ci_high = (
        mean_diff +
        t_critical * sem
    )

    t_stat, t_p = stats.ttest_rel(
        english,
        arabic
    )

    try:

        w_result = stats.wilcoxon(
            english,
            arabic,
            alternative="two-sided",
            method="auto",
        )

        w_stat = float(
            w_result.statistic
        )

        w_p = float(
            w_result.pvalue
        )

    except Exception:

        w_stat = np.nan
        w_p = np.nan

    return {
        "metric": metric,
        "n": int(n),
        "english_mean": float(
            np.mean(english)
        ),
        "arabic_mean": float(
            np.mean(arabic)
        ),
        "mean_difference_EN_minus_AR":
            mean_diff,
        "median_difference_EN_minus_AR":
            median_diff,
        "std_difference":
            sd_diff,
        "ci95_low":
            float(ci_low),
        "ci95_high":
            float(ci_high),
        "paired_t":
            float(t_stat),
        "paired_t_p_raw":
            float(t_p),
        "wilcoxon_stat":
            w_stat,
        "wilcoxon_p_raw":
            w_p,
        "cohens_dz":
            cohens_dz(diff),
    }

# ============================================================================
# [10] PAIRED Q(a)
# ============================================================================

print("\n" + "=" * 78)
print("PAIRED Q(a) ANALYSIS — ENGLISH vs ARABIC")
print("=" * 78)

q_result = paired_stats(
    paired["Q(a)__ENGLISH"],
    paired["Q(a)__ARABIC"],
    "Q(a)"
)

q_table = pd.DataFrame(
    [q_result]
)

print(
    q_table.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

# ============================================================================
# [11] PAIRED 5D
# ============================================================================

print("\n" + "=" * 78)
print("PAIRED 5D ANALYSIS — ENGLISH vs ARABIC")
print("=" * 78)

five_d_results = []

for metric in FIVE_D:

    result = paired_stats(
        paired[
            f"{metric}__ENGLISH"
        ],
        paired[
            f"{metric}__ARABIC"
        ],
        metric
    )

    five_d_results.append(
        result
    )

five_d_table = pd.DataFrame(
    five_d_results
)

print(
    five_d_table.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

# ============================================================================
# [12] HOLM CORRECTION — 5D ONLY
# ============================================================================

print("\n" + "=" * 78)
print("HOLM MULTIPLE-COMPARISON CORRECTION")
print("5D TESTS ONLY")
print("=" * 78)

print("""
Holm family contains exactly five planned tests:

1. Context Relevance
2. Factual Consistency
3. Faithfulness
4. Completeness
5. Language Quality

NOT included:
- Q(a)
- McNemar
- Task-family analysis
- Programming-language analysis
- Decision analysis
- Generation-mode analysis
""")

raw_p = five_d_table[
    "paired_t_p_raw"
].to_numpy(
    dtype=float
)

valid_indices = [
    i
    for i, p in enumerate(raw_p)
    if np.isfinite(p)
]

sorted_indices = sorted(
    valid_indices,
    key=lambda i: raw_p[i]
)

m = len(sorted_indices)

holm_p = np.full(
    len(raw_p),
    np.nan
)

previous = 0.0

for rank, idx in enumerate(
    sorted_indices
):

    adjusted = (
        m - rank
    ) * raw_p[idx]

    adjusted = min(
        adjusted,
        1.0
    )

    adjusted = max(
        adjusted,
        previous
    )

    holm_p[idx] = adjusted

    previous = adjusted

five_d_table[
    "holm_p"
] = holm_p

five_d_table[
    "holm_rank"
] = np.nan

for rank, idx in enumerate(
    sorted_indices,
    start=1
):

    five_d_table.loc[
        idx,
        "holm_rank"
    ] = rank

five_d_table[
    "significant_raw_alpha_0_05"
] = (
    five_d_table[
        "paired_t_p_raw"
    ] < ALPHA
)

five_d_table[
    "significant_holm_alpha_0_05"
] = (
    five_d_table[
        "holm_p"
    ] < ALPHA
)

five_d_table[
    "effect_interpretation"
] = (
    five_d_table[
        "cohens_dz"
    ].apply(
        lambda d:
            "negligible"
            if abs(d) < 0.20
            else
            "small"
            if abs(d) < 0.50
            else
            "medium"
            if abs(d) < 0.80
            else
            "large"
    )
)

holm_table = (
    five_d_table
    .sort_values(
        "paired_t_p_raw"
    )
    .reset_index(drop=True)
)

print(
    holm_table[
        [
            "metric",
            "paired_t_p_raw",
            "holm_p",
            "holm_rank",
            "cohens_dz",
            "effect_interpretation",
            "significant_raw_alpha_0_05",
            "significant_holm_alpha_0_05",
        ]
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

# ============================================================================
# [13] HOLM INTERPRETATION
# ============================================================================

print("\n" + "=" * 78)
print("HOLM-CORRECTED 5D INTERPRETATION")
print("=" * 78)

for _, row in holm_table.iterrows():

    metric = row["metric"]
    raw = row["paired_t_p_raw"]
    adjusted = row["holm_p"]

    if adjusted < ALPHA:
        status = (
            "SIGNIFICANT AFTER HOLM"
        )
    else:
        status = (
            "NOT SIGNIFICANT AFTER HOLM"
        )

    print(
        f"{metric:<25} "
        f"raw p={raw:.6f} | "
        f"Holm p={adjusted:.6f} | "
        f"{status}"
    )

# ============================================================================
# [14] Q(a) THRESHOLD — NOT HOLM CORRECTED
# ============================================================================

print("\n" + "=" * 78)
print("Q(a) THRESHOLD COMPARISON")
print("NOT PART OF HOLM FAMILY")
print("=" * 78)

english_q = (
    paired["Q(a)__ENGLISH"]
)

arabic_q = (
    paired["Q(a)__ARABIC"]
)

english_pass = (
    english_q >= Q_THRESHOLD
)

arabic_pass = (
    arabic_q >= Q_THRESHOLD
)

print(
    f"English Q(a) >= {Q_THRESHOLD:.2f}: "
    f"{int(english_pass.sum())}/"
    f"{len(english_pass)} "
    f"({100 * english_pass.mean():.2f}%)"
)

print(
    f"Arabic Q(a) >= {Q_THRESHOLD:.2f}: "
    f"{int(arabic_pass.sum())}/"
    f"{len(arabic_pass)} "
    f"({100 * arabic_pass.mean():.2f}%)"
)

both_pass = int(
    (
        english_pass &
        arabic_pass
    ).sum()
)

english_only = int(
    (
        english_pass &
        ~arabic_pass
    ).sum()
)

arabic_only = int(
    (
        ~english_pass &
        arabic_pass
    ).sum()
)

both_fail = int(
    (
        ~english_pass &
        ~arabic_pass
    ).sum()
)

print("\nPaired threshold contingency:")
print(f"Both pass    : {both_pass}")
print(f"English only : {english_only}")
print(f"Arabic only  : {arabic_only}")
print(f"Both fail    : {both_fail}")

discordant = (
    english_only +
    arabic_only
)

if discordant > 0:

    smaller = min(
        english_only,
        arabic_only
    )

    mcnemar_p = stats.binomtest(
        smaller,
        n=discordant,
        p=0.5,
        alternative="two-sided"
    ).pvalue

else:

    mcnemar_p = 1.0

print(
    f"McNemar exact p-value : "
    f"{mcnemar_p:.6f}"
)

# ============================================================================
# [15] DECISION COMPARISON
# ============================================================================

print("\n" + "=" * 78)
print("DECISION COMPARISON")
print("=" * 78)

decision_pair = (
    analysis_df[
        [
            "eval_id",
            "evaluation_language",
            "decision",
        ]
    ]
    .pivot(
        index="eval_id",
        columns="evaluation_language",
        values="decision"
    )
)

print(
    pd.crosstab(
        decision_pair["ENGLISH"],
        decision_pair["ARABIC"]
    ).to_string()
)

# ============================================================================
# [16] GENERATION MODE COMPARISON
# ============================================================================

print("\n" + "=" * 78)
print("GENERATION MODE COMPARISON")
print("=" * 78)

generation_pair = (
    analysis_df[
        [
            "eval_id",
            "evaluation_language",
            "generation_mode",
        ]
    ]
    .pivot(
        index="eval_id",
        columns="evaluation_language",
        values="generation_mode"
    )
)

print(
    pd.crosstab(
        generation_pair["ENGLISH"],
        generation_pair["ARABIC"]
    ).to_string()
)

# ============================================================================
# [17] TASK FAMILY — DESCRIPTIVE ONLY
# ============================================================================

print("\n" + "=" * 78)
print("TASK FAMILY — DESCRIPTIVE ANALYSIS")
print("NOT PART OF HOLM FAMILY")
print("=" * 78)

task_rows = []

for family, group in analysis_df.groupby(
    "task_family"
):

    pivot = (
        group[
            [
                "eval_id",
                "evaluation_language",
                "Q(a)",
            ]
        ]
        .pivot(
            index="eval_id",
            columns="evaluation_language",
            values="Q(a)"
        )
    )

    if not {
        "ENGLISH",
        "ARABIC"
    }.issubset(
        pivot.columns
    ):
        continue

    valid = pivot.dropna()

    n = len(valid)

    if n < 2:

        task_rows.append({
            "task_family": family,
            "n": n,
            "english_mean": np.nan,
            "arabic_mean": np.nan,
            "mean_difference_EN_minus_AR": np.nan,
            "paired_t_p_raw": np.nan,
            "wilcoxon_p_raw": np.nan,
            "cohens_dz": np.nan,
        })

        continue

    res = paired_stats(
        valid["ENGLISH"],
        valid["ARABIC"],
        f"Q(a) — {family}"
    )

    task_rows.append({
        "task_family": family,
        "n": n,
        "english_mean":
            res["english_mean"],
        "arabic_mean":
            res["arabic_mean"],
        "mean_difference_EN_minus_AR":
            res[
                "mean_difference_EN_minus_AR"
            ],
        "paired_t_p_raw":
            res[
                "paired_t_p_raw"
            ],
        "wilcoxon_p_raw":
            res[
                "wilcoxon_p_raw"
            ],
        "cohens_dz":
            res["cohens_dz"],
    })

task_family_table = pd.DataFrame(
    task_rows
)

print(
    task_family_table.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

# ============================================================================
# [18] PROGRAMMING LANGUAGE — DESCRIPTIVE ONLY
# ============================================================================

print("\n" + "=" * 78)
print("PROGRAMMING LANGUAGE ANALYSIS")
print("DESCRIPTIVE ONLY")
print("=" * 78)

programming_table = (
    analysis_df
    .groupby(
        "programming_language"
    )
    .agg(
        cases=("case_id", "count"),
        mean_q=("Q(a)", "mean"),
        q_pass=(
            "Q(a)",
            lambda x:
                int(
                    (
                        x >= Q_THRESHOLD
                    ).sum()
                )
        ),
    )
    .reset_index()
)

programming_table[
    "q_pass_rate"
] = (
    programming_table["q_pass"] /
    programming_table["cases"]
)

programming_table[
    "interpretation"
] = np.where(
    programming_table["cases"] < 5,
    "DESCRIPTIVE ONLY — SMALL N",
    "DESCRIPTIVE"
)

print(
    programming_table.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

# ============================================================================
# [19] 5D OVERALL RANKING
# ============================================================================

print("\n" + "=" * 78)
print("5D OVERALL RANKING")
print("=" * 78)

overall_rows = []

for metric in FIVE_D:

    values = analysis_df[
        metric
    ]

    overall_rows.append({
        "metric": metric,
        "mean": float(
            values.mean()
        ),
        "median": float(
            values.median()
        ),
        "std": float(
            values.std(ddof=1)
        ),
    })

overall_5d = (
    pd.DataFrame(
        overall_rows
    )
    .sort_values(
        "mean",
        ascending=False
    )
)

print(
    overall_5d.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

# ============================================================================
# [20] STATISTICAL INTERPRETATION
# ============================================================================

print("\n" + "=" * 78)
print("STATISTICAL INTERPRETATION NOTE")
print("=" * 78)

print("""
✓ Paired tests use the 30 matched EN/AR evaluation pairs.

✓ Mean differences are defined as:
  ENGLISH minus ARABIC.

✓ 95% CIs are computed for paired mean differences.

✓ Cohen's dz is the paired effect size.

✓ Wilcoxon signed-rank tests provide a
  non-parametric paired check.

✓ Q(a)=0.70 remains a PROPOSED / CALIBRATION
  threshold, not an established accuracy cutoff.

✓ Programming-language and task-family analyses
  are descriptive/exploratory.

✓ Holm correction is applied ONLY to the five
  planned 5D paired t-tests.

✓ Raw p-values remain available.

✓ Holm-adjusted p-values are reported separately.

✓ No Holm correction is applied to Q(a), McNemar,
  task-family, programming-language, decision,
  or generation-mode analyses.

✓ No model execution occurred.
✓ No training occurred.
✓ No dataset was modified.
✓ No evaluation matrix was modified.
""")

# ============================================================================
# [21] SAVE OUTPUTS
# ============================================================================

print("\n" + "=" * 78)
print("[21] SAVING STEP 6D RESULTS")
print("-" * 78)

# Main 5D table
five_d_table.to_csv(
    OUTPUT_DIR /
    "step6d_5d_statistics_raw_and_holm.csv",
    index=False,
    encoding="utf-8-sig"
)

# Holm sorted
holm_table.to_csv(
    OUTPUT_DIR /
    "step6d_5d_holm_sorted.csv",
    index=False,
    encoding="utf-8-sig"
)

# Q
q_table.to_csv(
    OUTPUT_DIR /
    "step6d_q_paired_statistics.csv",
    index=False,
    encoding="utf-8-sig"
)

# Task family
task_family_table.to_csv(
    OUTPUT_DIR /
    "step6d_task_family_descriptive.csv",
    index=False,
    encoding="utf-8-sig"
)

# Programming language
programming_table.to_csv(
    OUTPUT_DIR /
    "step6d_programming_language_descriptive.csv",
    index=False,
    encoding="utf-8-sig"
)

# Overall 5D
overall_5d.to_csv(
    OUTPUT_DIR /
    "step6d_5d_overall_ranking.csv",
    index=False,
    encoding="utf-8-sig"
)

# Paired dataset
paired.to_csv(
    OUTPUT_DIR /
    "step6d_paired_dataset.csv",
    index=False,
    encoding="utf-8-sig"
)

# Extracted 5D full dataset
analysis_df[
    [
        "case_id",
        "eval_id",
        "evaluation_language",
        "programming_language",
        "task_family",
    ] + FIVE_D
].to_csv(
    OUTPUT_DIR /
    "step6d_extracted_5d_scores.csv",
    index=False,
    encoding="utf-8-sig"
)

# ============================================================================
# [22] SAVE JSON SUMMARY
# ============================================================================

holm_significant = int(
    five_d_table[
        "significant_holm_alpha_0_05"
    ].sum()
)

summary = {
    "step": "6D",
    "version": "X-AutoAlign RAG V3.8.5.1",

    "cases": int(
        len(analysis_df)
    ),

    "paired_evaluations": int(
        len(paired)
    ),

    "q_threshold": Q_THRESHOLD,

    "confidence_level":
        CONFIDENCE_LEVEL,

    "alpha": ALPHA,

    "holm": {
        "applied": True,
        "scope": "5D_ONLY",
        "family_size": 5,
        "metrics": FIVE_D,
        "method": "Holm step-down",
    },

    "holm_significant_count":
        holm_significant,

    "mcnemar_exact_p":
        float(mcnemar_p),

    "five_d_results":
        five_d_table.replace(
            {np.nan: None}
        ).to_dict(
            orient="records"
        ),

    "overall_5d":
        overall_5d.replace(
            {np.nan: None}
        ).to_dict(
            orient="records"
        ),

    "model_execution": False,
    "training": False,
    "frozen_dataset_modified": False,
    "evaluation_matrix_modified": False,
}

with open(
    OUTPUT_DIR /
    "step6d_summary.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        summary,
        f,
        indent=2,
        ensure_ascii=False
    )

# ============================================================================
# [23] OUTPUT FILE CHECK
# ============================================================================

print("\n" + "=" * 78)
print("[23] OUTPUT FILE CHECK")
print("-" * 78)

files = sorted(
    OUTPUT_DIR.iterdir()
)

for path in files:

    if path.is_file():

        size_kb = (
            path.stat().st_size /
            1024
        )

        print(
            f"✓ {path.name:<55} "
            f"{size_kb:10.2f} KB"
        )

# ============================================================================
# FINAL STATUS
# ============================================================================

print("\n" + "=" * 78)
print("STEP 6D COMPLETED SUCCESSFULLY")
print("=" * 78)

print(f"""
✓ Cases analyzed           : {len(analysis_df)}
✓ Paired evaluations       : {len(paired)}
✓ 5D metrics               : 5
✓ 5D scores extracted      : YES

✓ Paired t-tests           : YES
✓ 95% confidence intervals : YES
✓ Wilcoxon tests           : YES
✓ Cohen's dz               : YES

✓ Raw p-values              : PRESERVED
✓ Holm correction           : APPLIED
✓ Holm scope                : 5D ONLY
✓ Holm family size          : 5
✓ Holm-significant metrics  : {holm_significant}/5

✓ Q(a) analysis             : COMPLETED
✓ McNemar analysis         : COMPLETED
✓ Task-family analysis     : DESCRIPTIVE
✓ Programming-language     : DESCRIPTIVE

✓ Model execution           : NO
✓ Training                  : NO
✓ Frozen Dataset modified   : NO
✓ Evaluation Matrix modified: NO

Output directory:
{OUTPUT_DIR}
""")

print("=" * 78)
print("STEP 6D FINAL STATUS")
print("=" * 78)

print(
    "PASS ✓ | STATISTICAL / COMPARATIVE ANALYSIS COMPLETED"
)

print("""
IMPORTANT:
✓ The five 5D tests form one Holm correction family.
✓ Raw and Holm-adjusted p-values are both retained.
✓ Q(a)=0.70 remains a proposed/calibration threshold.
✓ No exploratory small-n analysis is treated as confirmatory.
✓ STEP 6B does not need to be rerun.
""")

print("=" * 78)

X-AutoAlign RAG V3.8.5.1 — STEP 6D
STATISTICAL / COMPARATIVE EVALUATION
HOLM CORRECTION — 5D ONLY

Expected cases       : 60
Expected pairs       : 30
Q threshold          : 0.7
Confidence           : 95%
Alpha                : 0.05
Holm family          : 5D ONLY
Output directory     : /content/step6d_analysis

[1] INPUT CHECK
------------------------------------------------------------------------------
✓ STEP 6B JSON found
✓ STEP 6B CSV found
✓ STEP 6C output directory found

[2] LOAD STEP 6B JSON
------------------------------------------------------------------------------
Rows loaded : 60
✓ Exactly 60 STEP 6B records loaded

[3] LOAD STEP 6B CSV
------------------------------------------------------------------------------
Rows    : 60
Columns : 23
✓ CSV contains exactly 60 rows

[4] CORE COLUMN CHECK
------------------------------------------------------------------------------
✓ All required columns present
✓ 'scores' column confirmed

[5] JSON RECORD STRUCTURE CHECK
-----------